In [ ]:
install.packages("SoupX")
library(SoupX)

In [ ]:
# library(SoupX)
library(Seurat)
library(dplyr)

In [ ]:
# 方法1：用正斜杠（推荐）
root_dir <- "./data"


# 检查路径是否存在
file.exists(root_dir)           # 必须返回 TRUE
dir.exists(root_dir)            # 必须返回 TRUE

# 列出该目录下的内容
list.dirs(root_dir, recursive = FALSE)
list.files(root_dir)

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install("DropletUtils")

In [ ]:
library(SoupX)
library(Seurat)
library(dplyr)
library(Matrix)

# ========== 路径 ==========
root_dir <- "./data"

if (!dir.exists(root_dir)) {
  stop("路径不存在！请检查: ", root_dir)
}

# ========== 自定义读取 10X 数据 ==========
read10x_custom <- function(data.dir) {
  mat <- Matrix::readMM(gzfile(file.path(data.dir, "matrix.mtx.gz")))
  
  barcodes <- read.table(
    gzfile(file.path(data.dir, "barcodes.tsv.gz")),
    header = FALSE, stringsAsFactors = FALSE, sep = "\t"
  )[, 1]
  
  features <- read.table(
    gzfile(file.path(data.dir, "features.tsv.gz")),
    header = FALSE, stringsAsFactors = FALSE, sep = "\t"
  )
  
  if (ncol(features) >= 2) {
    rownames(mat) <- make.names(features[, 2], unique = TRUE)
  } else {
    rownames(mat) <- make.names(features[, 1], unique = TRUE)
  }
  colnames(mat) <- barcodes
  mat <- as(mat, "CsparseMatrix")
  return(mat)
}

# ========== 自定义保存 10X 数据 ==========
write10x_custom <- function(path, mat) {
  if (!dir.exists(path)) dir.create(path, recursive = TRUE, showWarnings = FALSE)
  
  features_df <- data.frame(
    gene_id = rownames(mat),
    gene_name = rownames(mat),
    feature_type = "Gene Expression",
    stringsAsFactors = FALSE
  )
  
  Matrix::writeMM(mat, file = file.path(path, "matrix.mtx"))
  writeLines(colnames(mat), con = file.path(path, "barcodes.tsv"))
  write.table(features_df, file = file.path(path, "features.tsv"),
              row.names = FALSE, col.names = FALSE, sep = "\t", quote = FALSE)
  
  system(paste("gzip -f", file.path(path, "matrix.mtx")))
  system(paste("gzip -f", file.path(path, "barcodes.tsv")))
  system(paste("gzip -f", file.path(path, "features.tsv")))
  
  cat("  已保存 10X 格式至:", path, "\n")
}

# ========== 主流程 ==========
output_dir <- file.path(root_dir, "soupx_corrected")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

sample_dirs <- list.dirs(root_dir, recursive = FALSE)
sample_dirs <- sample_dirs[grep("^cDNA-", basename(sample_dirs))]
cat("检测到样本:", length(sample_dirs), "\n")
print(basename(sample_dirs))

if (length(sample_dirs) == 0) stop("未找到样本")

soupx_results <- list()

for (sample_path in sample_dirs) {
  sample_name <- basename(sample_path)
  cat("\n==========", sample_name, "==========\n")
  
  raw_dir <- file.path(sample_path, "output", "raw_matrix")
  filtered_dir <- file.path(sample_path, "output", "filter_matrix")
  
  if (!dir.exists(raw_dir) || !dir.exists(filtered_dir)) {
    cat("  [跳过] raw/filter 目录不存在\n")
    next
  }
  
  tryCatch({
    cat("  读取 raw matrix...\n")
    raw_counts <- read10x_custom(raw_dir)
    cat("  raw:", nrow(raw_counts), "genes x", ncol(raw_counts), "cells\n")
    
    cat("  读取 filtered matrix...\n")
    filtered_counts <- read10x_custom(filtered_dir)
    cat("  filtered:", nrow(filtered_counts), "genes x", ncol(filtered_counts), "cells\n")
    
    # 对齐基因
    cat("  对齐基因集合...\n")
    common_genes <- intersect(rownames(raw_counts), rownames(filtered_counts))
    cat("  共同基因数:", length(common_genes), "\n")
    
    raw_counts <- raw_counts[common_genes, ]
    filtered_counts <- filtered_counts[common_genes, ]
    raw_counts <- as(raw_counts, "CsparseMatrix")
    filtered_counts <- as(filtered_counts, "CsparseMatrix")
    
    cat("  构建 SoupChannel...\n")
    sc <- SoupChannel(tod = raw_counts, toc = filtered_counts)
    
    # Seurat 聚类
    cat("  Seurat 聚类...\n")
    seu <- CreateSeuratObject(counts = filtered_counts)
    seu <- NormalizeData(seu)
    seu <- FindVariableFeatures(seu, selection.method = "vst", nfeatures = 2000)
    seu <- ScaleData(seu)
    seu <- RunPCA(seu, features = VariableFeatures(object = seu))
    seu <- FindNeighbors(seu, dims = 1:20)
    seu <- FindClusters(seu, resolution = 0.8)
    
    # ========== 关键修复：setClusters 是函数，不是方法 ==========
    cat("  传入聚类信息...\n")
    sc <- setClusters(sc, setNames(seu$seurat_clusters, colnames(seu)))
    
    # SoupX 估计污染
    cat("  估计污染率...\n")
    sc <- autoEstCont(sc, forceAccept = TRUE)
    
    rho <- sc$fit$rhoEst
    cat("  污染率:", round(rho * 100, 2), "%\n")
    
    corrected_counts <- adjustCounts(sc)
    
    # 保存
    sample_out <- file.path(output_dir, sample_name)
    dir.create(sample_out, showWarnings = FALSE, recursive = TRUE)
    
    write10x_custom(file.path(sample_out, "soupx_matrix"), corrected_counts)
    saveRDS(corrected_counts, file = file.path(sample_out, "soupx_counts.rds"))
    
    soupx_results[[sample_name]] <- data.frame(
      sample = sample_name,
      rho = rho,
      rho_percent = round(rho * 100, 2),
      stringsAsFactors = FALSE
    )
    cat("  [完成]\n")
    
  }, error = function(e) {
    cat("  [错误]", conditionMessage(e), "\n")
    soupx_results[[sample_name]] <<- data.frame(
      sample = sample_name, rho = NA, rho_percent = NA,
      error = conditionMessage(e), stringsAsFactors = FALSE
    )
  })
}

cat("\n========== 全部完成 ==========\n")
summary_df <- bind_rows(soupx_results)
print(summary_df)
write.csv(summary_df, file.path(output_dir, "soupx_summary.csv"), row.names = FALSE)

In [ ]:
# SoupX 校正后

In [ ]:
# ============================================
# 第二步：SoupX 校正后 QC 与合并
# ============================================

suppressMessages(library(tidyverse))
suppressMessages(library(Seurat))
suppressMessages(library(patchwork))
suppressMessages(library(Matrix))
suppressMessages(library(cowplot))
suppressMessages(library(ggplot2))
suppressMessages(library(harmony))

# ---- 1. 路径设置 ----
soupx_dir <- "./data/soupx_corrected"
save_dir_step2 <- "./data"
fig_dir <- file.path(save_dir_step2, "figures_QC")
dir.create(save_dir_step2, showWarnings = FALSE, recursive = TRUE)
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

# ---- 2. 样本信息（donor + region 映射）----
sample_info <- data.frame(
  sample_label = c("cDNA-250114087", "cDNA-250114088", "cDNA-250110016","cDNA-250110017",
 "cDNA-250110018","cDNA-250110019","cDNA-250114089", "cDNA-250114090",
                   "cDNA-250110006", "cDNA-250110007", "cDNA-250114082", "cDNA-250114083",
                   "cDNA-250114084", "cDNA-250114085", "cDNA-250114086"),
  donor = c(rep("Mq246", 8), rep("Mq277", 7)),
  samples = c(rep("DR-IV-TD", 8), rep("MR-DR-VR-IV", 7)),
  stringsAsFactors = FALSE
)

# ---- 3. 读取 SoupX 校正后的数据 ----
cat("读取 SoupX 校正后数据...\n")
seurat_list <- list()

for (i in 1:nrow(sample_info)) {
  s <- sample_info$sample_label[i]
  cat("  ", s, "\n")
  
  counts <- readRDS(file.path(soupx_dir, s, "soupx_counts.rds"))
  seu <- CreateSeuratObject(counts = counts, project = s)
  
  # 添加 meta data
  seu$sample_label <- s
  seu$donor <- sample_info$donor[i]
  seu$samples <- sample_info$samples[i]
  
  seurat_list[[s]] <- seu
}

# ---- 4. 合并 ----
cat("\n合并所有样本...\n")
seurat <- merge(seurat_list[[1]], y = seurat_list[-1])
cat("合并后:", ncol(seurat), "cells x", nrow(seurat), "genes\n")

# ---- 5. QC 指标计算 ----
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA

MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
found_mt <- MTGenes[MTGenes %in% rownames(seurat)]
cat("找到的 MT 基因:", length(found_mt), "个\n")
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = found_mt)

# ---- 6. 样本过滤（median nFeature > 500）----
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))
select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label
seurat <- subset(seurat, sample_label %in% select_group)
cat("过滤后样本数:", length(select_group), "\n")

# ---- 7. QC 前统计（按文库记录）----
oriCellNum <- ncol(seurat)
cat("QC 前细胞数:", oriCellNum, "\n")

qc_before <- as.data.frame(table(seurat$sample_label))
colnames(qc_before) <- c("sample_label", "cells_before_QC")

# ---- 8. QC 过滤 ----
seurat <- subset(seurat,
                 nFeature_RNA > 700 &
                 percent.mt < 5 &
                 countFeatureRatio > 1.2 &
                 nCount_RNA < 100000)
cellNum1 <- ncol(seurat)
cat("QC 后细胞数:", cellNum1, "(保留", round(cellNum1/oriCellNum*100, 2), "%)\n")

# ---- 8b. QC 后按文库统计 + 前后对照表 ----
qc_after <- as.data.frame(table(seurat$sample_label))
colnames(qc_after) <- c("sample_label", "cells_after_QC")

qc_per_library <- merge(qc_before, qc_after, by = "sample_label", all.x = TRUE)
qc_per_library$cells_after_QC[is.na(qc_per_library$cells_after_QC)] <- 0  # 被整体滤掉的文库补0
qc_per_library$retained_pct <- round(
  qc_per_library$cells_after_QC / qc_per_library$cells_before_QC * 100, 1)

# 补上 donor / region 信息
qc_per_library <- qc_per_library %>%
  left_join(distinct(sample_info, sample_label, donor, samples), by = "sample_label")

cat("\n各文库 QC 前后细胞数:\n")
print(qc_per_library, row.names = FALSE)

write.csv(qc_per_library,
          file.path(save_dir_step2, "QC_per_library_cell_counts.csv"),
          row.names = FALSE)
# ---- 9. QC 出图 ----
Idents(seurat) <- "sample_label"

# 小提琴图
p_vln <- VlnPlot(seurat, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), 
                 pt.size = 0, ncol = 1, group.by = "sample_label")
png(file.path(fig_dir, "QC_violin.png"), width = 1500, height = 1800, res = 150)
print(p_vln)
dev.off()

# 散点图
p1 <- FeatureScatter(seurat, "nCount_RNA", "nFeature_RNA", group.by = "sample_label", pt.size = 0.3)
p2 <- FeatureScatter(seurat, "nCount_RNA", "percent.mt", group.by = "sample_label", pt.size = 0.3)
p3 <- FeatureScatter(seurat, "nFeature_RNA", "percent.mt", group.by = "sample_label", pt.size = 0.3)
p_scatter <- (p1 + p2 + p3) + plot_layout(ncol = 3)

png(file.path(fig_dir, "QC_scatter.png"), width = 2000, height = 650, res = 150)
print(p_scatter)
dev.off()

cat("QC 图已保存至:", fig_dir, "\n")

# ---- 10. 保存第二步结果 ----
filename <- file.path(save_dir_step2, "macaque_DR_soupx_batch1.rds")
saveRDS(seurat, filename)
cat("第二步数据已保存:", filename, "\n")

In [ ]:
# ---- 10. 保存第二步结果 ----
filename <- file.path(save_dir_step2, "macaque_DR_soupx_batch1.rds")
saveRDS(seurat, filename)
cat("第二步数据已保存:", filename, "\n")

In [ ]:
# 单细胞分析

In [ ]:
library(Seurat)
library(dplyr)

seurat <- readRDS("./data/macaque_DR_soupx_batch1.rds")

# 样本过滤（med > 500）
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))
select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label

seurat <- subset(seurat, sample_label %in% select_group)
oriCellNum <- ncol(seurat)

# QC过滤
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA
MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = MTGenes)
seurat_qc <- subset(seurat, nFeature_RNA > 700 & nCount_RNA < 100000 &
                    percent.mt < 5 & countFeatureRatio > 1.2)
cellNum1 <- ncol(seurat_qc)

cat("QC前细胞数:", oriCellNum, "\n")
cat("QC后细胞数:", cellNum1, "\n")
cat("去除:", oriCellNum - cellNum1, 
    "(", round((oriCellNum - cellNum1)/oriCellNum*100, 2), "%)\n")

In [ ]:
# 第二步的出图（无降采样版本）- 修改版
library(Seurat)
library(ggplot2)
library(patchwork)
library(dplyr)

# ==================== 保存路径设置 ====================
save_dir <- "/./data/figures_QC-latest-812"
if (!dir.exists(save_dir)) {
  dir.create(save_dir, recursive = TRUE)
  cat("Created directory:", save_dir, "\n")
}

# ==================== (1) 样本过滤 ====================
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))

select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label
seurat <- subset(seurat, sample_label %in% select_group)

# ==================== (2) QC 指标计算 ====================
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA
MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = MTGenes)
seurat <- NormalizeData(seurat)

oriCellNum <- ncol(seurat)

# ==================== 全部数据用于绘图（不降采样）====================
seurat_plot <- seurat
cat("Plotting with ALL", ncol(seurat_plot), "cells\n")

# ==================== 新增：小提琴图专用子集（nCount_RNA <= 25000）====================
seurat_plot_vln <- subset(seurat_plot, nCount_RNA <= 25000)
cat("Violin plot using", ncol(seurat_plot_vln), "cells (nCount_RNA <= 25000)\n")

# ==================== 发表级小提琴图函数（按donor着色，nFeature每1000一个刻度）====================
myVlnPlot <- function(seurat_obj, feature, group.by = "sample_label", 
                      fill.by = "donor", title = NULL, ylabel = NULL) {
  meta <- seurat_obj@meta.data
  meta$group <- factor(meta[[group.by]])
  meta$fill_group <- factor(meta[[fill.by]])
  
  # donor颜色：Mq246蓝色, Mq277橙色（可根据实际donor名称调整）
  donor_colors <- c("Mq246" = "#1f77b4", "Mq277" = "#ff7f0e")
  
  p <- ggplot(meta, aes(x = group, y = .data[[feature]], fill = fill_group)) +
    geom_violin(scale = "width", trim = TRUE, linewidth = 0.3) +
    geom_violin(scale = "width", trim = TRUE, linewidth = 0.3, alpha = 0.85) +
    scale_fill_manual(values = donor_colors, name = "Donor") +
    theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10, color = "black"),
      axis.text.y = element_text(size = 10, color = "black"),
      axis.title.x = element_blank(),
      axis.title.y = element_text(size = 12, face = "bold"),
      plot.title = element_text(hjust = 0.5, face = "bold", size = 13),
      legend.position = "right",
      legend.key.size = unit(0.4, "cm"),
      legend.text = element_text(size = 9),
      legend.title = element_text(size = 10, face = "bold"),
      panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5)
    ) +
    labs(title = title, y = ifelse(is.null(ylabel), feature, ylabel))
  
  # nFeature_RNA 每隔 1000 一个刻度线
  if (feature == "nFeature_RNA") {
    ymax <- max(meta[[feature]], na.rm = TRUE)
    p <- p + scale_y_continuous(breaks = seq(0, ceiling(ymax / 1000) * 1000, 1000))
  }
  
  return(p)
}

# ==================== (a) QC前小提琴图（只显示nCount_RNA <= 25000，按donor着色）====================
cat("\n--- Saving (a) Pre-QC Violin ---\n")
p_a1 <- myVlnPlot(seurat_plot_vln, "nFeature_RNA", title = "Gene Count", ylabel = "nFeature_RNA")
p_a2 <- myVlnPlot(seurat_plot_vln, "nCount_RNA", title = "UMI Count", ylabel = "nCount_RNA")
p_a3 <- myVlnPlot(seurat_plot_vln, "percent.mt", title = "Mitochondrial %", ylabel = "percent.mt")

vln_pre <- (p_a1 + p_a2 + p_a3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1)) +
  plot_annotation(title = "Pre-QC Metrics by Sample (Colored by Donor)", 
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "a_violin_preQC_byDonor.png"), width = 2000, height = 650, res = 150)
print(vln_pre)
dev.off()
cat("Saved: a_violin_preQC_byDonor.png\n")

# ==================== (b) QC前散点图（不降采样，保持全部数据）====================
cat("\n--- Saving (b) Pre-QC Scatter ---\n")

scatter1 <- FeatureScatter(
  seurat_plot, feature1 = "nCount_RNA", feature2 = "nFeature_RNA",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("UMI vs Gene Count") + 
  theme_classic() +
  theme(legend.position = "none", plot.title = element_text(size = 12, face = "bold"))

scatter2 <- FeatureScatter(
  seurat_plot, feature1 = "nCount_RNA", feature2 = "percent.mt",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("UMI vs MT%") + 
  theme_classic() +
  theme(legend.position = "none", plot.title = element_text(size = 12, face = "bold"))

scatter3 <- FeatureScatter(
  seurat_plot, feature1 = "nFeature_RNA", feature2 = "percent.mt",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("Gene Count vs MT%") + 
  theme_classic() +
  theme(legend.position = "right", plot.title = element_text(size = 12, face = "bold"),
        legend.key.size = unit(0.3, "cm"), legend.text = element_text(size = 8))

scatter_pre <- (scatter1 + scatter2 + scatter3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1.2)) +
  plot_annotation(title = "Pre-QC Scatter Plots",
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "b_scatter_preQC.png"), width = 2000, height = 650, res = 150)
print(scatter_pre)
dev.off()
ggsave(file.path(save_dir, "b_scatter_preQC.pdf"),
       scatter_pre, width = 13.33, height = 4.33, device = cairo_pdf)   # ← 新增
cat("Saved: b_scatter_preQC.png / .pdf\n")

# ==================== (c) QC过滤 ====================
seurat_qc <- subset(seurat, 
                    nFeature_RNA > 700 & 
                    nCount_RNA < 100000 &
                    percent.mt < 5 & 
                    countFeatureRatio > 1.2)

cellNum1 <- ncol(seurat_qc)
cat("\nPost-QC:", cellNum1, "cells (removed", oriCellNum - cellNum1, 
    round((oriCellNum - cellNum1)/oriCellNum * 100, 1), "%)\n")

# ==================== 新增：QC后小提琴图专用子集（nCount_RNA <= 25000）====================
seurat_qc_vln <- subset(seurat_qc, nCount_RNA <= 25000)
cat("Post-QC Violin plot using", ncol(seurat_qc_vln), "cells (nCount_RNA <= 25000)\n")

# ==================== (d) QC后小提琴图（只显示nCount_RNA <= 25000，按donor着色）====================
cat("\n--- Saving (c) Post-QC Violin ---\n")

p_d1 <- myVlnPlot(seurat_qc_vln, "nFeature_RNA", title = "Gene Count", ylabel = "nFeature_RNA")
p_d2 <- myVlnPlot(seurat_qc_vln, "nCount_RNA", title = "UMI Count", ylabel = "nCount_RNA")
p_d3 <- myVlnPlot(seurat_qc_vln, "percent.mt", title = "Mitochondrial %", ylabel = "percent.mt")

vln_post <- (p_d1 + p_d2 + p_d3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1)) +
  plot_annotation(title = "Post-QC Metrics by Sample (Colored by Donor)",
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "c_violin_postQC_byDonor.png"), width = 2000, height = 650, res = 150)
print(vln_post)
dev.off()
ggsave(file.path(save_dir, "c_violin_postQC_byDonor.pdf"),
       vln_post, width = 13.33, height = 4.33, device = cairo_pdf)   # ← 新增
cat("Saved: c_violin_postQC_byDonor.png / .pdf\n")
# ==================== 验证文件 ====================
cat("\n========== Generated Files ==========\n")
files <- c("a_violin_preQC_byDonor.png", "b_scatter_preQC.png", 
           "c_violin_postQC_byDonor.png")
for (f in files) {
  fp <- file.path(save_dir, f)
  if (file.exists(fp)) {
    cat("OK:", f, "-", round(file.size(fp)/1024, 1), "KB\n")
  } else {
    cat("MISSING:", f, "\n")
  }
}

cat("\n========== QC Summary ==========\n")
cat("Original cells:", oriCellNum, "\n")
cat("Post-QC cells:", cellNum1, "\n")
cat("Removed:", oriCellNum - cellNum1, 
    "(", round((oriCellNum - cellNum1)/oriCellNum * 100, 2), "%)\n")
cat("Save directory:", save_dir, "\n")
cat("==================================\n")

In [ ]:
library(ggplot2)

# 提取 meta data
meta <- seurat@meta.data

# 画散点图：nCount_RNA vs nFeature_RNA，颜色 = percent.mt
p <- ggplot(meta, aes(x = nCount_RNA, y = nFeature_RNA, color = percent.mt)) +
  geom_point(size = 0.3, alpha = 0.5) +
  scale_color_distiller(palette = "Reds", direction = 1, name = "percent.mt (%)") +
  labs(x = "nCount_RNA", y = "nFeature_RNA") +
  theme_classic() +
  theme(
    axis.text = element_text(size = 11, color = "black"),
    axis.title = element_text(size = 12, face = "bold"),
    legend.title = element_text(size = 10),
    legend.text = element_text(size = 9)
  )

# 保存 + 显示
fig_dir <- "./data/figures_QC-latest-812"
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

png(file.path(fig_dir, "scatter_nCount_nFeature_mt.png"), width = 800, height = 700, res = 150)
print(p)
dev.off()

cat("已保存:", file.path(fig_dir, "scatter_nCount_nFeature_mt.png"), "\n")

In [ ]:
library(Seurat)
library(Matrix)
library(dplyr)
library(ggplot2)
library(patchwork)

# ========== 路径 ==========
soupx_dir <- "./data/soupx_corrected"
save_dir_step2 <- "./data"
fig_dir <- file.path(save_dir_step2, "figures_QC-latest-812")
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

# ========== 样本信息 ==========
sample_info <- data.frame(
  sample_label = c("cDNA-250114087", "cDNA-250114088", "cDNA-250110016","cDNA-250110017",
 "cDNA-250110018","cDNA-250110019","cDNA-250114089", "cDNA-250114090",
                   "cDNA-250110006", "cDNA-250110007", "cDNA-250114082", "cDNA-250114083",
                   "cDNA-250114084", "cDNA-250114085", "cDNA-250114086"),
  donor = c(rep("Mq246", 8), rep("Mq277", 7)),
  samples = c(rep("DR-IV-TD", 8), rep("MR-DR-VR-IV", 7)),
  stringsAsFactors = FALSE
)

# ========== 1. 读取所有 SoupX counts ==========
cat("读取 SoupX 校正后数据...\n")
counts_list <- list()

for (i in 1:nrow(sample_info)) {
  s <- sample_info$sample_label[i]
  cat("  ", s, "\n")
  
  counts <- readRDS(file.path(soupx_dir, s, "soupx_counts.rds"))
  
  # 给 barcode 加样本前缀，避免不同样本重复
  colnames(counts) <- paste0(colnames(counts), "-", s)
  
  counts_list[[s]] <- counts
}

# ========== 2. 取所有基因并集，对齐每个矩阵（缺失填0）==========
cat("\n对齐基因...\n")
all_genes <- unique(unlist(lapply(counts_list, rownames)))
cat("所有样本基因并集:", length(all_genes), "\n")

counts_list_aligned <- lapply(names(counts_list), function(s) {
  mat <- counts_list[[s]]
  missing_genes <- setdiff(all_genes, rownames(mat))
  
  if (length(missing_genes) > 0) {
    zero_mat <- Matrix(0, nrow = length(missing_genes), ncol = ncol(mat),
                       dimnames = list(missing_genes, colnames(mat)))
    mat <- rbind(mat, zero_mat)
  }
  # 统一排序
  mat <- mat[all_genes, , drop = FALSE]
  return(mat)
})
names(counts_list_aligned) <- names(counts_list)

# ========== 3. cbind 合并所有 counts（1个layer）==========
cat("合并 counts...\n")
combined_counts <- do.call(cbind, counts_list_aligned)
cat("合并后:", nrow(combined_counts), "genes x", ncol(combined_counts), "cells\n")
# 应该是 21883 genes x 137092 cells（QC前）

# ========== 4. 创建 Seurat 对象（只有1个 counts layer）==========
seurat <- CreateSeuratObject(counts = combined_counts)
cat("Seurat 对象:", nrow(seurat), "features,", ncol(seurat), "cells\n")
cat("Layers:", names(seurat@assays$RNA@layers), "\n")

# ========== 5. 添加 meta data ==========
cell_counts <- sapply(counts_list_aligned, ncol)
seurat$sample_label <- rep(sample_info$sample_label, cell_counts)
seurat$donor <- rep(sample_info$donor, cell_counts)
seurat$samples <- rep(sample_info$samples, cell_counts)

# ========== 6. QC 计算 ==========
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA

MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
found_mt <- MTGenes[MTGenes %in% rownames(seurat)]
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = found_mt)

# ========== 7. 样本过滤（median nFeature > 500）==========
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))
select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label
seurat <- subset(seurat, sample_label %in% select_group)

oriCellNum <- ncol(seurat)
cat("QC 前:", oriCellNum, "cells\n")

# ========== 8. QC 前出图 ==========
# （这里放你原来的出图代码，略）

# ========== 9. QC 过滤 ==========
seurat_qc <- subset(seurat,
                    nFeature_RNA > 700 &
                    percent.mt < 5 &
                    countFeatureRatio > 1.2 &
                    nCount_RNA < 100000)

cat("QC 后:", ncol(seurat_qc), "cells (保留", round(ncol(seurat_qc)/oriCellNum*100,2), "%)\n")
# 应该是 135419

# ========== 10. 保存 RDS（关键！）==========
seurat <- seurat_qc
rds_file <- file.path(save_dir_step2, "macaque_DR_soupx_batch1.rds")
saveRDS(seurat, rds_file)
cat("RDS 已保存:", rds_file, "\n")
cat("最终:", ncol(seurat), "cells x", nrow(seurat), "genes, 1 layer\n")

In [ ]:
readDir <- "./data/"
seurat1 <- readRDS(paste0(readDir, "macaque_DR_soupx_batch1.rds"))

In [ ]:
# 测双胞率

In [ ]:
library(Seurat)
library(scDblFinder)
library(SingleCellExperiment)
library(dplyr)
library(ggplot2)

seurat1$sample_label <- as.character(seurat1$sample_label)
samples <- unique(seurat1$sample_label)

# 初始化（显式设为字符型，避免 factor 问题）
seurat1$scDblFinder.class <- NA_character_
seurat1$scDblFinder.score <- NA_real_

for (s in samples) {
  message("Processing sample: ", s)
  
  subset_obj <- subset(seurat1, subset = sample_label == s)
  if (ncol(subset_obj) < 100) next
  
  # 标准化 + PCA
  subset_obj <- NormalizeData(subset_obj)
  subset_obj <- FindVariableFeatures(subset_obj, selection.method = "vst", nfeatures = 2000)
  subset_obj <- ScaleData(subset_obj)
  subset_obj <- RunPCA(subset_obj, features = VariableFeatures(object = subset_obj))
  
  # 转 SCE 并跑 scDblFinder
  sce <- as.SingleCellExperiment(subset_obj)
  sce <- scDblFinder(sce)
  
  # ========== 关键修复：用 AddMetaData 按细胞名匹配写回 ==========
  meta_to_add <- data.frame(
    scDblFinder.class = as.character(sce$scDblFinder.class),
    scDblFinder.score = sce$scDblFinder.score,
    stringsAsFactors = FALSE
  )
  rownames(meta_to_add) <- colnames(sce)  # 确保细胞名对齐
  
  seurat1 <- AddMetaData(seurat1, metadata = meta_to_add)
  
  dbl_n <- sum(sce$scDblFinder.class == "doublet", na.rm = TRUE)
  message("  Done. Doublets: ", dbl_n, "/", ncol(subset_obj), 
          " (", round(dbl_n / ncol(subset_obj) * 100, 2), "%)")
}

# ========== 统计各样本双胞率 ==========
doublet_summary <- seurat1@meta.data %>%
  group_by(sample_label) %>%
  summarise(
    total_cells = n(),
    doublet_cells = sum(scDblFinder.class == "doublet", na.rm = TRUE),
    doublet_rate = round(doublet_cells / total_cells * 100, 2)
  ) %>%
  arrange(desc(doublet_rate))

print(doublet_summary)

# ========== 可视化 ==========
# 柱形图
p1 <- ggplot(doublet_summary, aes(x = reorder(sample_label, -doublet_rate), y = doublet_rate)) +
  geom_bar(stat = "identity", fill = "steelblue") +
  geom_text(aes(label = paste0(doublet_rate, "%")), vjust = -0.5, size = 3) +
  labs(x = "Sample", y = "Doublet Rate (%)", title = "scDblFinder Doublet Rate per Sample") +
  theme_bw(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
print(p1)

# 如果已有 UMAP，看双胞分布
if ("umap" %in% names(seurat1@reductions)) {
  p2 <- DimPlot(seurat1, group.by = "scDblFinder.class", label = FALSE,
                cols = c("singlet" = "grey", "doublet" = "red")) +
    ggtitle("scDblFinder Predictions")
  print(p2)
}

# ========== 过滤双胞并保存 ==========
seurat1_clean <- subset(seurat1, subset = scDblFinder.class == "singlet")

message("Before filtering: ", ncol(seurat1), " cells")
message("After filtering:  ", ncol(seurat1_clean), " cells")
message("Removed:          ", ncol(seurat1) - ncol(seurat1_clean), " doublets")

# 保存
saveRDS(seurat1_clean, paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned.rds"))

In [ ]:
# 1. 看所有metadata列名
colnames(seurat1@meta.data)

# 2. 看sample_label分布
table(seurat1@meta.data$sample_label)

# 3. 看samples分布
table(seurat1@meta.data$samples)

# 4. 看cell barcode格式
head(Cells(seurat1))

# 5. 从barcode提取后缀作为文库ID
library(dplyr)
seurat1$library_id <- sapply(strsplit(Cells(seurat1), "-"), function(x) tail(x, 1))
table(seurat1$library_id)





In [ ]:
# === 先读取上一步保存的 RDS 文件 ===
readDir <- "./data/"

# 如果你上一步保存的是这个文件名：
seurat <- readRDS(paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned.rds"))

# 如果上面的文件不存在，试试双胞过滤后但未 QC 的版本：
# seurat <- readRDS(paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned.rds"))

cat("Loaded object with", ncol(seurat), "cells\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(patchwork)

# ============================================
# 0. 设置路径 & 加载数据
# ============================================
readDir  <- "./data/"
save_dir <- "./data"

dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)   # ← 加上这行

after_doublet   <- ncol(seurat)   
# ============================================
# 1. 兼容 donor 列
# ============================================
if (!"donor" %in% colnames(seurat@meta.data)) {
  seurat$donor <- seurat$sample_label
  donor_colors <- NULL
  message("Note: No 'donor' column found. Using sample_label for coloring.")
} else {
  donor_colors <- c("Mq246" = "#1f77b4", "Mq277" = "#ff7f0e")
}

# ============================================
# 2. 样本过滤（median nFeature > 500）
# ============================================
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))

select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label
seurat <- subset(seurat, sample_label %in% select_group)

# ============================================
# 3. QC 指标计算
# ============================================
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA

MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = MTGenes)
seurat <- NormalizeData(seurat)

oriCellNum <- ncol(seurat)

# ============================================
# 4. 绘图数据准备
# ============================================
seurat_plot <- seurat
cat("Plotting with ALL", ncol(seurat_plot), "cells\n")

seurat_plot_vln <- subset(seurat_plot, nCount_RNA <= 25000)
cat("Violin plot using", ncol(seurat_plot_vln), "cells (nCount_RNA <= 25000)\n")

# ============================================
# 5. 小提琴图函数
# ============================================
myVlnPlot <- function(seurat_obj, feature, group.by = "sample_label", 
                      fill.by = "donor", title = NULL, ylabel = NULL) {
  meta <- seurat_obj@meta.data
  meta$group <- factor(meta[[group.by]])
  meta$fill_group <- factor(meta[[fill.by]])
  
  p <- ggplot(meta, aes(x = group, y = .data[[feature]], fill = fill_group)) +
    geom_violin(scale = "width", trim = TRUE, linewidth = 0.3, alpha = 0.85)
  
  if (!is.null(donor_colors)) {
    p <- p + scale_fill_manual(values = donor_colors, name = "Donor")
  } else {
    p <- p + scale_fill_discrete(name = fill.by)
  }
  
  p <- p + theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10, color = "black"),
      axis.text.y = element_text(size = 10, color = "black"),
      axis.title.x = element_blank(),
      axis.title.y = element_text(size = 12, face = "bold"),
      plot.title = element_text(hjust = 0.5, face = "bold", size = 13),
      legend.position = "right",
      legend.key.size = unit(0.4, "cm"),
      legend.text = element_text(size = 9),
      legend.title = element_text(size = 10, face = "bold"),
      panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5)
    ) +
    labs(title = title, y = ifelse(is.null(ylabel), feature, ylabel))
  
  if (feature == "nFeature_RNA") {
    ymax <- max(meta[[feature]], na.rm = TRUE)
    p <- p + scale_y_continuous(breaks = seq(0, ceiling(ymax / 1000) * 1000, 1000))
  }
  
  return(p)
}

# ============================================
# 6. (a) 双胞过滤后 QC 小提琴图
# ============================================
cat("\n--- Saving (a) Post-Doublet Violin ---\n")
p_a1 <- myVlnPlot(seurat_plot_vln, "nFeature_RNA", title = "Gene Count", ylabel = "nFeature_RNA")
p_a2 <- myVlnPlot(seurat_plot_vln, "nCount_RNA", title = "UMI Count", ylabel = "nCount_RNA")
p_a3 <- myVlnPlot(seurat_plot_vln, "percent.mt", title = "Mitochondrial %", ylabel = "percent.mt")

vln_pre <- (p_a1 + p_a2 + p_a3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1)) +
  plot_annotation(title = "Post-Doublet Removal QC Metrics", 
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "a_violin_postDoublet_byDonor.png"), width = 2000, height = 650, res = 150)
print(vln_pre)
dev.off()
cat("Saved: a_violin_postDoublet_byDonor.png\n")

# ============================================
# 7. (b) 散点图
# ============================================
cat("\n--- Saving (b) Post-Doublet Scatter ---\n")

scatter1 <- FeatureScatter(
  seurat_plot, feature1 = "nCount_RNA", feature2 = "nFeature_RNA",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("UMI vs Gene Count") + 
  theme_classic() +
  theme(legend.position = "none", plot.title = element_text(size = 12, face = "bold"))

scatter2 <- FeatureScatter(
  seurat_plot, feature1 = "nCount_RNA", feature2 = "percent.mt",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("UMI vs MT%") + 
  theme_classic() +
  theme(legend.position = "none", plot.title = element_text(size = 12, face = "bold"))

scatter3 <- FeatureScatter(
  seurat_plot, feature1 = "nFeature_RNA", feature2 = "percent.mt",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("Gene Count vs MT%") + 
  theme_classic() +
  theme(legend.position = "right", plot.title = element_text(size = 12, face = "bold"),
        legend.key.size = unit(0.3, "cm"), legend.text = element_text(size = 8))

scatter_pre <- (scatter1 + scatter2 + scatter3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1.2)) +
  plot_annotation(title = "Post-Doublet Removal Scatter Plots",
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "b_scatter_postDoublet.png"), width = 2000, height = 650, res = 150)
print(scatter_pre)
dev.off()
cat("Saved: b_scatter_postDoublet.png\n")

# ============================================
# 8. (c) QC 过滤
# ============================================
seurat_qc <- subset(seurat, 
                    nFeature_RNA > 700 & 
                    nCount_RNA < 100000 &
                    percent.mt < 5 & 
                    countFeatureRatio > 1.2)

cellNum1 <- ncol(seurat_qc)
cat("\nPost-QC:", cellNum1, "cells (removed", oriCellNum - cellNum1, 
    round((oriCellNum - cellNum1)/oriCellNum * 100, 1), "%)\n")

seurat_qc_vln <- subset(seurat_qc, nCount_RNA <= 25000)
cat("Post-QC Violin plot using", ncol(seurat_qc_vln), "cells (nCount_RNA <= 25000)\n")

# ============================================
# 9. (d) QC 过滤后小提琴图
# ============================================
cat("\n--- Saving (c) Post-QC Violin ---\n")

p_d1 <- myVlnPlot(seurat_qc_vln, "nFeature_RNA", title = "Gene Count", ylabel = "nFeature_RNA")
p_d2 <- myVlnPlot(seurat_qc_vln, "nCount_RNA", title = "UMI Count", ylabel = "nCount_RNA")
p_d3 <- myVlnPlot(seurat_qc_vln, "percent.mt", title = "Mitochondrial %", ylabel = "percent.mt")

vln_post <- (p_d1 + p_d2 + p_d3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1)) +
  plot_annotation(title = "Post-Doublet Removal + Post-QC Metrics",
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "c_violin_postQC_postDoublet.png"), width = 2000, height = 650, res = 150)
print(vln_post)
dev.off()
cat("Saved: c_violin_postQC_postDoublet.png\n")

# ============================================
# 10. 保存最终对象
# ============================================
saveRDS(seurat_qc, paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned_QCed.rds"))

# ============================================
# 11. 汇总
# ============================================
cat("\n========== Summary ==========\n")
cat("Original cells:        ", original_total, "\n")
cat("After doublet removal: ", after_doublet, "\n")
cat("After sample filter:   ", oriCellNum, "\n")
cat("After QC filter:       ", cellNum1, "\n")
cat("Total removed:         ", original_total - cellNum1, 
    "(", round((original_total - cellNum1)/original_total * 100, 2), "%)\n")
cat("Save directory:        ", save_dir, "\n")
cat("Final RDS:             ", paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned_QCed.rds"), "\n")
cat("==================================\n")

In [ ]:
# 确保 percent.mt 已计算
if (!"percent.mt" %in% colnames(seurat@meta.data)) {
  MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
  seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = MTGenes)
}

meta <- seurat@meta.data

p <- ggplot(meta, aes(x = nCount_RNA, y = nFeature_RNA, color = percent.mt)) +
  geom_point(size = 0.3, alpha = 0.5) +
  scale_color_distiller(palette = "Reds", direction = 1, name = "percent.mt (%)") +
  labs(x = "nCount_RNA", y = "nFeature_RNA") +
  theme_classic() +
  theme(
    axis.text = element_text(size = 11, color = "black"),
    axis.title = element_text(size = 12, face = "bold"),
    legend.title = element_text(size = 10),
    legend.text = element_text(size = 9)
  )

fig_dir <- "./data"
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

png(file.path(fig_dir, "scatter_nCount_nFeature_mt.png"), width = 800, height = 700, res = 150)
print(p)
dev.off()

cat("已保存:", file.path(fig_dir, "scatter_nCount_nFeature_mt.png"), "\n")

In [ ]:
# 第三步分大类+随机森林

In [ ]:
# Plot each clusters
myHighlightDimPlot = function(seuratObj, highlightBy, size=1, alpha=1, showAllPoints=FALSE, ...) {
    umapTb = as_tibble(seuratObj@reductions$umap@cell.embeddings)
    if (!identical(rownames(seuratObj@reductions$umap@cell.embeddings), rownames(seuratObj@meta.data)))
        stop("Error: cell names not identical!")
    umapTb = cbind(umapTb, seuratObj@meta.data)
    highlightByClass = sort(unique(umapTb[[highlightBy]]))
    gpList = lapply(highlightByClass, function(cl) {
        subUmapTb = subset(umapTb, umapTb[[highlightBy]] == cl)
        if (showAllPoints) {
            gp = ggplot() + 
                geom_point(data=umapTb, aes(x=UMAP_1, y=UMAP_2), color="grey", shape=16, size=size, alpha=alpha) +
                geom_point(data=subUmapTb, aes(x=UMAP_1, y=UMAP_2), color="red", shape=16, size=size, alpha=alpha) +
                xlim(min(umapTb$UMAP_1), max(umapTb$UMAP_1)) +
                ylim(min(umapTb$UMAP_2), max(umapTb$UMAP_2)) +
                coord_fixed() +
                theme_void() 
        } else {
            gp = ggplot() + 
                geom_point(data=umapTb, aes(x=UMAP_1, y=UMAP_2), color="grey", shape=16, size=size, alpha=alpha) +
                geom_point(data=subUmapTb, aes(x=UMAP_1, y=UMAP_2), color="red", shape=16, size=size, alpha=alpha) +
                xlim(min(umapTb$UMAP_1), max(umapTb$UMAP_1)) +
                ylim(min(umapTb$UMAP_2), max(umapTb$UMAP_2)) +
                coord_fixed() +
                theme_void()
        }
        return(gp)
    })
    gpAll = plot_grid(plotlist=gpList, labels=highlightByClass, ...)
    return(gpAll)
}

In [ ]:
# # 1. 先更新 Rcpp
# install.packages("Rcpp")

# # 2. 然后完全重启 R（必须！）
# # RStudio: Session -> Restart R
# # 或 Jupyter: Kernel -> Restart

In [ ]:
suppressMessages(library(tidyverse))
suppressMessages(library(parallel))
suppressMessages(library(Seurat))
suppressMessages(library(harmony))
suppressMessages(library(SeuratData))
suppressMessages(library(patchwork))
suppressMessages(library(Matrix))
suppressMessages(library(viridis))
suppressMessages(library(cowplot))
suppressMessages(library(ggsci))
suppressMessages(library(pheatmap))
suppressMessages(library(RColorBrewer))
suppressMessages(library(reshape2))
suppressMessages(library(LSD))
suppressMessages(library(readxl))
suppressMessages(library(ggrepel))
suppressMessages(library(randomForest))


In [ ]:
plan("multicore", workers = 12)
options(future.globals.maxSize = 500000 * 1024^5)

In [ ]:
readDir <- "./data/"
seurat1 <- readRDS(paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned_QCed.rds"))

In [ ]:
saveDir <- "./data"
dir.create(saveDir, showWarnings = FALSE, recursive = TRUE)

In [ ]:
classFiltSeurat=seurat1
classFiltSeurat

In [ ]:
classFiltSeurat = SCTransform(
classFiltSeurat, assay="RNA",
ncells=ncol(classFiltSeurat[["RNA"]]),
variable.features.n=5000,
return.only.var.genes = T,
vars.to.regress="percent.mt",
method="glmGamPoi"
)

In [ ]:
classFiltSeurat = classFiltSeurat %>%
RunPCA(npcs=100, verbose = FALSE) %>%
FindNeighbors(dims = 1:50) %>%
FindClusters(verbose = FALSE) %>%
RunUMAP(dims = 1:50)
saveRDS(classFiltSeurat,'./data/20260821batch_merge_batchharmony_pre.rds')

In [ ]:
classFiltSeurat = readRDS('./data/20260821batch_merge_batchharmony_pre.rds')
# Run harmony
DefaultAssay(classFiltSeurat) = "SCT"
options(repr.plot.width=6, repr.plot.height=2.5)

In [ ]:
classFiltSeurat@meta.data$donor = gsub('Q','q',classFiltSeurat@meta.data$donor)

In [ ]:
classFiltSeurat@meta.data

In [ ]:
classFiltSeurat = RunHarmony(
classFiltSeurat, group.by.vars="donor", plot_convergence=TRUE,
assay.use="SCT", reduction.use = "pca", dims.use=1:30
)

In [ ]:
classFiltSeurat <- classFiltSeurat %>%
  FindNeighbors(reduction = "harmony", dims = 1:30, k.param = 20) %>%   # 20 即可
  FindClusters(verbose = FALSE) %>%
  RunUMAP(reduction = "harmony", dims = 1:30, n.neighbors = 20)
saveRDS(classFiltSeurat,'./data/20260821harmonydonorbatch_merge_batchharmony.rds')

In [ ]:
seurat=readRDS('./data/20260821harmonydonorbatch_merge_batchharmony.rds')

In [ ]:
# seurat@meta.data
seurat@meta.data$donor = gsub('Q','q',seurat@meta.data$donor)
seurat@meta.data

In [ ]:
# 自定义绘图函数
myHighlightDimPlot <- function(seurat, highlightBy, showAllPoints = TRUE, size = 0.1) {
  # 提取 UMAP 数据
  umap_data <- as.data.frame(seurat@reductions$umap@cell.embeddings)
  umap_data$group <- seurat@meta.data[[highlightBy]]
  
  # 检查列名并调整
  colnames(umap_data)[1:2] <- c("UMAP_1", "UMAP_2")  # 确保列名正确
  
  # 获取唯一的 donor 列表
  donors <- unique(umap_data$group)
  
  # 创建存储图的列表
  plot_list <- list()
  
  # 遍历每个 donor，分别绘图
  for (donor in donors) {
    # 创建基础绘图
    p <- ggplot(umap_data, aes(x = UMAP_1, y = UMAP_2)) +
      theme_minimal() +
      theme(legend.position = "right")
    
    # 如果需要显示所有点
    if (showAllPoints) {
      p <- p + geom_point(data = umap_data, aes(x = UMAP_1, y = UMAP_2), color = "gray", size = size)
    }
    
    # 添加当前 donor 的高亮点
    p <- p + geom_point(data = subset(umap_data, group == donor), aes(x = UMAP_1, y = UMAP_2, color = group), size = size)
    
    # 添加标题
    p <- p + ggtitle(paste("Donor:", donor))
    
    # 将图添加到列表中
    plot_list[[donor]] <- p
  }
  
  # 使用 patchwork 包将两张图并排显示
  combined_plot <- wrap_plots(plot_list, ncol = 2)
  
  return(combined_plot)
}

In [ ]:
options(repr.plot.width=14, repr.plot.height=7)
myHighlightDimPlot(seurat, highlightBy="donor", showAllPoints=T, size=0.1)

In [ ]:
# 定义 selectMarkers 向量
selectMarkers = c(
    'SLC17A6',# Excit
    "GAD1", "GAD2", # Inhib,
    "GFAP",'EDNRB', # Astrocytes
    "FOXJ1",  "RSPH1",# Ependymal
    "TAGLN", "ALPHA-SMA", # VSMC
    'COL1A1','LUM',# VLMC
    'FLT1','RAMP2',# Endothelial
    "MCAM", "RGS5", # Pericytes
    "VCAN",'SOX6', # OPC
    "PLP1",'MOBP', # Oligo
    'C1QB','OLR1',# Microglia
    'Olig2',#Polydendrocytes
    'CD52',#Lymphocytes
    'MRC1',#pvMFs
    'TAGLN',#Smooth Muscle
    'TPH2','SLC6A4',#Serotonergic Neurons
    'DRD2','TH',#Dopaminergic Neurons
    'CCK','CARTPT',#Peptidergic Neurons
    'PPP1R1B','BCL11B','PDE1B',  
    'SLC5A7','CHAT',#choli
    "SNAP25"#Neuron
)

# 移除空元素
selectMarkers = selectMarkers[!is.null(selectMarkers) & selectMarkers != ""]

# 去重
selectMarkers = selectMarkers[!duplicated(selectMarkers)]

# 设置 Seurat 对象的标识
Idents(seurat) = "seurat_clusters"

# 设置绘图参数
options(repr.plot.height=10, repr.plot.width=60)

# 绘制点图
DotPlot(seurat, features = selectMarkers)

In [ ]:
seurat2=seurat 

In [ ]:
seurat2@meta.data$class='Uncertain'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(0,13,34),]$class='Astrocytes'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(2,3,4,7,8,14,22,25,32),]$class='Oligo'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(20),]$class='Endothelial'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(1,27,36,31,33),]$class='OPC'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(10),]$class='Microglia'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(5,6,9,11,15,17,19,18,23,24,28,26,29,30,35,37),]$class='Neurons'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(16),]$class='Ependymal'
# seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(11),]$class='5-HT'

In [ ]:
colnames(seurat2@meta.data)

In [ ]:
# # 1. 新建列，默认 "Unknown"
# seurat2@meta.data$subclass3 <- "Unknown"

# ## 2. 映射表（cluster → 小类）
# cls.map <- list(
#   Astrocytes  = c(1,18,20,36),
#   Oligo       = c(0,3,4,7,10,16),
#   Endothelial = 23,
#   OPC         = c(5,22,31,34),
#   Microglia   = c(9,17),
#   Neurons     = c(2,6,8,13,14,19,21,24,25,27,28,30,32,35),
#   `DA`        = 33,
#   `5-HT`      = c(11)
# )

# ## 3. 一次性写入
# for (nm in names(cls.map)) {
#   idx <- seurat2@meta.data$seurat_clusters %in% cls.map[[nm]]
#   if (sum(idx) > 0) seurat2@meta.data$subclass3[idx] <- nm
# }

# ## 4. 检查
# table(seurat2@meta.data$subclass3, useNA = "ifany")

In [ ]:
# 看大类
table(seurat2@meta.data$class)

# 看小类
# table(seurat2@meta.data$subclass3)

# 一起看
head(seurat2@meta.data[, c("seurat_clusters", "class")])

In [ ]:
df_test=seurat2@reductions$umap@cell.embeddings
df_test=as.data.frame(df_test)
df_test$X=rownames(df_test)

df_meta=seurat2@meta.data
df_meta=as.data.frame(df_meta)
df_meta$X=rownames(df_meta)
df_meta=df_meta[,c('nFeature_RNA','X','seurat_clusters')]

df_test=left_join(df_test,df_meta,by='X')


Idents(seurat2) = "seurat_clusters"
options(repr.plot.width=9, repr.plot.height=7)
DimPlot(seurat2, reduction="umap", label=T,raster=FALSE)+coord_fixed()

In [ ]:
classMarkerTop = c('SLC17A6','GAD1','GAD2','SNAP25','GFAP','PECAM1','FLT1','VCAN','MOBP','C1QC','FOXJ1','SLC6A4','TH')

options(repr.plot.width=20, repr.plot.height=15)
FeaturePlot(classFiltSeurat, features = classMarkerTop,raster=FALSE)
Idents(classFiltSeurat)='class'
DotPlot(classFiltSeurat, features = classMarkerTop)

In [ ]:
unique(seurat2@meta.data[seurat2@meta.data$class %in% 'Uncertain',]$seurat_clusters
)


In [ ]:
classFiltSeurat = RunHarmony(
classFiltSeurat, group.by.vars="donor", plot_convergence=TRUE,
assay.use="SCT", reduction.use = "pca", dims.use=1:30
)

In [ ]:
Idents(seurat2) = "class"
options(repr.plot.width=9, repr.plot.height=7)
DimPlot(seurat2, reduction="umap", label=T,raster=FALSE)+coord_fixed()

In [ ]:
classFiltSeurat=seurat1
classFiltSeurat

In [ ]:
# 统计每个类别的细胞数量
cell_counts <- table(seurat2@meta.data$class)

# 打印结果
print(cell_counts)

In [ ]:
# 每个class抽样80%，最多3000个细胞
sampleRatio = 0.8
maxCell = 3000
sampleDf = seurat2@meta.data[, c("class"), drop=F]
sampleList = split(sampleDf, sampleDf$class)
sampleList = lapply(sampleList, function(x) {
    sampleSize = as.integer(nrow(x)*0.8)
    if (sampleSize > maxCell) {
        sampleSize = maxCell
    }
    sampleIdx = 1:nrow(x)
    sampleIdx = sample(1:nrow(x), size=sampleSize, replace=F)
    return(x[sampleIdx,,drop=F])
})
sampleDf = Reduce(rbind, sampleList)
str(sampleDf)
table(sampleDf$class)

trainSeurat = seurat2[,rownames(sampleDf)]
table(trainSeurat$class)

In [ ]:
trainSeurat$class %>% unique()

In [ ]:
Idents(trainSeurat) = "class"
classMarker = FindAllMarkers(trainSeurat)

In [ ]:
#classMarker = read_tsv(paste0(dataDir, "/allSnRNAseq.soupx.mergeSeuratFiltlibFilt800mt5Ratio1d2.downsample5000ClassMarker.20220822.tsv"))
classMarker = subset(classMarker, p_val_adj < 0.01 & avg_log2FC > 1)
table(classMarker$cluster)

classMarkerTop = classMarker %>% group_by(cluster) %>% top_n(n = 20, wt = avg_log2FC)
options(repr.plot.width=length(unique(classMarkerTop$gene))/3, repr.plot.height=5)
Idents(seurat) = "class"

In [ ]:
seurat3 = seurat
# 查看当前的群体信息
# levels(Idents(seurat3))

# 重新设置群体的顺序，例如按特定顺序排序
new_levels <- c("Endothelial",'OPC','Microglia','Oligo','Astrocytes','Neurons','Uncertain')
Idents(seurat3) <- factor(Idents(seurat3), levels = new_levels)

# 验证新的群体顺序
levels(Idents(seurat3))

In [ ]:
trainDf = as.data.frame(t(as.matrix(trainSeurat[["SCT"]]@data[unique(classMarker$gene), ])))
trainDf$class = trainSeurat$class
trainDf = subset(trainDf, class != "Uncertain")
trainDf$class = factor(trainDf$class)
# We need to modify the name of the columns
colnames(trainDf) = paste0("col_", colnames(trainDf))
colnames(trainDf) = gsub("-", "_", colnames(trainDf))

rf = randomForest(col_class~., data=trainDf, ntree=1000)

In [ ]:
# 每个class抽样80%，最多3000个细胞
sampleRatio = 0.8
maxCell = 3000
sampleDf = seurat2@meta.data[, c("class"), drop=F]
sampleList = split(sampleDf, sampleDf$class)
sampleList = lapply(sampleList, function(x) {
    sampleSize = as.integer(nrow(x)*0.8)
    if (sampleSize > maxCell) {
        sampleSize = maxCell
    }
    sampleIdx = 1:nrow(x)
    sampleIdx = sample(1:nrow(x), size=sampleSize, replace=F)
    return(x[sampleIdx,,drop=F])
})
sampleDf = Reduce(rbind, sampleList)
str(sampleDf)
table(sampleDf$class)

trainSeurat = seurat2[,rownames(sampleDf)]
table(trainSeurat$class)

In [ ]:
dim(rf$confusion)
colnames(rf$confusion)
rownames(rf$confusion)

In [ ]:
# 加载必要的包
library(ggplot2)
library(pheatmap)

# 假设 rf 是你的随机森林模型对象
# rf <- readRDS(paste0(dataDir, "/allSnRNAseq.soupx.mergeSeuratFiltlibFilt800mt5Ratio1d2.class.randomForest.20220822.rds"))

# 查看混淆矩阵的结构
str(rf$confusion)

# 修改混淆矩阵中的行名和列名
# 假设混淆矩阵的行名和列名是分类标签
# rownames(rf$confusion) <- gsub("Endotheial", "Endothelial", rownames(rf$confusion))
# colnames(rf$confusion) <- gsub("Endotheial", "Endothelial", colnames(rf$confusion))

# 查看修改后的混淆矩阵
str(rf$confusion)

# 设置绘图参数
options(repr.plot.width=10, repr.plot.height=5)
plot(rf)

# 计算混淆矩阵的比例
confusionRatio <- t(apply(rf$confusion, 1, function(x) {x / sum(x)}))

# 设置热图参数
options(repr.plot.width=10, repr.plot.height=10)
pheatmap(confusionRatio, 
         breaks=seq(0, 1, 0.01), 
         cluster_rows=FALSE, 
         cluster_cols=FALSE, 
         border_color=NA, 
         cellwidth=30, 
         cellheight=30, 
         fontsize=15)

In [ ]:
#rf = readRDS(paste0(dataDir, "/allSnRNAseq.soupx.mergeSeuratFiltlibFilt800mt5Ratio1d2.class.randomForest.20220822.rds"))
str(rf$confusion)
options(repr.plot.width=10, repr.plot.height=5)
plot(rf)

confusionRatio = t(apply(rf$confusion, 1, function(x) {x/sum(x)}))
options(repr.plot.width=10, repr.plot.height=10)
pheatmap(confusionRatio, breaks=seq(0,1,0.01), cluster_rows=F, cluster_cols=F, border_color=NA, cellwidth=30, cellheight=30, fontsize=15)

In [ ]:
# 删除 class.error 列
confusionMatrix <- rf$confusion[, -which(colnames(rf$confusion) == "class.error")]

# 查看删除后的矩阵
str(confusionMatrix)
# 计算混淆矩阵的比例
confusionRatio = t(apply(confusionMatrix, 1, function(x) {x/sum(x)}))

# 绘制热图
pheatmap(confusionRatio, breaks=seq(0,1,0.01), cluster_rows=F, cluster_cols=F, border_color=NA, cellwidth=30, cellheight=30, fontsize=15)

In [ ]:
# 加载必要的库
library(ggplot2)
library(pheatmap)

# 假设 rf 是已经加载的随机森林分类结果
# 如果需要加载数据，可以取消注释以下代码
# dataDir <- "你的数据目录路径"
# rf <- readRDS(paste0(dataDir, "/allSnRNAseq.soupx.mergeSeuratFiltlibFilt800mt5Ratio1d2.class.randomForest.20220822.rds"))

# 查看混淆矩阵的结构
str(rf$confusion)

# 绘制随机森林的混淆矩阵图
options(repr.plot.width = 10, repr.plot.height = 5)
plot(rf)

# 去掉混淆矩阵中的 'class.error' 列
confusion_matrix <- rf$confusion
if ("class.error" %in% colnames(confusion_matrix)) {
  confusion_matrix <- confusion_matrix[, !colnames(confusion_matrix) %in% c("class.error")]
}

# 计算混淆矩阵的比率
confusionRatio <- t(apply(confusion_matrix, 1, function(x) { x / sum(x) }))

# 绘制混淆矩阵的热图，并显示相关性数值
options(repr.plot.width = 10, repr.plot.height = 10)
pheatmap(
  confusionRatio,
  breaks = seq(0, 1, 0.01), # 设置颜色分段
  cluster_rows = FALSE, # 不对行进行聚类
  cluster_cols = FALSE, # 不对列进行聚类
  border_color = NA, # 去除单元格边框颜色
  cellwidth = 30, # 单元格宽度
  cellheight = 30, # 单元格高度
  fontsize = 15, # 字体大小
  color = colorRampPalette(c("#e65200", "#fffde7"))(100), # 设置颜色渐变
  show_rownames = TRUE, # 显示行名
  show_colnames = TRUE, # 显示列名
  main = "Confusion Matrix Heatmap", # 设置标题
  legend = TRUE, # 显示图例
  display_numbers = TRUE, # 在单元格中显示数值
  number_format = "%.2f" # 数值格式，保留两位小数
)

In [ ]:
# Apply the trainer to filter cells
seuratCountDf = as.data.frame(t(as.matrix(seurat[["SCT"]]@data[unique(classMarker$gene), ])))
colnames(seuratCountDf) = paste0("col_", colnames(seuratCountDf))
colnames(seuratCountDf) = gsub("-", "_", colnames(seuratCountDf))
seuratClassPred = predict(rf, seuratCountDf, type="prob")
str(seuratClassPred)

seuratClassPredTb = tibble(
    cellId = rownames(seuratClassPred),
    maxPredClassScore = apply(seuratClassPred, 1, max),
    maxPredClass = apply(seuratClassPred, 1, function(x) {names(x)[which.max(x)]})
)
head(seuratClassPredTb)

summary(seuratClassPredTb$maxPredClassScore)
options(repr.plot.width=6, repr.plot.height=4)
plot(density(seuratClassPredTb$maxPredClassScore, adjust=0.1))

identical(seuratClassPredTb$cellId, rownames(seurat@meta.data))
tmpDf = as.data.frame(seuratClassPredTb[c("maxPredClassScore", "maxPredClass")])
rownames(tmpDf) = seuratClassPredTb$cellId

seurat = AddMetaData(seurat, tmpDf)

options(repr.plot.width=9, repr.plot.height=7)
FeaturePlot(seurat, reduction="umap", features="maxPredClassScore",raster=FALSE)+coord_fixed()


In [ ]:
# 1. 看被剔除的细胞是否集中在特定 cluster
seurat$passFilt <- seurat$maxPredClassScore >= 0.9
table(seurat$seurat_clusters, seurat$passFilt)

# 2. 看被剔除细胞的质量指标
VlnPlot(seurat, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        group.by = "passFilt", pt.size = 0)

In [ ]:
# 1. 看被剔除的细胞是否集中在特定 cluster
seurat$passFilt <- seurat$maxPredClassScore >= 0.8
table(seurat$seurat_clusters, seurat$passFilt)

# 2. 看被剔除细胞的质量指标
VlnPlot(seurat, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        group.by = "passFilt", pt.size = 0)

In [ ]:
Idents(seurat) = "seurat_clusters"
cellNum0 = ncol(seurat)
classFiltSeurat = subset(seurat, maxPredClassScore >= 0.8)
cellNum1 = ncol(classFiltSeurat)
message("Filt cell ratio: ", cellNum1/cellNum0)

In [ ]:
seurat
classFiltSeurat

In [ ]:
table(classFiltSeurat$maxPredClass)

In [ ]:
saveRDS(classFiltSeurat,'./data/20260821subclass3macaque_snrna_batch1_soupX-forest.rds')

In [ ]:
classFiltSeurat=readRDS('./data/20260821subclass3macaque_snrna_batch1_soupX-forest.rds')

In [ ]:
# seurat

# # 2. 看 meta.data 里有哪些列（重点找 class、subclass、maxPredClassScore、maxPredClass、passFilt 之类）
# colnames(seurat@meta.data)

# # 3. 当前总细胞数
# ncol(seurat)

# # 4. 大类注释分布（如果 class 这列还在）
# table(seurat@meta.data$class, useNA = "ifany")

# # 5. Uncertain 还有多少个、是哪些聚类
# table(seurat@meta.data[seurat@meta.data$class == "Uncertain", ]$seurat_clusters)
# # 或者直接看 Uncertain 的 subclass 分布：
# table(seurat@meta.data[seurat@meta.data$class == "Uncertain", ]$subclass, useNA = "ifany")

# # 6. 随机森林预测结果概览（如果存进去了）
# summary(seurat@meta.data$maxPredClassScore)
# table(seurat@meta.data$maxPredClass, useNA = "ifany")

# # 7. 如果已经按 0.8 过滤过，看过滤后的情况
# if ("passFilt" %in% colnames(seurat@meta.data)) {
#   table(seurat@meta.data$passFilt)
#   table(seurat@meta.data[seurat@meta.data$passFilt == FALSE, ]$seurat_clusters)
# }

In [ ]:
# ncol(seurat)                                   # 应该是 123,245（batch1）
# colnames(seurat@meta.data)                     # 确认有没有 class / maxPredClassScore
# table(seurat@meta.data$class, useNA = "ifany")
# table(seurat@meta.data[seurat@meta.data$class == "Uncertain", ]$seurat_clusters)

In [ ]:
# 第四步注释小类

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)   # ← 加这一行

In [ ]:
library(Seurat)
library(dplyr)

# ========== 1. 直接从 maxPredClass 提取 Neurons ==========
nd_seurat <- subset(classFiltSeurat, subset = maxPredClass == "Neurons")
cat("Neurons 细胞数:", ncol(nd_seurat), "\n")

# ========== 2. 重新找高变基因 + PCA（Neurons 内部专用） ==========
nd_seurat <- FindVariableFeatures(nd_seurat, selection.method = "vst", nfeatures = 3000)
nd_seurat <- RunPCA(nd_seurat, features = VariableFeatures(nd_seurat), npcs = 50, verbose = FALSE)

# ========== 3. 聚类 + UMAP ==========
nd_seurat <- FindNeighbors(nd_seurat, reduction = "pca", dims = 1:30)
nd_seurat <- FindClusters(nd_seurat, resolution = 0.2)   # 根据你的需要调
nd_seurat <- RunUMAP(nd_seurat, reduction = "pca", dims = 1:30)

cat("新聚类分布:\n")
print(table(nd_seurat$seurat_clusters))

# ========== 4. 命名为 Neurons_0, Neurons_1... ==========
clusters <- as.character(nd_seurat$seurat_clusters)
unique_clusters <- sort(unique(as.numeric(clusters)))

neuron_counter <- 0
cluster_map <- character(length(unique_clusters))
names(cluster_map) <- as.character(unique_clusters)

for(cl in as.character(unique_clusters)) {
  cluster_map[cl] <- paste0("Neurons_", neuron_counter)
  neuron_counter <- neuron_counter + 1
}

nd_seurat$Subclass_new <- unname(cluster_map[clusters])
cat("\n重命名后:\n")
print(table(nd_seurat$Subclass_new))

# ========== 5. 可视化 ==========
options(repr.plot.width = 14, repr.plot.height = 10)
DimPlot(nd_seurat, reduction = "umap", group.by = "Subclass_new",
        label = TRUE, repel = TRUE, raster = FALSE) + coord_fixed()

In [ ]:
# 1. 在 classFiltSeurat 里新建 cluster 列，默认用大类名
classFiltSeurat$cluster <- as.character(classFiltSeurat$maxPredClass)

# 2. 把 Neurons 细胞的小类标签写回去
nd_cells <- colnames(nd_seurat)
classFiltSeurat$cluster[nd_cells] <- as.character(nd_seurat$Subclass_new)

# 3. 检查
table(classFiltSeurat$cluster)

In [ ]:
output_path <- "./data/"

In [ ]:
print(output_path)

In [ ]:
# ========== 放在所有画图代码之前 ==========
dir.create(output_path, showWarnings = FALSE, recursive = TRUE)


In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)

# ========== 路径设置 ==========
output_path <- "./data/"
dir.create(output_path, showWarnings = FALSE, recursive = TRUE)

# ============================================
# 只提取 Neurons 小类（用之前做好的 nd_seurat）
# ============================================

# 确保 nd_seurat 存在且包含 Subclass_new
# 如果不确定，可以从 classFiltSeurat 重新提取：
# nd_seurat <- subset(classFiltSeurat, subset = grepl("^Neurons_", cluster))

cat("Neurons 细胞数:", ncol(nd_seurat), "\n")
cat("Neurons 小类数:", length(unique(nd_seurat$Subclass_new)), "\n")

# ============================================
# 1. UMAP 总览（Neurons 小类）
# ============================================
cat("\n--- 1. Neurons UMAP ---\n")

p_umap <- DimPlot(nd_seurat, reduction = "umap", group.by = "Subclass_new",
                  label = TRUE, repel = TRUE, raster = FALSE, pt.size = 0.3) +
  coord_fixed() +
  ggtitle("Neurons Subclusters") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14))

print(p_umap)
ggsave(file.path(output_path, "Neurons_only_UMAP.png"), 
       plot = p_umap, width = 14, height = 10, dpi = 300)

# ============================================
# 2. 经典标记基因
# ============================================
cat("\n--- 2. Classic Markers (Neurons only) ---\n")

marker_genes <- c("SLC17A6","SLC17A7" ,"SLC6A4", "TPH2", "GAD1", "GAD2", "TH", "SLC17A8", "SNAP25",'GFAP','PECAM1','FLT1','VCAN','MOBP','C1QC','FOXJ1')
avail_genes <- intersect(marker_genes, rownames(nd_seurat))

if (length(avail_genes) > 0) {
  
  # FeaturePlot（在 Neurons UMAP 上看）
  p_feature <- FeaturePlot(nd_seurat, features = avail_genes, ncol = 3,
                           pt.size = 0.3, cols = c("lightgrey", "darkblue"),
                           order = TRUE, raster = FALSE) &
    theme(plot.title = element_text(face = "bold.italic", size = 12))
  print(p_feature)
  ggsave(file.path(output_path, "Neurons_only_FeaturePlot.png"), 
         plot = p_feature, width = 15, height = 12, dpi = 300)
  
  # DotPlot（按 Neurons 小类分组）
  # 56个小类，y轴很长，需要加大高度
  p_dot_classic <- DotPlot(nd_seurat, features = avail_genes, 
                           group.by = "Subclass_new",
                           dot.scale = 8, cols = c("lightgrey", "darkblue")) +
    RotatedAxis() +
    ggtitle("Neurons Classic Markers") +
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 11, face = "italic"),
          axis.text.y = element_text(size = 6),   # 56个小类，字体缩小
          plot.title = element_text(hjust = 0.5, face = "bold", size = 13))
  print(p_dot_classic)
  ggsave(file.path(output_path, "Neurons_only_ClassicMarkers_DotPlot.png"), 
         plot = p_dot_classic, width = 10, height = 20, dpi = 300)  # 高度加大
  
  # VlnPlot（按 Neurons 小类分组）
  p_vln <- VlnPlot(nd_seurat, features = avail_genes, 
                   group.by = "Subclass_new",
                   pt.size = 0, ncol = 3) &
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 7),
          axis.text.y = element_text(size = 6),
          plot.title = element_text(face = "bold.italic", size = 12))
  print(p_vln)
  ggsave(file.path(output_path, "Neurons_only_ClassicMarkers_VlnPlot.png"), 
         plot = p_vln, width = 16, height = 24, dpi = 300)  # 高度加大
  
  cat("✅ Neurons 经典标记基因图已保存\n")
} else {
  cat("⚠️ 标记基因不在表达矩阵中\n")
}

In [ ]:
# ============================================
# 3. FindAllMarkers（Neurons 小类之间比较）
# ============================================
cat("\n--- 3. FindAllMarkers (Neurons only) ---\n")

DefaultAssay(nd_seurat) <- "RNA"
Idents(nd_seurat) <- "Subclass_new"

markers <- FindAllMarkers(
  nd_seurat,
  assay = "RNA",
  only.pos = TRUE,
  min.pct = 0.25,
  logfc.threshold = 0.25,
  verbose = TRUE
)
cat("找到 markers 总数:", nrow(markers), "\n")

# 保存
write.csv(markers, file.path(output_path, "Neurons_only_FindAllMarkers.csv"), row.names = FALSE)

# ============================================
# 4. Top20 Markers DotPlot
# ============================================
cat("\n--- 4. Top20 Markers DotPlot ---\n")

if (nrow(markers) > 0) {
  top20 <- markers %>%
    group_by(cluster) %>%
    slice_max(order_by = avg_log2FC, n = 20) %>%
    ungroup()
  
  top20_genes <- unique(top20$gene)
  top20_genes <- top20_genes[top20_genes %in% rownames(nd_seurat)]
  
  if (length(top20_genes) > 0) {
    gene_order <- top20 %>% 
      filter(gene %in% top20_genes) %>% 
      arrange(cluster) %>% 
      pull(gene) %>% 
      unique()
    
    p_dot_top20 <- DotPlot(
      nd_seurat,
      features = gene_order,
      group.by = "Subclass_new",
      dot.scale = 4,
      cols = c("lightgrey", "red")
    ) +
      RotatedAxis() +
      ggtitle("Neurons Top20 Markers per Subcluster") +
      theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 7),
        axis.text.y = element_text(size = 6),
        plot.title = element_text(hjust = 0.5, face = "bold", size = 12)
      )
    
    print(p_dot_top20)
    ggsave(
      file.path(output_path, "Neurons_only_Top20Markers_DotPlot.png"),
      plot = p_dot_top20,
      width = 50,
      height = 24,
      dpi = 300,
      limitsize = FALSE
    )
    cat("✅ Top20 Markers 点图已保存\n")
  }
}

In [ ]:
# ==========================================================================
# 神经元亚群 marker 小提琴图 —— 完整脚本
# 针对问题: VlnPlot 报 'S4SXP' 错误 + 非交互环境(headless)无图形设备
# 方案: FetchData 取数 + ggplot2 手动画 + ggsave 优先存 PDF
# 用法: 重启 R 会话后直接 source 或 Rscript 运行
# ==========================================================================

suppressPackageStartupMessages({
  library(Seurat)
  library(ggplot2)
  library(dplyr)
  library(tidyr)
})

# ============================ 0. 参数区（按需修改） ============================
group_col   <- "Subclass_new"
output_path <- "./data/"
markers_csv <- file.path(output_path, "Neurons_only_FindAllMarkers.csv")
# nd_seurat_rds <- "/path/to/nd_seurat.rds"   # 若对象不在内存，填路径并取消下一行注释
# nd_seurat <- readRDS(nd_seurat_rds)

dir.create(output_path, showWarnings = FALSE, recursive = TRUE)

# ============================ 1. 对象检查 ============================
if (!exists("nd_seurat")) stop("nd_seurat 不在内存，请先 readRDS")
stopifnot(group_col %in% colnames(nd_seurat@meta.data))
Idents(nd_seurat) <- group_col

cat("== 环境诊断 ==\n")
cat("交互式:", interactive(), " X11:", capabilities("X11"), " cairo:", capabilities("cairo"), "\n")
cat("细胞数:", ncol(nd_seurat), " 基因数:", nrow(nd_seurat), "\n")
cat("分组水平:", paste(sort(unique(nd_seurat@meta.data[[group_col]])), collapse = ", "), "\n")

if (!exists("markers")) {
  if (file.exists(markers_csv)) {
    markers <- read.csv(markers_csv, stringsAsFactors = FALSE)
    cat("从 CSV 读入 markers:", nrow(markers), "行\n")
  } else {
    stop("markers 不在内存，且找不到文件: ", markers_csv)
  }
}

# ============================ 2. marker panel 定义 ============================

# ---- 2.1 神经递质身份 panel ----
nt_markers <- c(
  "SLC17A6", "SLC17A7", "SLC17A8",   # Glut（SLC17A8 同时是 5-HT 共释放 marker）
  "GAD1",    "GAD2",    "SLC32A1",   # GABA
  "CHAT",    "SLC18A3", "SLC5A7",    # 胆碱能
  "TH",      "SLC6A3",  "NR4A2",     # 多巴胺能
  "TPH2",    "SLC6A4",  "DDC",       # 5-HT
  "DBH",     "SLC6A2"                # 去甲（LC 方向）
)

# ---- 2.2 邻区身份 marker panel ----
region_mk_list <- list(
  `5HT_raphe`   = c("TPH2","SLC6A4","FEV","SLC17A8","LMX1B","GATA2"),
  `PAG`         = c("PAX7","PAX8","ESR1","NTN1","DPP10","GUCY1A3"),
  `PBN`         = c("EGR2","PRLR","CALCA","NMB","DRD3"),
  `LC_NE`       = c("DBH","SLC6A2","PHOX2B","GATA3"),
  `VTA_SNc_DA`  = c("TH","NR4A2","PITX3","SLC6A3","SOX6","KCNJ6"),
  `ACh_IPN_MHb` = c("CHAT","SLC18A3","SLC5A7","GPR151","TACR3"),
  `Striatum`    = c("BCL11B","FOXP2","PTPN5","TAC1","ADORA2A","DRD1","DRD2"),
  `Ctx_Hip_GABA`= c("PVALB","SST","VIP","LAMP5","NPY")
)

# ============================ 3. 核心函数 ============================

# ---- 3.1 取数：基因表达 或 meta.data 列（含打分列）转长表 ----
get_expr_df <- function(features, obj = nd_seurat, gc = group_col) {
  dat <- tryCatch(
    FetchData(obj, vars = c(features, gc)),
    error = function(e) NULL
  )
  if (is.null(dat)) {
    # FetchData 失败时兜底：直接从表达矩阵取（此时只支持基因，不支持 meta 列）
    mat <- tryCatch(GetAssayData(obj, assay = "RNA", layer = "data"),
                    error = function(e) GetAssayData(obj, assay = "RNA", slot = "data"))
    genes <- intersect(features, rownames(mat))
    dat <- as.data.frame(t(as.matrix(mat[genes, , drop = FALSE])))
    dat[[gc]] <- obj@meta.data[[gc]]
  }
  feats_ok <- intersect(features, colnames(dat))
  dat %>%
    select(all_of(c(feats_ok, gc))) %>%
    pivot_longer(cols = all_of(feats_ok), names_to = "gene", values_to = "expr")
}

# ---- 3.2 画小提琴图并保存（PDF 必存，PNG 尝试） ----
plot_vln_manual <- function(features, file_base, ncol = 4, same.y = TRUE,
                            title = NULL, width = NULL, height = NULL) {
  df <- get_expr_df(features)
  genes <- intersect(features, unique(df$gene))
  if (length(genes) == 0) {
    cat("!! 无可用基因，跳过:", basename(file_base), "\n")
    return(invisible(NULL))
  }
  df <- df %>% mutate(gene = factor(gene, levels = genes))
  n_row <- ceiling(length(genes) / ncol)
  if (is.null(width))  width  <- 3.0 * ncol
  if (is.null(height)) height <- 2.8 * n_row

  p <- ggplot(df, aes(x = .data[[group_col]], y = expr, fill = .data[[group_col]])) +
    geom_violin(scale = "width", trim = TRUE) +
    facet_wrap(~gene, ncol = ncol, scales = ifelse(same.y, "fixed", "free_y")) +
    theme_bw(base_size = 8) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1),
          legend.position = "none",
          strip.background = element_rect(fill = "grey90")) +
    labs(x = NULL, y = "Expression", title = title)

  ggsave(paste0(file_base, ".pdf"), p, width = width, height = height, limitsize = FALSE)
  ok_png <- tryCatch({
    ggsave(paste0(file_base, ".png"), p, width = width, height = height,
           dpi = 300, limitsize = FALSE)
    TRUE
  }, error = function(e) FALSE)
  cat(">> 完成:", basename(file_base), "(pdf 已存", ifelse(ok_png, ", png 已存)", ")"), "\n")
  invisible(p)
}

# ---- 3.3 区域打分（AddModuleScore 失败时手动打分兜底） ----
add_scores_safe <- function(obj, sig_list) {
  obj <- tryCatch({
    obj <- AddModuleScore(obj, features = sig_list, name = "RS", seed = 1)
    for (i in seq_along(sig_list)) {
      obj@meta.data[[names(sig_list)[i]]] <- obj@meta.data[[paste0("RS", i)]]
    }
    cat("AddModuleScore 正常完成\n")
    obj
  }, error = function(e) {
    cat("!! AddModuleScore 失败，改手动打分:", conditionMessage(e), "\n")
    for (nm in names(sig_list)) {
      genes <- intersect(sig_list[[nm]], rownames(obj))
      if (length(genes) == 0) next
      expr <- FetchData(obj, vars = genes)
      obj@meta.data[[nm]] <- rowMeans(expr)
    }
    obj
  })
  obj
}

# ============================ 4. 正式画图 ============================

# ---- 4.1 递质身份总览 ----
cat("\n--- 4.1 递质身份小提琴图 ---\n")
plot_vln_manual(nt_markers,
                file.path(output_path, "Neurons_only_NeuroTransmitter_VlnPlot"),
                ncol = 4, same.y = TRUE,
                title = "Neurotransmitter identity")

# ---- 4.2 邻区 marker ----
cat("\n--- 4.2 邻区 marker 小提琴图 ---\n")
plot_vln_manual(unique(unlist(region_mk_list)),
                file.path(output_path, "Neurons_only_RegionMarkers_VlnPlot"),
                ncol = 5, same.y = FALSE,
                title = "Adjacent-region markers")

# ---- 4.3 区域打分 ----
cat("\n--- 4.3 区域打分 ---\n")
nd_seurat <- add_scores_safe(nd_seurat, region_mk_list)
plot_vln_manual(names(region_mk_list),
                file.path(output_path, "Neurons_only_RegionScore_VlnPlot"),
                ncol = 4, same.y = FALSE,
                title = "Region module scores")

# ---- 4.4 每个 subclass 的 Top20 marker ----
cat("\n--- 4.4 Top20 markers per subclass ---\n")
top20 <- markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20, with_ties = FALSE) %>%
  ungroup()

for (cl in sort(unique(top20$cluster))) {
  genes_cl <- top20 %>%
    filter(cluster == cl) %>%
    arrange(desc(avg_log2FC)) %>%
    pull(gene)
  ncol_use <- min(5, max(length(intersect(genes_cl, rownames(nd_seurat))), 1))
  nrow_use <- ceiling(length(intersect(genes_cl, rownames(nd_seurat))) / ncol_use)
  plot_vln_manual(genes_cl,
                  file.path(output_path, paste0("Neurons_only_Top20_VlnPlot_", cl)),
                  ncol = ncol_use, same.y = FALSE,
                  width = 3.2 * ncol_use, height = 3.2 * nrow_use,
                  title = paste0("Top20 markers of ", cl))
}

# ============================ 5. 完成确认 ============================
cat("\n== 全部完成，输出文件: ==\n")
print(list.files(output_path, pattern = "VlnPlot", full.names = TRUE))

In [ ]:
library(Seurat)
library(dplyr)
library(pheatmap)

group_col <- "Subclass_new"
clusters_check <- c("Neurons_5","Neurons_9","Neurons_10","Neurons_11","Neurons_12",
                    "Neurons_13","Neurons_14","Neurons_15","Neurons_16","Neurons_17",
                    "Neurons_18","Neurons_20","Neurons_21","Neurons_22","Neurons_8")

sigs <- list(
  Glut   = c("SLC17A6","SLC17A7"),
  GABA   = c("GAD1","GAD2","SLC32A1"),
  DA     = c("TH","SLC6A3","NR4A2","PITX3","LMX1A"),
  X5HT   = c("TPH2","SLC6A4","FEV","LMX1B","GATA2","SLC17A8","DDC"),
  NE     = c("DBH","SLC6A2","PHOX2B","GATA3"),
  ACh    = c("CHAT","SLC18A3","SLC5A7"),
  PAG        = c("PAX7","PAX8","ESR1","DPP10","NTN1"),
  PBN        = c("EGR2","PRLR","CALCA","NMB","PDYN"),
  RTN        = c("SHOX2","PHOX2B","MEIS1"),
  PonsOlive  = c("EBF2","EBF1","OTOF","NFIB","TAC1","CALB2"),
  Striatum   = c("PTPN5","PDE1B","TAC1","PENK","FOXP2","BCL11B","ADORA2A"),
  Oligo   = c("PLP1","MBP","OPALIN","MOBP","MAG"),
  Astro   = c("AQP4","SLC1A3","GJA1","ALDH1A1","SPARCL1"),
  Endo    = c("CLDN5","VWF","FLT1","PECAM1"),
  Micro   = c("AIF1","CSF1R","CX3CR1","C1QA","FCGR3A"),
  ChoriEpend = c("TTR","KRT8","KRT18","FOXJ1"),
  Pericyte= c("RGS5","ABCC9","KCNJ8","PDGFRB")
)

# ---- 平均表达（v4/v5 兼容）----
all_genes <- unique(unlist(sigs))
missing_genes <- setdiff(all_genes, rownames(nd_seurat))
if (length(missing_genes) > 0) {
  cat("以下基因不在 RNA assay，已自动从 signature 中剔除:",
      paste(missing_genes, collapse = ", "), "\n")
}

avg <- tryCatch(
  AverageExpression(nd_seurat, assays = "RNA", layer = "data",
                    features = all_genes, group.by = group_col)$RNA,
  error = function(e)
    AverageExpression(nd_seurat, assays = "RNA", slot = "data",
                      features = all_genes, group.by = group_col)$RNA
)

# ---- 关键修复：把 Seurat 换掉的 "_" 还原回来 ----
colnames(avg) <- gsub("-", "_", colnames(avg))

matched <- intersect(clusters_check, colnames(avg))
cat("匹配到 cluster:", length(matched), "/", length(clusters_check), "\n")
not_matched <- setdiff(clusters_check, colnames(avg))
if (length(not_matched) > 0) {
  cat("未匹配到（编号可能不存在或拼写不同）:", paste(not_matched, collapse = ", "), "\n")
}
stopifnot("至少匹配到一个 cluster" = length(matched) > 0)

avg <- avg[, matched, drop = FALSE]

# ---- z-score ----
m <- as.matrix(avg)
z <- t(scale(t(m)))
z[!is.finite(z)] <- 0   # 处理某些基因在所有 cluster 里表达相同导致的 NaN

# ---- signature 得分 ----
sig_score <- sapply(sigs, function(g) {
  g <- intersect(g, rownames(z))
  if (length(g) == 0) return(setNames(rep(0, ncol(z)), colnames(z)))
  v <- colMeans(z[g, , drop = FALSE], na.rm = TRUE)
  v[is.na(v)] <- 0
  v
})
sig_score <- t(sig_score)
sig_score[is.na(sig_score)] <- 0

# ---- 自动注释表 ----
transmitter <- c("Glut","GABA","DA","X5HT","NE","ACh")
region      <- c("PAG","PBN","RTN","PonsOlive","Striatum")
glial       <- c("Oligo","Astro","Endo","Micro","ChoriEpend","Pericyte")

anno <- lapply(colnames(sig_score), function(cl) {
  tibble(
    cluster      = cl,
    transmitter  = transmitter[which.max(sig_score[transmitter, cl])],
    region_hit   = region[which.max(sig_score[region, cl])],
    region_score = round(max(sig_score[region, cl]), 2),
    glial_hit    = glial[which.max(sig_score[glial, cl])],
    glial_score  = round(max(sig_score[glial, cl]), 2),
    verdict = case_when(
      max(sig_score[glial, cl]) > 1.0   ~ paste0("非神经元/双联体(", glial[which.max(sig_score[glial, cl])], ")"),
      max(sig_score[region, cl]) > 1.0   ~ paste0("疑", region[which.max(sig_score[region, cl])], "污染"),
      max(sig_score[transmitter, cl]) < 0 ~ "递质marker全阴，需人工看",
      TRUE                                ~ "raphe本地可能性大"
    )
  )
}) %>% bind_rows()

print(as.data.frame(anno))
write.csv(anno, file.path(output_path, "cluster_auto_annotation.csv"), row.names = FALSE)

# ---- 热图 ----
pdf(file.path(output_path, "cluster_signature_heatmap.pdf"), width = 11, height = 8)
pheatmap(sig_score, fontsize_row = 8, fontsize_col = 8,
         main = "Signature scores by cluster (row z-score)",
         cluster_cols = FALSE)
dev.off()
cat("完成，结果已保存\n")

In [ ]:
# # ============================================
# # 3. FindAllMarkers（Neurons 小类之间比较）
# # ============================================
# cat("\n--- 3. FindAllMarkers (Neurons only) ---\n")

# DefaultAssay(nd_seurat) <- "RNA"
# Idents(nd_seurat) <- "Subclass_new"

# markers <- FindAllMarkers(
#   nd_seurat,
#   assay = "RNA",
#   only.pos = TRUE,
#   min.pct = 0.25,
#   logfc.threshold = 0.25,
#   verbose = TRUE
# )
# cat("找到 markers 总数:", nrow(markers), "\n")

# # 保存
# write.csv(markers, file.path(output_path, "Neurons_only_FindAllMarkers.csv"), row.names = FALSE)

# # ============================================
# # 4. Top20 Markers DotPlot
# # ============================================
# cat("\n--- 4. Top20 Markers DotPlot ---\n")

# if (nrow(markers) > 0) {
#   top20 <- markers %>%
#     group_by(cluster) %>%
#     slice_max(order_by = avg_log2FC, n = 20) %>%
#     ungroup()
  
#   top20_genes <- unique(top20$gene)
#   top20_genes <- top20_genes[top20_genes %in% rownames(nd_seurat)]
  
#   if (length(top20_genes) > 0) {
#     gene_order <- top20 %>% 
#       filter(gene %in% top20_genes) %>% 
#       arrange(cluster) %>% 
#       pull(gene) %>% 
#       unique()
    
#     p_dot_top20 <- DotPlot(
#       nd_seurat,
#       features = gene_order,
#       group.by = "Subclass_new",
#       dot.scale = 4,
#       cols = c("lightgrey", "red")
#     ) +
#       RotatedAxis() +
#       ggtitle("Neurons Top20 Markers per Subcluster") +
#       theme(
#         axis.text.x = element_text(angle = 45, hjust = 1, size = 7),
#         axis.text.y = element_text(size = 6),
#         plot.title = element_text(hjust = 0.5, face = "bold", size = 12)
#       )
    
#     print(p_dot_top20)
#     ggsave(
#       file.path(output_path, "Neurons_only_Top20Markers_DotPlot.png"),
#       plot = p_dot_top20,
#       width = 50,
#       height = 24,
#       dpi = 300,
#       limitsize = FALSE
#     )
#     cat("✅ Top20 Markers 点图已保存\n")
#   }
# }

In [ ]:
# 根据marker得到Raphe 邻区（NE）	Neurons_22	NE	DBH, PHOX2A, SLC6A2, TH	蓝斑 LC 或 A1/A2 核团
# Raphe 邻区（ACh）	Neurons_13, 18	ACh	CHAT, SLC5A7	PPN/LDT 胆碱能核团
# Raphe 邻区（PAG）	Neurons_6, 8	PAG	PAX3, PAX7, OTP	导水管周围灰质

In [ ]:
library(Seurat)
library(harmony)

In [ ]:
library(Seurat)
library(dplyr)

# ========== 1. 定义要删除的 Neurons 亚类 ==========
remove_neurons <- c("Neurons_11", "Neurons_12", "Neurons_13", "Neurons_16", "Neurons_22","Neurons_20", "Neurons_9", "Neurons_5", "Neurons_14", "Neurons_10")

# ========== 2. 从 classFiltSeurat 中删除这些细胞 ==========
cells_to_remove <- colnames(classFiltSeurat)[classFiltSeurat$cluster %in% remove_neurons]
cat("将要删除的细胞数:", length(cells_to_remove), "\n")

# 创建新的 Seurat 对象（删除指定细胞）
classFiltSeurat_filt <- subset(classFiltSeurat, cells = setdiff(colnames(classFiltSeurat), cells_to_remove))

cat("删除前总细胞数:", ncol(classFiltSeurat), "\n")
cat("删除后总细胞数:", ncol(classFiltSeurat_filt), "\n")

# ========== 3. 查看删除后的 cluster 分布 ==========
cat("\n删除后的 cluster 分布:\n")
print(table(classFiltSeurat_filt$cluster))

# ========== 4. 重新对 Neurons 子集进行降维（可选，如果需要重新整合） ==========
# 提取 Neurons 细胞
neuron_clusters <- grep("^Neurons_", unique(classFiltSeurat_filt$cluster), value = TRUE)
nd_cells_filt <- colnames(classFiltSeurat_filt)[classFiltSeurat_filt$cluster %in% neuron_clusters]

nd_seurat_filt <- subset(classFiltSeurat_filt, cells = nd_cells_filt)

cat("\n剩余 Neurons 细胞数:", ncol(nd_seurat_filt), "\n")

# 重新找高变基因 + PCA
nd_seurat_filt <- FindVariableFeatures(nd_seurat_filt, selection.method = "vst", nfeatures = 3000)
nd_seurat_filt <- RunPCA(nd_seurat_filt, features = VariableFeatures(nd_seurat_filt), npcs = 50, verbose = FALSE)

# 新增：供体批次校正
nd_seurat_filt <- RunHarmony(nd_seurat_filt,
                             group.by.vars = "donor",
                             reduction.use = "pca",
                             dims.use = 1:30,
                             assay.use = "RNA",
                             verbose = FALSE)

nd_seurat_filt <- FindNeighbors(nd_seurat_filt, reduction = "harmony", dims = 1:30)
nd_seurat_filt <- FindClusters(nd_seurat_filt, resolution = 0.2)
nd_seurat_filt <- RunUMAP(nd_seurat_filt, reduction = "harmony", dims = 1:30)

# 重新命名为 Neurons_0, Neurons_1...（保持连续编号）
clusters <- as.character(nd_seurat_filt$seurat_clusters)
unique_clusters <- sort(unique(as.numeric(clusters)))

neuron_counter <- 0
cluster_map <- character(length(unique_clusters))
names(cluster_map) <- as.character(unique_clusters)

for(cl in as.character(unique_clusters)) {
  cluster_map[cl] <- paste0("Neurons_", neuron_counter)
  neuron_counter <- neuron_counter + 1}

nd_seurat_filt$Subclass_new <- unname(cluster_map[clusters])

cat("\n重新聚类后的分布:\n")
print(table(nd_seurat_filt$Subclass_new))

# ========== 5. 更新 classFiltSeurat_filt 中的 Neurons 标签 ==========
# 非 Neurons 细胞保持原标签
classFiltSeurat_filt$cluster_final <- as.character(classFiltSeurat_filt$cluster)

# 更新 Neurons 细胞的新标签
nd_cells_new <- colnames(nd_seurat_filt)
classFiltSeurat_filt$cluster_final[nd_cells_new] <- as.character(nd_seurat_filt$Subclass_new)

cat("\n最终 cluster 分布:\n")
print(table(classFiltSeurat_filt$cluster_final))

# ========== 6. 可视化 ==========
# Neurons 子集 UMAP
options(repr.plot.width = 12, repr.plot.height = 10)
DimPlot(nd_seurat_filt, reduction = "umap", group.by = "Subclass_new",
        label = TRUE, repel = TRUE, raster = FALSE) + coord_fixed()

# 全体细胞 UMAP（按最终 cluster 着色）
# 需要先对全体细胞重新跑 UMAP，或者直接用原来的
# 这里提供基于现有 PCA 重新跑 UMAP 的代码：
classFiltSeurat_filt <- RunUMAP(classFiltSeurat_filt, reduction = "pca", dims = 1:30)

options(repr.plot.width = 14, repr.plot.height = 10)
DimPlot(classFiltSeurat_filt, reduction = "umap", group.by = "cluster_final",
        label = TRUE, repel = TRUE, raster = FALSE) + coord_fixed()



In [ ]:
# ========== 7. 保存对象（可选） ==========
saveRDS(classFiltSeurat_filt, file = "classFiltSeurat_filt.rds")
saveRDS(nd_seurat_filt, file = "nd_seurat_filt.rds")

In [ ]:
library(Seurat)
library(ggplot2)
library(patchwork)

# ========== 基因列表 ==========
genes <- c("SLC17A6", "SLC17A7", "SLC6A4", "TPH2", "GAD1", "GAD2", 
           "TH", "SLC17A8", "SNAP25", "GFAP", "PECAM1", "FLT1", 
           "VCAN", "MOBP", "C1QC", "FOXJ1")

# 检查哪些基因存在于数据中
genes_avail <- genes[genes %in% rownames(nd_seurat_filt)]
genes_missing <- setdiff(genes, rownames(nd_seurat_filt))
if (length(genes_missing) > 0) {
  cat("以下基因在数据中不存在（请检查大小写或物种）:", paste(genes_missing, collapse = ", "), "\n")
}
cat("可用基因:", paste(genes_avail, collapse = ", "), "\n")

# ============================================================
# 一、Neurons 子集 (nd_seurat_filt) 的图
# ============================================================

# -------- 1.1 DotPlot (按 Neurons 亚群) --------
p_dot_neurons <- DotPlot(nd_seurat_filt, features = genes_avail, group.by = "Subclass_new") +
  coord_flip() +
  scale_color_gradient2(low = "blue", mid = "white", high = "red", midpoint = 0) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  ggtitle("Neurons Subclusters - Marker DotPlot")

print(p_dot_neurons)

In [ ]:
# -------- 1.2 FeaturePlot (UMAP) --------
# 分两组画避免太多图挤在一起
p_feat_neurons_1 <- FeaturePlot(nd_seurat_filt, features = genes_avail[1:8], 
                                  cols = c("lightgrey", "darkred"), 
                                  raster = FALSE, ncol = 4)
p_feat_neurons_2 <- FeaturePlot(nd_seurat_filt, features = genes_avail[9:length(genes_avail)], 
                                  cols = c("lightgrey", "darkred"), 
                                  raster = FALSE, ncol = 4)

print(p_feat_neurons_1)
print(p_feat_neurons_2)

# -------- 1.3 5-HT 核心 marker 单独展示 --------
ht_core <- c("TPH2", "SLC6A4")
if (all(ht_core %in% genes_avail)) {
  p_ht <- FeaturePlot(nd_seurat_filt, features = ht_core, 
                      cols = c("lightgrey", "darkred"), 
                      raster = FALSE, ncol = 2) &
    theme(plot.title = element_text(face = "bold"))
  print(p_ht)
}


In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)

# ============================================================
# 步骤 1：在 nd_seurat_filt 中标记 5-HT vs non-5-HT-neurons
# ============================================================
ht_clusters <- c("Neurons_4", "Neurons_10", "Neurons_16")

nd_seurat_filt$neuron_type <- ifelse(nd_seurat_filt$Subclass_new %in% ht_clusters,
                                     "5-HT", "non-5-HT-neurons")

cat("Neurons 分类结果:\n")
print(table(nd_seurat_filt$neuron_type))

# 可视化确认
DimPlot(nd_seurat_filt, group.by = "neuron_type", label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT vs non-5-HT-neurons")

In [ ]:
# ============================================================
# 步骤 2：将标签写回 classFiltSeurat_filt
# ============================================================
classFiltSeurat_filt$cluster_final <- as.character(classFiltSeurat_filt$cluster)

# 更新 Neurons 细胞的标签
nd_cells_new <- colnames(nd_seurat_filt)
classFiltSeurat_filt$cluster_final[nd_cells_new] <- nd_seurat_filt$neuron_type

cat("\n全体细胞最终分类:\n")
print(table(classFiltSeurat_filt$cluster_final))



In [ ]:
# ============================================================
# 步骤 3：分别提取 5-HT 和 non-5-HT-neurons 并单独重新聚类
# ============================================================

# -------- 3.1 提取 5-HT 神经元并单独聚类 --------
ht_seurat <- subset(nd_seurat_filt, subset = neuron_type == "5-HT")
cat("\n5-HT 神经元数:", ncol(ht_seurat), "\n")

# 重新找高变基因 + PCA + 聚类 + UMAP
ht_seurat <- FindVariableFeatures(ht_seurat, selection.method = "vst", nfeatures = 3000)
ht_seurat <- RunPCA(ht_seurat, features = VariableFeatures(ht_seurat), npcs = 50, verbose = FALSE)
ht_seurat <- FindNeighbors(ht_seurat, reduction = "pca", dims = 1:30)
ht_seurat <- FindClusters(ht_seurat, resolution = 0.5)  # 可适当调整
ht_seurat <- RunUMAP(ht_seurat, reduction = "pca", dims = 1:30)

# 重命名为 5-HT_0, 5-HT_1...
ht_clusters_new <- as.character(ht_seurat$seurat_clusters)
ht_unique <- sort(unique(as.numeric(ht_clusters_new)))

ht_counter <- 0
ht_map <- character(length(ht_unique))
names(ht_map) <- as.character(ht_unique)

for(cl in as.character(ht_unique)) {
  ht_map[cl] <- paste0("5-HT_", ht_counter)
  ht_counter <- ht_counter + 1
}

ht_seurat$ht_subtype <- unname(ht_map[ht_clusters_new])

cat("\n5-HT 亚群分布:\n")
print(table(ht_seurat$ht_subtype))

# 5-HT 可视化
p_ht_umap <- DimPlot(ht_seurat, group.by = "ht_subtype", label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT Neurons Subtypes")

p_ht_vln <- VlnPlot(ht_seurat, features = c("TPH2", "SLC6A4", "SLC17A6", "GAD1", "TH"),
                    pt.size = 0, ncol = 3)

print(p_ht_umap)
print(p_ht_vln)


In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)

# ============================================================
# 1. UMAP 图（按 5-HT 亚群）
# ============================================================
options(repr.plot.width = 10, repr.plot.height = 8)

p_ht_umap <- DimPlot(ht_seurat, group.by = "ht_subtype", 
                     label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("5-HT Neurons Subtypes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))

print(p_ht_umap)



In [ ]:
library(Seurat)
library(ggplot2)

# ============================================================
# 5-HT UMAP - 无标签版本
# ============================================================
options(repr.plot.width = 10, repr.plot.height = 8)

p_ht_umap_nolabel <- DimPlot(ht_seurat, 
                             group.by = "ht_subtype", 
                             label = FALSE,           # 关闭标签
                             raster = FALSE) +
  coord_fixed() + 
  ggtitle("5-HT Neurons Subtypes (No Label)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        legend.position = "right")  # 保留右侧图例，方便辨认颜色

print(p_ht_umap_nolabel)

In [ ]:
# ============================================================
# 2. 找 Top 20 Marker 基因
# ============================================================
ht_seurat <- SetIdent(ht_seurat, value = "ht_subtype")

ht_markers <- FindAllMarkers(ht_seurat, 
                             only.pos = TRUE, 
                             min.pct = 0.25, 
                             logfc.threshold = 0.25)

top20 <- ht_markers %>%
  group_by(cluster) %>%
  top_n(n = 20, wt = avg_log2FC) %>%
  arrange(cluster, desc(avg_log2FC))

cat("Top 20 markers per 5-HT subtype:\n")
print(top20 %>% select(gene, cluster, avg_log2FC, p_val_adj) %>% head(40))

write.csv(top20, "ht_top20_markers.csv", row.names = FALSE)


In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

# ============================================
# 0. 参数设置
# ============================================
output_path <- "./data/"

# ============================================
# 1. 确保 5-HT 子集 & 因子顺序固定
# ============================================
# 你的数据已有 ht_subtype: 5-HT_0 ~ 5-HT_8
ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, 
                                levels = paste0("5-HT_", 0:13))

# 设置 ident 为 ht_subtype，用于 FindAllMarkers
ht_seurat <- SetIdent(ht_seurat, value = "ht_subtype")

cat("5-HT 亚群分布:\n")
print(table(ht_seurat$ht_subtype))

In [ ]:
# ============================================
# 2. 找 Marker 基因 (Wilcoxon)
# ============================================
ht_markers <- FindAllMarkers(ht_seurat,
                             only.pos = TRUE,
                             min.pct = 0.25,
                             logfc.threshold = 0.25,
                             test.use = "wilcox")

# Top 20 保存
top20 <- ht_markers %>%
  group_by(cluster) %>%
  top_n(n = 20, wt = avg_log2FC) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(top20, paste0(output_path, "ht_top20_markers.csv"), row.names = FALSE)
cat("Top 20 markers 已保存\n")

# ============================================
# 3. 提取每个亚型的 Top5 差异基因
# ============================================
top5_genes <- ht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

cat("\n每个亚型 Top5 差异基因:\n")
print(top5_genes %>% select(cluster, gene, avg_log2FC, p_val_adj))

# 基因顺序（按 5-HT_0 → 5-HT_8 排列，去重）
gene_order <- top5_genes %>%
  arrange(cluster) %>%
  pull(gene) %>%
  unique()

cat("\n点图基因顺序 (按 5-HT_0→8 排列), 共", length(gene_order), "个基因:\n")
cat(paste(gene_order, collapse = ", "), "\n")

# ============================================
# 4. Top5 DotPlot
# ============================================
p_dot <- DotPlot(
  ht_seurat,
  features = gene_order,
  group.by = "ht_subtype",         # 使用 ht_subtype
  dot.scale = 5,                   # 基因多，点稍小
  cols = c("lightgrey", "red")
) +
  coord_flip() +
  RotatedAxis() +
  ggtitle("5-HT 亚型 Top5 Marker 基因表达点图") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),      # 基因多，字体缩小防重叠
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )

print(p_dot)

ggsave(
  filename = paste0(output_path, "5HT_Top5_Markers_DotPlot.png"),
  plot = p_dot,
  width = 14,      # 9个亚群+45个基因，需要更宽
  height = 16,     # 需要更高
  dpi = 300
)
cat("\nTop5 点图已保存!\n")

# 保存表格
write.csv(
  top5_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
  file = paste0(output_path, "5HT_Top5_Markers_table.csv"),
  row.names = FALSE
)

In [ ]:
# ============================================
# 5. 小提琴图：nFeature_RNA
# ============================================
plot_data <- ht_seurat@meta.data
plot_data$ht_subtype <- factor(plot_data$ht_subtype, levels = paste0("5-HT_", 0:13))

p_nfeature <- ggplot(plot_data, aes(x = ht_subtype, y = nFeature_RNA, fill = ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  labs(x = "", y = "nFeature RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_blank(),
    axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"),
    axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 13),
    legend.position = "none"
  )

print(p_nfeature)

png(paste0(output_path, "violin_5HT_nFeature_RNA.png"), width = 1200, height = 800, res = 150)
print(p_nfeature)
dev.off()

pdf(paste0(output_path, "violin_5HT_nFeature_RNA.pdf"), width = 12, height = 8)
print(p_nfeature)
dev.off()
cat("nFeature_RNA 小提琴图已保存\n")

In [ ]:
# ============================================
# 6. 小提琴图：nCount_RNA
# ============================================
p_ncount <- ggplot(plot_data, aes(x = ht_subtype, y = nCount_RNA, fill = ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  labs(x = "", y = "nCount RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_blank(),
    axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"),
    axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 13),
    legend.position = "none"
  )

print(p_ncount)

png(paste0(output_path, "violin_5HT_nCount_RNA.png"), width = 1200, height = 800, res = 150)
print(p_ncount)
dev.off()

pdf(paste0(output_path, "violin_5HT_nCount_RNA.pdf"), width = 12, height = 8)
print(p_ncount)
dev.off()
cat("nCount_RNA 小提琴图已保存\n")

In [ ]:
# ============================================
# 7. Ranger + 热图（只在对角线显示数值）
# ============================================
expr_matrix <- GetAssayData(ht_seurat, layer = "data")
genes <- VariableFeatures(ht_seurat)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(ht_seurat$ht_subtype)

rf_ht <- ranger(
  x = X,
  y = y,
  num.trees = 500,
  num.threads = 8,
  classification = TRUE
)

cat("\nOOB Error:", rf_ht$prediction.error, "\n")

conf_mat <- rf_ht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("对角线准确率:\n")
print(round(diag(confusionRatio), 2))

# 过滤细胞数 >= 20 的亚群
df_num <- as.data.frame(table(ht_seurat$ht_subtype))
select_cell <- as.character(df_num[df_num$Freq >= 20, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))

mat_to_plot <- confusionRatio[select_cell, select_cell, drop = FALSE]

# 只在对角线显示数值
num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

# 屏幕显示
options(repr.plot.width = 10, repr.plot.height = 8)
pheatmap(
  mat_to_plot,
  breaks = seq(0, 1, 0.01),
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  border_color = NA,
  cellwidth = 45,
  cellheight = 45,
  fontsize = 14,
  color = viridis(100),
  main = "5-HT Subtypes Ranger Confusion Matrix",
  display_numbers = num_labels,
  fontsize_number = 12,
  number_color = "white"
)

# 保存 PNG
png(paste0(output_path, "5HT_Ranger_Heatmap.png"), width = 1200, height = 1000, res = 150)
pheatmap(
  mat_to_plot,
  breaks = seq(0, 1, 0.01),
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  border_color = NA,
  cellwidth = 45,
  cellheight = 45,
  fontsize = 14,
  color = viridis(100),
  main = "5-HT Subtypes Ranger Confusion Matrix",
  display_numbers = num_labels,
  fontsize_number = 12,
  number_color = "white"
)
dev.off()
cat("Ranger 热图已保存\n")


In [ ]:
# library(Seurat)
# library(ggplot2)
# library(dplyr)
# library(ranger)
# library(pheatmap)
# library(viridis)

# # ============================================
# # 0. 参数设置
# # ============================================
# output_path <- "/mnt/90-connectome/Personal/XQ/a-scientific data投稿数据+代码/第五步数据保存829/"

# # ============================================
# # 1. 确保 5-HT 子集 & 因子顺序固定 (5-HT_0 ~ 5-HT_8)
# # ============================================
# ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = paste0("5-HT_", 0:8))
# ht_seurat <- SetIdent(ht_seurat, value = "ht_subtype")

# # ============================================
# # 2. 找 Marker 基因 (Wilcoxon) — 如果已跑过可跳过
# # ============================================
# ht_markers <- FindAllMarkers(ht_seurat,
#                              only.pos = TRUE,
#                              min.pct = 0.25,
#                              logfc.threshold = 0.25,
#                              test.use = "wilcox")

# # ============================================
# # 3. 提取每个亚型的 Top10 差异基因
# # ============================================
# top10_genes <- ht_markers %>%
#   group_by(cluster) %>%
#   slice_max(order_by = avg_log2FC, n = 10) %>%
#   ungroup() %>%
#   arrange(cluster)

# cat("每个亚型 Top10 差异基因:\n")
# print(top10_genes %>% select(cluster, gene, avg_log2FC, p_val_adj))

# # 基因顺序（按 5-HT_0 → 5-HT_8 排列，去重）
# gene_order <- top10_genes %>%
#   arrange(cluster) %>%
#   pull(gene) %>%
#   unique()

# cat("\n点图基因顺序 (按 5-HT_0→8 排列), 共", length(gene_order), "个基因:\n")

# # ============================================
# # 4. Top10 DotPlot
# # ============================================
# p_dot10 <- DotPlot(
#   ht_seurat,
#   features = gene_order,
#   group.by = "ht_subtype",
#   dot.scale = 4,                   # 基因多，点更小
#   cols = c("lightgrey", "red")
# ) +
#   coord_flip() +
#   RotatedAxis() +
#   ggtitle("5-HT 亚型 Top10 Marker 基因表达点图") +
#   theme(
#     axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
#     axis.text.y = element_text(size = 7),      # 90个基因，字体再缩小
#     plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
#   )

# print(p_dot10)

# ggsave(
#   filename = paste0(output_path, "5HT_Top10_Markers_DotPlot.png"),
#   plot = p_dot10,
#   width = 16,      # 9个亚群，需要更宽
#   height = 22,     # 约90个基因，需要很高
#   dpi = 300
# )
# cat("Top10 点图已保存!\n")

# # 保存表格
# write.csv(
#   top10_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
#   file = paste0(output_path, "5HT_Top10_Markers_table.csv"),
#   row.names = FALSE
# )

# # ============================================
# # 5. 小提琴图：nFeature_RNA（刻度每 1000 一个）
# # ============================================
# plot_data <- ht_seurat@meta.data
# plot_data$ht_subtype <- factor(plot_data$ht_subtype, levels = paste0("5-HT_", 0:8))

# # 计算合适的刻度范围
# nfeature_max <- max(plot_data$nFeature_RNA, na.rm = TRUE)
# nfeature_breaks <- seq(0, ceiling(nfeature_max / 1000) * 1000, by = 1000)

# p_nfeature <- ggplot(plot_data, aes(x = ht_subtype, y = nFeature_RNA, fill = ht_subtype)) +
#   geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
#   scale_y_continuous(breaks = nfeature_breaks) +   # 刻度每 1000 一个
#   labs(x = "", y = "nFeature RNA") +
#   theme_bw() +
#   theme(
#     panel.grid.major = element_blank(),
#     panel.grid.minor = element_blank(),
#     panel.border = element_blank(),
#     axis.line = element_line(color = "black"),
#     axis.ticks = element_line(color = "black"),
#     axis.ticks.length = unit(6, "pt"),
#     axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
#     axis.text.y = element_text(size = 10),
#     axis.title = element_text(size = 13),
#     legend.position = "none"
#   )

# print(p_nfeature)

# png(paste0(output_path, "violin_5HT_nFeature_RNA_1k.png"), width = 1200, height = 800, res = 150)
# print(p_nfeature)
# dev.off()

# pdf(paste0(output_path, "violin_5HT_nFeature_RNA_1k.pdf"), width = 12, height = 8)
# print(p_nfeature)
# dev.off()
# cat("nFeature_RNA 小提琴图 (刻度1000) 已保存\n")

# # ============================================
# # 6. 小提琴图：nCount_RNA
# # ============================================
# p_ncount <- ggplot(plot_data, aes(x = ht_subtype, y = nCount_RNA, fill = ht_subtype)) +
#   geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
#   labs(x = "", y = "nCount RNA") +
#   theme_bw() +
#   theme(
#     panel.grid.major = element_blank(),
#     panel.grid.minor = element_blank(),
#     panel.border = element_blank(),
#     axis.line = element_line(color = "black"),
#     axis.ticks = element_line(color = "black"),
#     axis.ticks.length = unit(6, "pt"),
#     axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
#     axis.text.y = element_text(size = 10),
#     axis.title = element_text(size = 13),
#     legend.position = "none"
#   )

# print(p_ncount)

# png(paste0(output_path, "violin_5HT_nCount_RNA.png"), width = 1200, height = 800, res = 150)
# print(p_ncount)
# dev.off()

# pdf(paste0(output_path, "violin_5HT_nCount_RNA.pdf"), width = 12, height = 8)
# print(p_ncount)
# dev.off()
# cat("nCount_RNA 小提琴图已保存\n")

# # ============================================
# # 7. Ranger + 热图（不带数值版本）
# # ============================================
# expr_matrix <- GetAssayData(ht_seurat, layer = "data")
# genes <- VariableFeatures(ht_seurat)

# X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
# y <- factor(ht_seurat$ht_subtype)

# rf_ht <- ranger(
#   x = X,
#   y = y,
#   num.trees = 500,
#   num.threads = 8,
#   classification = TRUE
# )

# cat("\nOOB Error:", rf_ht$prediction.error, "\n")

# conf_mat <- rf_ht$confusion.matrix
# confusionRatio <- t(apply(conf_mat, 1, function(x) {
#   if(sum(x) == 0) return(rep(0, length(x)))
#   x / sum(x)
# }))

# cat("对角线准确率:\n")
# print(round(diag(confusionRatio), 2))

# # 过滤细胞数 >= 20 的亚群
# df_num <- as.data.frame(table(ht_seurat$ht_subtype))
# select_cell <- as.character(df_num[df_num$Freq >= 20, ]$Var1)
# select_cell <- intersect(select_cell, rownames(confusionRatio))

# mat_to_plot <- confusionRatio[select_cell, select_cell, drop = FALSE]

# # ---------- 7.1 带数值版本（保留）----------
# num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
# diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

# png(paste0(output_path, "5HT_Ranger_Heatmap_withNum.png"), width = 1200, height = 1000, res = 150)
# pheatmap(
#   mat_to_plot,
#   breaks = seq(0, 1, 0.01),
#   cluster_rows = FALSE,
#   cluster_cols = FALSE,
#   border_color = NA,
#   cellwidth = 45,
#   cellheight = 45,
#   fontsize = 14,
#   color = viridis(100),
#   main = "5-HT Subtypes Ranger Confusion Matrix",
#   display_numbers = num_labels,
#   fontsize_number = 12,
#   number_color = "white"
# )
# dev.off()
# cat("Ranger 热图 (带数值) 已保存\n")

# # ---------- 7.2 不带数值版本（新增）----------
# # ---------- 7.2 不带数值版本（新增）----------
# png(paste0(output_path, "5HT_Ranger_Heatmap_noNum.png"), width = 1200, height = 1000, res = 150)
# pheatmap(
#   mat_to_plot,
#   breaks = seq(0, 1, 0.01),
#   cluster_rows = FALSE,
#   cluster_cols = FALSE,
#   border_color = NA,
#   cellwidth = 45,
#   cellheight = 45,
#   fontsize = 14,
#   color = viridis(100),
#   main = "5-HT Subtypes Ranger Confusion Matrix",
#   display_numbers = FALSE      # 不显示数值
# )
# dev.off()
# cat("Ranger 热图 (不带数值) 已保存\n")

# # 屏幕显示不带数值版本
# pheatmap(
#   mat_to_plot,
#   breaks = seq(0, 1, 0.01),
#   cluster_rows = FALSE,
#   cluster_cols = FALSE,
#   border_color = NA,
#   cellwidth = 45,
#   cellheight = 45,
#   fontsize = 14,
#   color = viridis(100),
#   main = "5-HT Subtypes Ranger Confusion Matrix (No Numbers)",
#   display_numbers = FALSE
# )

# cat("\n===== 全部完成 =====\n")


In [ ]:
# library(Seurat)
# library(ggplot2)
# library(dplyr)

# output_path <- "/mnt/90-connectome/Personal/XQ/a-scientific data投稿数据+代码/第五步数据保存829/"

# # ============================================
# # 1. 确保因子顺序 & ident 设置
# # ============================================
# ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = paste0("5-HT_", 0:11))
# ht_seurat <- SetIdent(ht_seurat, value = "ht_subtype")

# # ============================================
# # 2. 提取每个亚型的 Top20 差异基因
# # ============================================
# top20_genes <- ht_markers %>%
#   group_by(cluster) %>%
#   slice_max(order_by = avg_log2FC, n = 20) %>%
#   ungroup() %>%
#   arrange(cluster)

# cat("每个亚型 Top20 差异基因:\n")
# print(top20_genes %>% select(cluster, gene, avg_log2FC, p_val_adj))

# # 基因顺序（按 5-HT_0 → 5-HT_8 排列，去重）
# gene_order <- top20_genes %>%
#   arrange(cluster) %>%
#   pull(gene) %>%
#   unique()

# cat("\n点图基因总数:", length(gene_order), "\n")

# # ============================================
# # 3. Top20 DotPlot
# # ============================================
# p_dot20 <- DotPlot(
#   ht_seurat,
#   features = gene_order,
#   group.by = "ht_subtype",
#   dot.scale = 3,                   # 基因很多，点要小
#   cols = c("lightgrey", "red")
# ) +
#   coord_flip() +
#   RotatedAxis() +
#   ggtitle("5-HT 亚型 Top20 Marker 基因表达点图") +
#   theme(
#     axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
#     axis.text.y = element_text(size = 5.5),     # ~180个基因，字体必须很小
#     plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
#     panel.grid.major = element_blank(),
#     panel.grid.minor = element_blank()
#   )

# print(p_dot20)

# ggsave(
#   filename = paste0(output_path, "5HT_Top20_Markers_DotPlot.png"),
#   plot = p_dot20,
#   width = 18,      # 9个亚群，横轴需要很宽
#   height = 32,     # ~180个基因，纵轴需要很高
#   dpi = 300
# )
# cat("Top20 点图已保存!\n")

# # 保存表格
# write.csv(
#   top20_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
#   file = paste0(output_path, "5HT_Top20_Markers_table.csv"),
#   row.names = FALSE
# )
# cat("Top20 差异基因表格已保存!\n")

In [ ]:
# library(Seurat)
# library(ggplot2)

# output_path <- "/mnt/90-connectome/Personal/XQ/a-scientific data投稿数据+代码/第五步数据保存829/"

# # ============================================
# # 1. 确保因子顺序
# # ============================================
# ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = paste0("5-HT_", 0:8))

# # ============================================
# # 2. 指定基因列表（按你给出的9组顺序）
# # ============================================
# gene_list <- c(
#   "ARPP21", "TLL1", "RORB",
#   "CHN2", "TMTC1", "KHDRBS2",
#   "PLPP4", "ITPR2", "NRP2",
#   "LDB2", "CACNG3", "ZNF536",
#   "ABI3BP", "CDH7", "KIAA1456",
#   "FRAS1", "CRYBG3", "GRM1",
#   "PTPRT", "PDLIM5", "TSHZ2",
#   "ARHGAP15", "ZEB2", "CALCR",
#   "CNTN5", "UNC13C", "ARHGEF3"
# )

# # 检查数据中存在哪些基因
# available_genes <- gene_list[gene_list %in% rownames(ht_seurat)]
# missing_genes <- gene_list[!gene_list %in% rownames(ht_seurat)]

# if(length(missing_genes) > 0) {
#   cat("⚠️ 以下基因在表达矩阵中未找到，将被跳过:\n")
#   cat(paste(missing_genes, collapse = ", "), "\n\n")
# }
# cat("✅ 可用基因 (", length(available_genes), "个):\n")
# cat(paste(available_genes, collapse = ", "), "\n\n")

# # ============================================
# # 3. 绘制 DotPlot
# # ============================================
# # coord_flip() 实现：X轴 = 细胞类型(5-HT_0~8)，Y轴 = 基因
# p_custom <- DotPlot(
#   ht_seurat,
#   features = available_genes,
#   group.by = "ht_subtype",
#   dot.scale = 5,
#   cols = c("lightgrey", "red")
# ) +
#   coord_flip() +                       # 翻转：细胞类型→X轴，基因→Y轴
#   RotatedAxis() +
#   ggtitle("5-HT 亚型指定 Marker 基因表达点图") +
#   xlab("5-HT 亚型") +
#   ylab("基因") +
#   theme(
#     axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
#     axis.text.y = element_text(size = 10),
#     axis.title = element_text(size = 12, face = "bold"),
#     plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
#     panel.grid.major = element_blank(),
#     panel.grid.minor = element_blank()
#   )

# print(p_custom)

# ggsave(
#   filename = paste0(output_path, "5HT_Custom_Markers_DotPlot.png"),
#   plot = p_custom,
#   width = 12,
#   height = 10,
#   dpi = 300
# )
# cat("指定基因点图已保存!\n")

In [ ]:
# 先查看所有列名
colnames(classFiltSeurat@meta.data)

In [ ]:
library(Seurat)

# ============================================
# 步骤 5：将 5-HT 精细亚群写回原始对象
# ============================================

# 用现有的 cluster 列作为默认值
classFiltSeurat$cell_type <- as.character(classFiltSeurat$cluster)

# 5-HT 细胞：覆盖为精细亚群 (5-HT_0 ~ 5-HT_8)
ht_cells <- colnames(ht_seurat)
classFiltSeurat$cell_type[ht_cells] <- as.character(ht_seurat$ht_subtype)

# 固定因子顺序
all_types <- unique(classFiltSeurat$cell_type)
ht_types <- paste0("5-HT_", 0:8)
other_types <- setdiff(all_types, ht_types)
type_levels <- c(ht_types, sort(other_types))

classFiltSeurat$cell_type <- factor(classFiltSeurat$cell_type, levels = type_levels)

cat("\n全体 cell_type 分布:\n")
print(table(classFiltSeurat$cell_type))

In [ ]:
save_path <- "./data/"
dir.create(save_path, showWarnings = FALSE, recursive = TRUE)

# 1. 完整对象 RDS（包含 cell_type 列）
saveRDS(classFiltSeurat,
        file = paste0(save_path, "classFiltSeurat_with_ht_subtype.rds"))
cat("✅ 完整 Seurat 对象已保存!\n")

# 2. Metadata CSV
write.csv(classFiltSeurat@meta.data,
          file = paste0(save_path, "classFiltSeurat_metadata.csv"),
          row.names = TRUE)
cat("✅ Metadata 已保存!\n")

# 3. 5-HT 子集 RDS
saveRDS(ht_seurat,
        file = paste0(save_path, "ht_seurat_harmony_base.rds"))
cat("✅ 5-HT 子集已保存!\n")

In [ ]:
# 上面进行初步5-ht分类，因为怕后面non-5-HT-neurons掺杂5-ht

In [ ]:
# 提取 non-5-HT-neurons 并单独重新聚类

In [ ]:
library(Seurat)
save_path <- "./data/"

# ---- 1. 读回两个对象 ----
classFiltSeurat <- readRDS(paste0(save_path, "classFiltSeurat_with_ht_subtype.rds"))
ht_seurat       <- readRDS(paste0(save_path, "ht_seurat_harmony_base.rds"))

ht_cells <- intersect(colnames(ht_seurat), colnames(classFiltSeurat))
stopifnot(length(ht_cells) == 3844)

# ---- 2. 重建标签（修正版，就改在这里）----
cl        <- as.character(classFiltSeurat$cluster)
is_neuron <- startsWith(cl, "Neurons")     # ← 原来的 == "Neurons" 一行匹配不到任何细胞，换成这个
all_bc    <- colnames(classFiltSeurat)

classFiltSeurat$cluster_final <- cl
classFiltSeurat$cluster_final[all_bc %in% ht_cells] <- "5-HT"
classFiltSeurat$cluster_final[is_neuron & !(all_bc %in% ht_cells)] <- "non-5-HT-neurons"

# ---- 3. 看结果 ----
final_table <- table(classFiltSeurat$cluster_final)
print(final_table)


In [ ]:
cf <- classFiltSeurat$cluster_final
nf <- classFiltSeurat$nFeature_RNA

print(addmargins(table(cf, nf > 1500)))   # 每个类过滤前/后各多少

cat("non-5-HT 且 nFeature>1500:", sum(cf == "non-5-HT-neurons" & nf > 1500), "\n")
cat("5-HT 且 nFeature>1500:",     sum(cf == "5-HT" & nf > 1500), "\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(harmony)

# ============================================================
# 步骤 3.1：提取 non-5-HT 神经元（含 nFeature > 1500 过滤）
# ============================================================
non_ht_seurat <- subset(classFiltSeurat,
                        subset = cluster_final == "non-5-HT-neurons" &
                                 nFeature_RNA > 1500)

cat("non-5-HT 神经元数（提取 + nFeature>1500 后）:", ncol(non_ht_seurat), "\n")
stopifnot(ncol(non_ht_seurat) == 41209)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(harmony)
library(clustree)   # ← 新增

# ============================================================
# 步骤 3.2：标准化 + Harmony（不变）
# ============================================================
non_ht_seurat <- SCTransform(non_ht_seurat,
                             vst.flavor = "v2",
                             variable.features.n = 3000,
                             vars.to.regress = "percent.mt",
                             method = "glmGamPoi",
                             verbose = FALSE)

non_ht_seurat <- RunPCA(non_ht_seurat, npcs = 50, verbose = FALSE)

non_ht_seurat <- RunHarmony(non_ht_seurat,
                            group.by.vars = "donor",
                            reduction.use = "pca",
                            dims.use = 1:30,
                            assay.use = "SCT",
                            verbose = FALSE)

non_ht_seurat <- FindNeighbors(non_ht_seurat, reduction = "harmony", dims = 1:30)

# ============================================================
# 新增：多分辨率聚类 + clustree
# ============================================================
# 注意：这里不设 resolution，默认用 0.8
# 然后手动跑一串分辨率，结果都存进 meta.data
resolutions <- seq(0.1, 1.0, by = 0.1)

for (res in resolutions) {
  non_ht_seurat <- FindClusters(non_ht_seurat,
                                resolution = res,
                                future.seed = TRUE,
                                verbose = FALSE)
  cat("完成 resolution =", res, "\n")
}

# clustree 可视化（列名前缀是 "SCT_snn_res."）
p_clustree <- clustree(non_ht_seurat@meta.data,
                       prefix = "SCT_snn_res.",
                       node_colour = "sc3_stability") +
  ggtitle("Clustree: non-5-HT neurons (res 0.1–1.0)") +
  theme(legend.position = "right")
print(p_clustree)

ggsave("./data/nonHT_clustree.png",
       p_clustree, width = 12, height = 8, dpi = 300)

In [ ]:


# ============================================================
# 步骤 3.2：标准化 + Harmony + 聚类
# ============================================================
non_ht_seurat <- SCTransform(non_ht_seurat,
                             vst.flavor = "v2",
                             variable.features.n = 3000,
                             vars.to.regress = "percent.mt",
                             method = "glmGamPoi",
                             verbose = FALSE)

non_ht_seurat <- RunPCA(non_ht_seurat, npcs = 50, verbose = FALSE)

non_ht_seurat <- RunHarmony(non_ht_seurat,
                            group.by.vars = "donor",
                            reduction.use = "pca",
                            dims.use = 1:30,
                            assay.use = "SCT",
                            verbose = FALSE)

non_ht_seurat <- FindNeighbors(non_ht_seurat, reduction = "harmony", dims = 1:30)
non_ht_seurat <- FindClusters(non_ht_seurat, resolution = 0.3, future.seed = TRUE)
non_ht_seurat <- RunUMAP(non_ht_seurat, reduction = "harmony", dims = 1:30)

# ---------- 重命名为 nonHT_0, nonHT_1... ----------
non_ht_clusters_new <- as.character(non_ht_seurat$seurat_clusters)
non_ht_unique <- sort(unique(as.numeric(non_ht_clusters_new)))

non_ht_counter <- 0
non_ht_map <- character(length(non_ht_unique))
names(non_ht_map) <- as.character(non_ht_unique)

for (cl in as.character(non_ht_unique)) {
  non_ht_map[cl] <- paste0("nonHT_", non_ht_counter)
  non_ht_counter <- non_ht_counter + 1
}

non_ht_seurat$non_ht_subtype <- unname(non_ht_map[non_ht_clusters_new])

cat("\nnon-5-HT 亚群分布（res 0.3, n=41209）:\n")
print(table(non_ht_seurat$non_ht_subtype))

# ---------- 质控可视化 ----------
p1 <- DimPlot(non_ht_seurat, group.by = "donor", raster = FALSE) +
  ggtitle("non-5-HT neurons by donor (n=41209, nFeature>1500, harmony)") +
  coord_fixed()
print(p1)

p2 <- DimPlot(non_ht_seurat, group.by = "non_ht_subtype",
              label = TRUE, repel = TRUE, raster = FALSE) +
  ggtitle("non-5-HT neurons reclustered (res 0.3, n=41209)") +
  coord_fixed()
print(p2)

p3 <- DotPlot(non_ht_seurat,
              features = c("SNAP25", "SLC17A6", "SLC17A8", "GAD1", "GAD2", "TH"),
              group.by = "non_ht_subtype") +
  RotatedAxis()
print(p3)




In [ ]:
# ---------- 跑通后立即存档（别再只留内存）----------
saveRDS(non_ht_seurat,
        file = paste0(save_path, "non_ht_seurat_final.rds"))
saveRDS(classFiltSeurat,
        file = paste0(save_path, "classFiltSeurat_final.rds"))
cat("✅ non_ht_seurat 和 classFiltSeurat 均已落盘\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

# ============================================
# 0. 参数设置
# ============================================
output_path <- "./data/"

# ============================================
# 1. 数据准备：确保因子顺序 & ident
# ============================================
# 动态获取 non-HT 亚群水平并排序
non_ht_levels <- sort(unique(as.character(non_ht_seurat$non_ht_subtype)))
non_ht_seurat$non_ht_subtype <- factor(non_ht_seurat$non_ht_subtype, levels = non_ht_levels)
non_ht_seurat <- SetIdent(non_ht_seurat, value = "non_ht_subtype")

cat("non-5-HT 亚群分布:\n")
print(table(non_ht_seurat$non_ht_subtype))

In [ ]:
# ============================================
# 1. 数据准备：确保因子顺序 & ident（按 0,1,2... 数字顺序）
# ============================================
# 先获取所有唯一的亚群名
non_ht_levels <- unique(as.character(non_ht_seurat$non_ht_subtype))

# 提取数字部分并按数值排序（确保 nonHT_0,1,2... 而不是字典序）
non_ht_nums <- as.numeric(gsub("nonHT_", "", non_ht_levels))
non_ht_levels <- non_ht_levels[order(non_ht_nums)]

# 或者直接一行：
# non_ht_levels <- paste0("nonHT_", sort(as.numeric(gsub("nonHT_", "", unique(as.character(non_ht_seurat$non_ht_subtype))))))

non_ht_seurat$non_ht_subtype <- factor(non_ht_seurat$non_ht_subtype, levels = non_ht_levels)
non_ht_seurat <- SetIdent(non_ht_seurat, value = "non_ht_subtype")

cat("non-5-HT 亚群分布:\n")
print(table(non_ht_seurat$non_ht_subtype))

In [ ]:
# ============================================
# 2. UMAP 图
# ============================================
# 2.1 有标签版本
p_nonht_umap <- DimPlot(non_ht_seurat, group.by = "non_ht_subtype", 
                        label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("non-5-HT Neurons Subtypes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))

print(p_nonht_umap)
ggsave(paste0(output_path, "nonHT_UMAP_labeled.png"), p_nonht_umap, width = 10, height = 8, dpi = 300)

# 2.2 无标签版本
p_nonht_umap_nolabel <- DimPlot(non_ht_seurat, group.by = "non_ht_subtype", 
                                label = FALSE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("non-5-HT Neurons Subtypes (No Label)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        legend.position = "right")

print(p_nonht_umap_nolabel)
ggsave(paste0(output_path, "nonHT_UMAP_nolabel.png"), p_nonht_umap_nolabel, width = 10, height = 8, dpi = 300)

cat("UMAP 图已保存\n")


In [ ]:
# ============================================
# 3. 找 Marker 基因 (Wilcoxon) — 只跑一次
# ============================================
nonht_markers <- FindAllMarkers(non_ht_seurat,
                                only.pos = TRUE,
                                min.pct = 0.25,
                                logfc.threshold = 0.25,
                                test.use = "wilcox")

# 保存 Top20
top20 <- nonht_markers %>%
  group_by(cluster) %>%
  top_n(n = 20, wt = avg_log2FC) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(top20, paste0(output_path, "nonHT_top20_markers.csv"), row.names = FALSE)
cat("Top20 markers 已保存\n")

# ============================================
# 4. Top5 DotPlot
# ============================================
top5_genes <- nonht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- top5_genes %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot5 <- DotPlot(non_ht_seurat, features = gene_order5, group.by = "non_ht_subtype",
                  dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT 亚型 Top5 Marker 基因表达点图") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )

print(p_dot5)
ggsave(paste0(output_path, "nonHT_Top5_Markers_DotPlot.png"), p_dot5, 
       width = 14, height = 16, dpi = 300)

write.csv(top5_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_Top5_Markers_table.csv"), row.names = FALSE)
cat("Top5 点图已保存\n")

# ============================================
# 5. Top10 DotPlot
# ============================================
top10_genes <- nonht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- top10_genes %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot10 <- DotPlot(non_ht_seurat, features = gene_order10, group.by = "non_ht_subtype",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT 亚型 Top10 Marker 基因表达点图") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )

print(p_dot10)
ggsave(paste0(output_path, "nonHT_Top10_Markers_DotPlot.png"), p_dot10, 
       width = 16, height = 22, dpi = 300)

write.csv(top10_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_Top10_Markers_table.csv"), row.names = FALSE)
cat("Top10 点图已保存\n")

# ============================================
# 6. Top20 DotPlot
# ============================================
gene_order20 <- top20 %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot20 <- DotPlot(non_ht_seurat, features = gene_order20, group.by = "non_ht_subtype",
                   dot.scale = 3, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT 亚型 Top20 Marker 基因表达点图") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 5.5),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )

print(p_dot20)
ggsave(paste0(output_path, "nonHT_Top20_Markers_DotPlot.png"), p_dot20, 
       width = 18, height = 32, dpi = 300)

write.csv(top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_Top20_Markers_table.csv"), row.names = FALSE)
cat("Top20 点图已保存\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

output_path <- "/mnt/90-connectome/Personal/XQ/a-nc-raphe/第五步数据保存1019/"

# ============================================
# 0. 数据准备（使用清洗后的对象）
# ============================================
# 重新获取并固定因子水平（按数值 0,1,2...排序）
nonht_levels <- paste0("nonHT_", sort(as.numeric(gsub("nonHT_", "", 
                unique(as.character(non_ht_seurat$non_ht_subtype))))))
non_ht_seurat$non_ht_subtype <- factor(non_ht_seurat$non_ht_subtype, levels = nonht_levels)
non_ht_seurat_clean <- SetIdent(non_ht_seurat, value = "non_ht_subtype")

cat("non-5-HT 亚群分布:\n")
print(table(non_ht_seurat_clean$non_ht_subtype))


In [ ]:
# ============================================
# 1. UMAP 图
# ============================================
# 1.1 有标签版本
p_nonht_umap <- DimPlot(non_ht_seurat_clean, group.by = "non_ht_subtype", 
                        label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("non-5-HT Neurons Subtypes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_nonht_umap)
ggsave(paste0(output_path, "nonHT_UMAP_labeled.png"), p_nonht_umap, width = 10, height = 8, dpi = 300)

# 1.2 无标签版本
p_nonht_umap_nolabel <- DimPlot(non_ht_seurat_clean, group.by = "non_ht_subtype", 
                                label = FALSE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("non-5-HT Neurons Subtypes (No Label)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        legend.position = "right")
print(p_nonht_umap_nolabel)
ggsave(paste0(output_path, "nonHT_UMAP_nolabel.png"), p_nonht_umap_nolabel, width = 10, height = 8, dpi = 300)
cat("UMAP 图已保存\n")


In [ ]:
# ============================================
# 6. 指定基因 DotPlot + UMAP FeaturePlot
# ============================================
# 用户指定基因，自动修正大小写，ChE 匹配为 ACHE
gene_list_input <- c('SLC17A6','SLC17A7','SLC17A8','GAD1','GAD2','TH')
gene_list <- toupper(gene_list_input)  # 统一大写
gene_list[gene_list == "CHE"] <- "ACHE"  # ChE 通常指 ACHE（乙酰胆碱酯酶）

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

if(length(missing_genes) > 0) {
  cat("⚠️ 以下基因在表达矩阵中未找到，将被跳过:\n")
  cat(paste(missing_genes, collapse = ", "), "\n\n")
}
cat("✅ 可用基因 (", length(available_genes), "个):\n")
cat(paste(available_genes, collapse = ", "), "\n\n")

# 6.1 DotPlot
p_custom <- DotPlot(non_ht_seurat_clean, features = available_genes, group.by = "non_ht_subtype",
                    dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT 亚型指定 Marker 基因表达点图") +
  xlab("non-5-HT 亚型") + ylab("基因") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 12, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_custom)
ggsave(paste0(output_path, "nonHT_Custom_Markers_DotPlot.png"), p_custom, 
       width = 12, height = 10, dpi = 300)
cat("指定基因点图已保存\n")

# 6.2 UMAP FeaturePlot（分面展示）
p_features <- FeaturePlot(non_ht_seurat_clean, features = available_genes, 
                          ncol = 4, raster = FALSE, cols = c("lightgrey", "red"))
print(p_features)
ggsave(paste0(output_path, "nonHT_Custom_Markers_UMAP.png"), p_features, 
       width = 16, height = 12, dpi = 300)
cat("指定基因 UMAP FeaturePlot 已保存\n")

In [ ]:
# ============================================
# 6. 指定基因 DotPlot + UMAP FeaturePlot
# ============================================
# 用户指定基因，自动修正大小写，ChE 匹配为 ACHE
gene_list_input <- c('SLC17A6','SLC17A8','GAD1','GAD2','SOX5','GFAP','PECAM1','FLT1','VCAN','MOBP','C1QC','SLC6A4','TH')
gene_list <- toupper(gene_list_input)  # 统一大写
gene_list[gene_list == "CHE"] <- "ACHE"  # ChE 通常指 ACHE（乙酰胆碱酯酶）

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

if(length(missing_genes) > 0) {
  cat("⚠️ 以下基因在表达矩阵中未找到，将被跳过:\n")
  cat(paste(missing_genes, collapse = ", "), "\n\n")
}
cat("✅ 可用基因 (", length(available_genes), "个):\n")
cat(paste(available_genes, collapse = ", "), "\n\n")

# 6.1 DotPlot
p_custom <- DotPlot(non_ht_seurat_clean, features = available_genes, group.by = "non_ht_subtype",
                    dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT 亚型指定 Marker 基因表达点图") +
  xlab("non-5-HT 亚型") + ylab("基因") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 12, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_custom)
ggsave(paste0(output_path, "nonHT_Custom_Markers_DotPlot.png"), p_custom, 
       width = 12, height = 10, dpi = 300)
cat("指定基因点图已保存\n")

# 6.2 UMAP FeaturePlot（分面展示）
p_features <- FeaturePlot(non_ht_seurat_clean, features = available_genes, 
                          ncol = 4, raster = FALSE, cols = c("lightgrey", "red"))
print(p_features)
ggsave(paste0(output_path, "nonHT_Custom_Markers_UMAP.png"), p_features, 
       width = 16, height = 12, dpi = 300)
cat("指定基因 UMAP FeaturePlot 已保存\n")

In [ ]:
# ============================================
# 7. 小提琴图
# ============================================
plot_data <- non_ht_seurat_clean@meta.data
plot_data$non_ht_subtype <- factor(plot_data$non_ht_subtype, levels = nonht_levels)

# 7.1 nFeature_RNA（刻度每 1000 一个）
nfeature_max <- max(plot_data$nFeature_RNA, na.rm = TRUE)
nfeature_breaks <- seq(0, ceiling(nfeature_max / 1000) * 1000, by = 1000)

p_nfeature <- ggplot(plot_data, aes(x = non_ht_subtype, y = nFeature_RNA, fill = non_ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  scale_y_continuous(breaks = nfeature_breaks) +
  labs(x = "", y = "nFeature RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"), axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10), axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_nfeature)
ggsave(paste0(output_path, "violin_nonHT_nFeature_RNA_1k.png"), p_nfeature, width = 8, height = 6, dpi = 150)
ggsave(paste0(output_path, "violin_nonHT_nFeature_RNA_1k.pdf"), p_nfeature, width = 12, height = 8)

# 7.2 nCount_RNA
p_ncount <- ggplot(plot_data, aes(x = non_ht_subtype, y = nCount_RNA, fill = non_ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  labs(x = "", y = "nCount RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"), axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10), axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_ncount)
ggsave(paste0(output_path, "violin_nonHT_nCount_RNA.png"), p_ncount, width = 8, height = 6, dpi = 150)
ggsave(paste0(output_path, "violin_nonHT_nCount_RNA.pdf"), p_ncount, width = 12, height = 8)
cat("小提琴图已保存\n")


In [ ]:
# ============================================
# 8. Ranger + 热图
# ============================================
expr_matrix <- GetAssayData(non_ht_seurat_clean, layer = "data")
genes <- VariableFeatures(non_ht_seurat_clean)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(non_ht_seurat_clean$non_ht_subtype)

rf_nonht <- ranger(x = X, y = y, num.trees = 500, num.threads = 8, classification = TRUE)

cat("\nOOB Error:", rf_nonht$prediction.error, "\n")

conf_mat <- rf_nonht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("对角线准确率:\n")
print(round(diag(confusionRatio), 2))

# 过滤细胞数 >= 20 的亚群
df_num <- as.data.frame(table(non_ht_seurat_clean$non_ht_subtype))
select_cell <- as.character(df_num[df_num$Freq >= 20, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))
mat_to_plot <- confusionRatio[select_cell, select_cell, drop = FALSE]

# 8.1 带数值版本
num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

png(paste0(output_path, "nonHT_Ranger_Heatmap_withNum.png"), width = 1200, height = 1000, res = 150)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 45, cellheight = 45, fontsize = 14,
         color = viridis(100), main = "non-5-HT Subtypes Ranger Confusion Matrix",
         display_numbers = num_labels, fontsize_number = 12, number_color = "white")
dev.off()

# 8.2 不带数值版本
png(paste0(output_path, "nonHT_Ranger_Heatmap_noNum.png"), width = 1200, height = 1000, res = 150)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 45, cellheight = 45, fontsize = 14,
         color = viridis(100), main = "non-5-HT Subtypes Ranger Confusion Matrix",
         display_numbers = FALSE)
dev.off()

# 屏幕显示不带数值版本
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 45, cellheight = 45, fontsize = 14,
         color = viridis(100), main = "non-5-HT Subtypes Ranger Confusion Matrix (No Numbers)",
         display_numbers = FALSE)

cat("Ranger 热图已保存\n")

cat("\n===== non-5-HT 分析全部完成 =====\n")

In [ ]:
# ============================================
# 6.3 SLC17A6 / SLC17A8 / GAD1 / GAD2 各亚型小提琴图
# ============================================
library(tidyr)   # pivot_longer 在这里
vln_genes <- c("SLC17A6", "SLC17A8", "GAD1", "GAD2", "TH")
vln_genes <- vln_genes[vln_genes %in% rownames(non_ht_seurat_clean)]

expr_vln <- FetchData(non_ht_seurat_clean, vars = c("non_ht_subtype", vln_genes))

expr_vln_long <- expr_vln %>%
  pivot_longer(cols = -non_ht_subtype,
               names_to = "gene", values_to = "Expression") %>%
  mutate(gene = factor(gene, levels = vln_genes),
         non_ht_subtype = factor(non_ht_subtype))   # 想固定顺序的话在这里加 levels = ...

p_vln <- ggplot(expr_vln_long,
                aes(x = non_ht_subtype, y = Expression, fill = non_ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE) +
  geom_boxplot(width = 0.15, outlier.size = 0.15, alpha = 0.7, color = "grey30") +
  facet_wrap(~ gene, nrow = 1, scales = "free_y") +
  labs(title = "SLC17A6 / SLC17A8 / GAD1 / GAD2: non-5-HT 亚型表达",
       x = NULL, y = "Normalized Expression") +
  theme_bw(base_size = 13) +
  theme(legend.position = "none",
        panel.grid = element_blank(),
        strip.text = element_text(face = "bold", size = 12),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9))

print(p_vln)
ggsave(paste0(output_path, "nonHT_VlnPlot_Glut_GABA_markers.png"), p_vln,
       width = 14, height = 5, dpi = 300)
cat("指定基因小提琴图已保存\n")

In [ ]:


# # # ============================================library(Seurat)
library(dplyr)

save_path <- "./data/"

# ============================================
# 1. 定义神经递质类型映射（nonHT_15 改为 Glut）
# ============================================
glut_map <- c(
  "nonHT_0"  = "Glut_1",
  "nonHT_9" = "Glut_2",
  "nonHT_10" = "Glut_3", 
  "nonHT_17" = "Glut_4",
  "nonHT_22" = "Glut_5",   
  "nonHT_24" = "Glut_6",   
  "nonHT_25" = "Glut_7"
)

glut_gaba_map <- c(
  "nonHT_0"  = "Glut-GABA_1",
  "nonHT_4"  = "Glut-GABA_2",
  "nonHT_8"  = "Glut-GABA_3",
  "nonHT_11"  = "Glut-GABA_4",
  "nonHT_13"  = "Glut-GABA_5",
  "nonHT_28"  = "Glut-GABA_6"
)

gaba_map <- c(
  "nonHT_1"  = "GABA_1",
  "nonHT_2"  = "GABA_2",
  "nonHT_3"  = "GABA_3",
  "nonHT_5"  = "GABA_4",
  "nonHT_6"  = "GABA_5",
  "nonHT_7"  = "GABA_6",
  "nonHT_12" = "GABA_7",
  "nonHT_14" = "GABA_8",
  "nonHT_15" = "GABA_9",
  "nonHT_16" = "GABA_10",
  "nonHT_18" = "GABA_11",
  "nonHT_19" = "GABA_12",
  "nonHT_20" = "GABA_13",
  "nonHT_21" = "GABA_14",
  "nonHT_23" = "GABA_15",
  "nonHT_27" = "GABA_16"
)

da_map <- c("nonHT_26" = "DA_1")

# 合并所有映射
all_map <- c(glut_map, glut_gaba_map, gaba_map, da_map)

cat("映射表:\n")
print(all_map)

# ============================================
# 修正：重命名 neuro_type
# ============================================

# 先按细胞顺序查表
neuro_vec <- all_map[as.character(non_ht_seurat_clean$non_ht_subtype)]

# 关键：去掉 names，否则 Seurat 会按名称匹配细胞
names(neuro_vec) <- NULL

# 再赋值
non_ht_seurat_clean$neuro_type <- neuro_vec

# 固定因子顺序
type_order <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 1:6),
  paste0("GABA_", 1:16),
  "DA_1"
)
non_ht_seurat_clean$neuro_type <- factor(non_ht_seurat_clean$neuro_type, levels = type_order)

cat("\nnon-5-HT 神经递质类型分布:\n")
print(table(non_ht_seurat_clean$neuro_type))

# # # 4. 保存
# # # ============================================
# # saveRDS(non_ht_seurat_clean, paste0(save_path, "non_ht_seurat_neurotype.rds"))
# # saveRDS(classFiltSeurat, paste0(save_path, "classFiltSeurat_neurotype.rds"))

# # cat("\n✅ 修正完成并已保存!\n")

In [ ]:
# ============================================
# 3. 同步更新 classFiltSeurat
# ============================================

# 先把 factor 转成 character（关键！否则新类型会变成 NA）
classFiltSeurat$cell_type <- as.character(classFiltSeurat$cell_type)

# 提取 non-5-HT 细胞名
nonht_cells <- colnames(non_ht_seurat_clean)

# 赋值（现在不会报 NA 了）
classFiltSeurat$cell_type[nonht_cells] <- as.character(non_ht_seurat_clean$neuro_type)

# 重新整理全体因子水平
all_types <- unique(classFiltSeurat$cell_type)

ht_types       <- sort(grep("^5-HT", all_types, value = TRUE))
glut_types     <- sort(grep("^Glut_", all_types, value = TRUE))
glutgaba_types <- sort(grep("^Glut-GABA_", all_types, value = TRUE))
gaba_types     <- sort(grep("^GABA_", all_types, value = TRUE))
da_types       <- sort(grep("^DA_", all_types, value = TRUE))
other_types    <- sort(setdiff(all_types, c(ht_types, glut_types, glutgaba_types, gaba_types, da_types)))

type_levels <- c(ht_types, glut_types, glutgaba_types, gaba_types, da_types, other_types)
classFiltSeurat$cell_type <- factor(classFiltSeurat$cell_type, levels = type_levels)

cat("\nclassFiltSeurat 全体 cell_type 分布:\n")
print(table(classFiltSeurat$cell_type))



In [ ]:
# ============================================
# 删除 cell_type 以 "Neurons_" 开头的细胞
# ============================================

# 先看一眼要删多少
neuron_idx <- grepl("^Neurons_", classFiltSeurat$cell_type)
cat("将删除的 Neurons_* 细胞数:", sum(neuron_idx), "\n")
cat("删除前总细胞数:", ncol(classFiltSeurat), "\n")

# subset 掉这些细胞
classFiltSeurat <- subset(classFiltSeurat, subset = cell_type %in% 
                            levels(classFiltSeurat$cell_type)[!grepl("^Neurons_", levels(classFiltSeurat$cell_type))])

# 或者更直接的写法（等价）：
# classFiltSeurat <- subset(classFiltSeurat, cells = colnames(classFiltSeurat)[!neuron_idx])

cat("删除后总细胞数:", ncol(classFiltSeurat), "\n")

# 重新整理因子水平（把空水平去掉）
classFiltSeurat$cell_type <- droplevels(classFiltSeurat$cell_type)

cat("\n删除后的 cell_type 分布:\n")
print(table(classFiltSeurat$cell_type))

In [ ]:
# ============================================
# 4. 保存
# ============================================
saveRDS(non_ht_seurat_clean, paste0(save_path, "non_ht_seurat_neurotype.rds"))
saveRDS(classFiltSeurat, paste0(save_path, "classFiltSeurat_neurotype.rds"))

cat("\n✅ 修正完成并已保存!\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)

save_path <- "./data/"

# ============================================
# 读入两个对象
# ============================================
non_ht_seurat_clean <- readRDS(paste0(save_path, "non_ht_seurat_neurotype.rds"))
# classFiltSeurat     <- readRDS(paste0(save_path, "classFiltSeurat_neurotype.rds"))

# ============================================
# 读入后检查（确认没读错）
# ============================================
cat("===== non_ht_seurat_clean =====\n")
print(non_ht_seurat_clean)                      # assay / reduction 概况
cat("细胞数:", ncol(non_ht_seurat_clean), "\n")
cat("neuro_type 分布:\n")
print(table(non_ht_seurat_clean$neuro_type))

# cat("\n===== classFiltSeurat =====\n")
# print(classFiltSeurat)
# cat("细胞数:", ncol(classFiltSeurat), "\n")
# cat("cell_type_fine 分布:\n")
# print(table(classFiltSeurat$cell_type_fine))

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

output_path <- "./data/"

# ============================================
# 0. 确认数据 & 因子顺序
# ============================================
# 确保 neuro_type 已存在且为因子
if(!"neuro_type" %in% colnames(non_ht_seurat_clean@meta.data)) {
  stop("请先运行神经递质类型重命名代码！")
}

type_order <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 1:6),
  paste0("GABA_", 1:16),
  "DA_1"
)
non_ht_seurat_clean$neuro_type <- factor(non_ht_seurat_clean$neuro_type, levels = type_order)
non_ht_seurat_clean <- SetIdent(non_ht_seurat_clean, value = "neuro_type")

cat("non-5-HT 神经递质类型分布:\n")
print(table(non_ht_seurat_clean$neuro_type))

# ============================================
# 1. UMAP 图（按神经递质类型）
# ============================================
# 1.1 有标签版本
p_neuro_umap <- DimPlot(non_ht_seurat_clean, group.by = "neuro_type",
                        label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() +
  ggtitle("non-5-HT Neurons by Neurotransmitter Type") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_neuro_umap)

ggsave(paste0(output_path, "nonHT_NeuroType_UMAP_labeled.png"),
       p_neuro_umap, width = 12, height = 10, dpi = 300)
ggsave(paste0(output_path, "nonHT_NeuroType_UMAP_labeled.pdf"),   # ← 新增
       p_neuro_umap, width = 12, height = 10, limitsize = FALSE)

# 1.2 无标签版本
p_neuro_umap_nolabel <- DimPlot(non_ht_seurat_clean, group.by = "neuro_type",
                                label = FALSE, raster = FALSE) +
  coord_fixed() +
  ggtitle("non-5-HT Neurons by Neurotransmitter Type (No Label)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        legend.position = "right")
print(p_neuro_umap_nolabel)

ggsave(paste0(output_path, "nonHT_NeuroType_UMAP_nolabel.png"),
       p_neuro_umap_nolabel, width = 12, height = 10, dpi = 300)
ggsave(paste0(output_path, "nonHT_NeuroType_UMAP_nolabel.pdf"),   # ← 新增
       p_neuro_umap_nolabel, width = 12, height = 10, limitsize = FALSE)

cat("UMAP 图已保存（png + pdf）\n")


In [ ]:
# ============================================
# 2. 找 Marker 基因 (Wilcoxon) — 按 neuro_type 分组，只跑一次
# ============================================
nonht_markers <- FindAllMarkers(non_ht_seurat_clean,
                                only.pos = TRUE,
                                min.pct = 0.25,
                                logfc.threshold = 0.25,
                                test.use = "wilcox")

# 保存 Top20
top20 <- nonht_markers %>%
  group_by(cluster) %>%
  top_n(n = 20, wt = avg_log2FC) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(top20, paste0(output_path, "nonHT_NeuroType_top20_markers.csv"), row.names = FALSE)
cat("Top20 markers 已保存\n")

# ============================================
# 3. Top5 DotPlot（按神经递质类型）
# ============================================
top5_genes <- nonht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- top5_genes %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot5 <- DotPlot(non_ht_seurat_clean, features = gene_order5, group.by = "neuro_type",
                  dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT 神经递质类型 Top5 Marker 基因表达点图") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot5)
ggsave(paste0(output_path, "nonHT_NeuroType_Top5_DotPlot.png"), p_dot5, 
       width = 16, height = 18, dpi = 300)

write.csv(top5_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_NeuroType_Top5_table.csv"), row.names = FALSE)
cat("Top5 点图已保存\n")

# ============================================
# 4. Top10 DotPlot
# ============================================
top10_genes <- nonht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- top10_genes %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot10 <- DotPlot(non_ht_seurat_clean, features = gene_order10, group.by = "neuro_type",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT 神经递质类型 Top10 Marker 基因表达点图") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot10)
ggsave(paste0(output_path, "nonHT_NeuroType_Top10_DotPlot.png"), p_dot10, 
       width = 18, height = 24, dpi = 300)

write.csv(top10_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_NeuroType_Top10_table.csv"), row.names = FALSE)
cat("Top10 点图已保存\n")

# ============================================
# 5. Top20 DotPlot
# ============================================
gene_order20 <- top20 %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot20 <- DotPlot(non_ht_seurat_clean, features = gene_order20, group.by = "neuro_type",
                   dot.scale = 3, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT 神经递质类型 Top20 Marker 基因表达点图") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 5.5),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_dot20)
ggsave(paste0(output_path, "nonHT_NeuroType_Top20_DotPlot.png"), p_dot20, 
       width = 20, height = 34, dpi = 300)

write.csv(top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_NeuroType_Top20_table.csv"), row.names = FALSE)
cat("Top20 点图已保存\n")

In [ ]:
# OTOF,NPTX1;SLC17A8,LMX1B;ADGRD2,CDH5;TRIM55,IRX4;PRDM6,LHX9;FST,BCHE;SEMA3A,SCTR;PENK.TACR3;SKOR2,CCDC60;CCK,RIN2;ATP12A,PGM5;SLC32A1,GSTP1;SLC6A5,LAMP5;FGFR2,SATB1;OTX2,SERPINF1;GALR1,SYT6;TAC3,IL13RA1;GLI3,GATA3;TLL1,ADAMTSL1;GSN,CXCL14;FLT1,ONECUT1;SNTG2,CHRNA2;KIF25,BCL11B;SST,CHST9;FOXA2,EMX2;CPA4,RLN3;TH,SLC6A2


In [ ]:
library(Seurat)
library(ggplot2)

output_path <- "./data/"

# ============================================
# 0. 确认合并后的 neuro_type 顺序
# ============================================
type_order <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 1:6),
  paste0("GABA_", 1:16),
  "DA_1"
)
non_ht_seurat_clean$neuro_type <- factor(non_ht_seurat_clean$neuro_type, levels = type_order)

# ============================================
# 1. 新基因列表（28个，按配对顺序展开）
# ============================================
gene_list <- c("OTOF","NPTX1","SLC17A8","LMX1B","ADGRD2","CDH5","TRIM55","IRX4",
               "PRDM6","LHX9","FST","BCHE","SEMA3A","SCTR","PENK","TACR3",
               "SKOR2","CCDC60","CCK","RIN2","ATP12A","PGM5","SLC32A1","GSTP1",
               "SLC6A5","LAMP5","FGFR2","SATB1","OTX2","SERPINF1","GALR1","SYT6",
               "TAC3","IL13RA1","GLI3","GATA3","TLL1","ADAMTSL1","GSN","CXCL14",
               "FLT1","ONECUT1","SNTG2","CHRNA2","KIF25","BCL11B","SST","CHST9",
               "FOXA2","EMX2","CPA4","RLN3","TH","SLC6A2")

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes   <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

cat("总基因数:", length(gene_list), "，可用:", length(available_genes), "\n")
if(length(missing_genes) > 0) {
  cat("⚠️ 未找到，已跳过:", paste(missing_genes, collapse = ", "), "\n")
}

# ============================================
# 2. DotPlot
# ============================================
p_custom <- DotPlot(
  non_ht_seurat_clean,
  features = available_genes,
  group.by = "neuro_type",
  dot.scale = 6,
  cols = c("lightgrey", "red")
) +
  coord_flip() +
  RotatedAxis() +
  ggtitle("non-5-HT neurotransmitter types: selected markers") +
  xlab("Neurotransmitter type") +
  ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 9),
    axis.title  = element_text(size = 12, face = "bold"),
    plot.title  = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid  = element_blank()
  )

print(p_custom)

ggsave(
  filename = paste0(output_path, "nonHT_NeuroType_Merged_markers_DotPlot.png"),
  plot = p_custom,
  width = 16,
  height = 16,
  dpi = 300
)

# 再加一个 PDF 版本（矢量图，放大不糊）
ggsave(
  filename = paste0(output_path, "nonHT_NeuroType_Merged_markers_DotPlot.pdf"),
  plot = p_custom,
  width = 16,
  height = 16
)

cat("PNG 和 PDF 都已保存!\n")

In [ ]:
# ============================================
# 6. 指定基因 DotPlot + UMAP FeaturePlot
# ============================================
gene_list_input <- c('SLC17A6','SLC17A7','SLC17A8','GAD1','GAD2','TH')
gene_list <- toupper(gene_list_input)
gene_list[gene_list == "CHE"] <- "ACHE"

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

if(length(missing_genes) > 0) {
  cat("⚠️ 以下基因未找到，已跳过:", paste(missing_genes, collapse = ", "), "\n\n")
}
cat("✅ 可用基因 (", length(available_genes), "个):\n")
cat(paste(available_genes, collapse = ", "), "\n\n")

# 6.1 DotPlot（按 neuro_type）
p_custom <- DotPlot(non_ht_seurat_clean, features = available_genes, group.by = "neuro_type",
                    dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT 神经递质类型指定 Marker 基因表达点图") +
  xlab("神经递质类型") + ylab("基因") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 12, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_custom)
ggsave(paste0(output_path, "nonHT_NeuroType_Custom_DotPlot.png"), p_custom, 
       width = 14, height = 10, dpi = 300)
cat("指定基因点图已保存\n")

# # 6.2 UMAP FeaturePlot
# p_features <- FeaturePlot(non_ht_seurat_clean, features = available_genes, 
#                           ncol = 3, raster = FALSE, cols = c("lightgrey", "red"))
# print(p_features)
# ggsave(paste0(output_path, "nonHT_NeuroType_Custom_UMAP.png"), p_features, 
#        width = 15, height = 12, dpi = 300)
# cat("指定基因 UMAP FeaturePlot 已保存\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(ranger)
library(pheatmap)
library(viridis)
library(grid)  # pdf() 设备需要

output_path <- "./data/"

# ============================================
# 0. 确认合并后的 neuro_type 顺序
# ============================================
type_order <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 1:6),
  paste0("GABA_", 1:16),
  "DA_1"
)
non_ht_seurat_clean$neuro_type <- factor(non_ht_seurat_clean$neuro_type, levels = type_order)

# ============================================
# 1. 新基因列表（28个，按配对顺序展开）
# ============================================
gene_list <- c("OTOF","NPTX1","SLC17A8","LMX1B","ADGRD2","CDH5","TRIM55","IRX4",
               "PRDM6","LHX9","FST","BCHE","SEMA3A","SCTR","PENK","TACR3",
               "SKOR2","CCDC60","CCK","RIN2","ATP12A","PGM5","SLC32A1","GSTP1",
               "SLC6A5","LAMP5","FGFR2","SATB1","OTX2","SERPINF1","GALR1","SYT6",
               "TAC3","IL13RA1","GLI3","GATA3","TLL1","ADAMTSL1","GSN","CXCL14",
               "FLT1","ONECUT1","SNTG2","CHRNA2","KIF25","BCL11B","SST","CHST9",
               "FOXA2","EMX2","CPA4","RLN3","TH","SLC6A2")

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes   <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

cat("总基因数:", length(gene_list), "，可用:", length(available_genes), "\n")
if(length(missing_genes) > 0) {
  cat("⚠️ 未找到，已跳过:", paste(missing_genes, collapse = ", "), "\n")
}

# ============================================
# 2. DotPlot
# ============================================
p_custom <- DotPlot(
  non_ht_seurat_clean,
  features = available_genes,
  group.by = "neuro_type",
  dot.scale = 6,
  cols = c("lightgrey", "red")
) +
  coord_flip() +
  RotatedAxis() +
  ggtitle("non-5-HT neurotransmitter types: selected markers") +
  xlab("Neurotransmitter type") +
  ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 9),
    axis.title  = element_text(size = 12, face = "bold"),
    plot.title  = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid  = element_blank()
  )

print(p_custom)

ggsave(
  filename = paste0(output_path, "nonHT_NeuroType_Merged_markers_DotPlot.png"),
  plot = p_custom,
  width = 16,
  height = 16,
  dpi = 300
)
ggsave(
  filename = paste0(output_path, "nonHT_NeuroType_Merged_markers_DotPlot.pdf"),
  plot = p_custom,
  width = 16,
  height = 16
)
cat("DotPlot PNG + PDF 已保存!\n")

# ============================================
# 7. 小提琴图（按神经递质类型）
# ============================================
plot_data <- non_ht_seurat_clean@meta.data
plot_data$neuro_type <- factor(plot_data$neuro_type, levels = type_order)

# 7.1 nFeature_RNA（刻度每 1000 一个）
nfeature_max <- max(plot_data$nFeature_RNA, na.rm = TRUE)
nfeature_breaks <- seq(0, ceiling(nfeature_max / 1000) * 1000, by = 1000)

p_nfeature <- ggplot(plot_data, aes(x = neuro_type, y = nFeature_RNA, fill = neuro_type)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  scale_y_continuous(breaks = nfeature_breaks) +
  labs(x = "", y = "nFeature RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"), axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10), axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_nfeature)

ggsave(paste0(output_path, "violin_nonHT_NeuroType_nFeature_RNA_1k.png"),
       p_nfeature, width = 10, height = 6, dpi = 150)
ggsave(paste0(output_path, "violin_nonHT_NeuroType_nFeature_RNA_1k.pdf"),
       p_nfeature, width = 14, height = 8)

# 7.2 nCount_RNA
p_ncount <- ggplot(plot_data, aes(x = neuro_type, y = nCount_RNA, fill = neuro_type)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  labs(x = "", y = "nCount RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"), axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10), axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_ncount)

ggsave(paste0(output_path, "violin_nonHT_NeuroType_nCount_RNA.png"),
       p_ncount, width = 10, height = 6, dpi = 150)
ggsave(paste0(output_path, "violin_nonHT_NeuroType_nCount_RNA.pdf"),
       p_ncount, width = 14, height = 8)
cat("小提琴图 PNG + PDF 已保存\n")

# ============================================
# 8. Ranger + 热图（按 neuro_type）
# ============================================
expr_matrix <- GetAssayData(non_ht_seurat_clean, layer = "data")
genes <- VariableFeatures(non_ht_seurat_clean)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(non_ht_seurat_clean$neuro_type)

rf_nonht <- ranger(x = X, y = y, num.trees = 500, num.threads = 8, classification = TRUE)

cat("\nOOB Error:", rf_nonht$prediction.error, "\n")

conf_mat <- rf_nonht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("对角线准确率:\n")
print(round(diag(confusionRatio), 2))

# 过滤细胞数 >= 20 的亚群
df_num <- as.data.frame(table(non_ht_seurat_clean$neuro_type))
select_cell <- as.character(df_num[df_num$Freq >= 20, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))
mat_to_plot <- confusionRatio[select_cell, select_cell, drop = FALSE]

# 8.1 带数值版本
num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

# --- PNG ---
png(paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_withNum.png"), width = 1400, height = 1200, res = 150)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
         display_numbers = num_labels, fontsize_number = 10, number_color = "white")
dev.off()

# --- PDF ---
pdf(paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_withNum.pdf"),
    width = 14, height = 12)  # 单位是英寸：1400px/150dpi≈9.3in，放宽到14×12
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
         display_numbers = num_labels, fontsize_number = 10, number_color = "white")
dev.off()

# 8.2 不带数值版本
# --- PNG ---
png(paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_noNum.png"), width = 1400, height = 1200, res = 150)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
         display_numbers = FALSE)
dev.off()

# --- PDF ---
pdf(paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_noNum.pdf"),
    width = 14, height = 12)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
         display_numbers = FALSE)
dev.off()

# 屏幕显示不带数值版本
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger (No Numbers)",
         display_numbers = FALSE)

cat("Ranger 热图 PNG + PDF 已保存\n")

cat("\n===== non-5-HT 神经递质类型分析全部完成 =====\n")

In [ ]:
library(viridis)
library(grid)

# ============================================
# 8. Ranger + 热图（按 neuro_type）— 完整版
# ============================================
expr_matrix <- GetAssayData(non_ht_seurat_clean, layer = "data")
genes <- VariableFeatures(non_ht_seurat_clean)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(non_ht_seurat_clean$neuro_type)

rf_nonht <- ranger(x = X, y = y, num.trees = 500, num.threads = 8, classification = TRUE)

cat("\nOOB Error:", rf_nonht$prediction.error, "\n")

conf_mat <- rf_nonht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("对角线准确率:\n")
print(round(diag(confusionRatio), 2))

cat("\n所有 neuro_type 细胞数:\n")
print(table(non_ht_seurat_clean$neuro_type))

# ---- 取全部类型（阈值 10）----
df_num <- as.data.frame(table(non_ht_seurat_clean$neuro_type))
select_cell <- as.character(df_num[df_num$Freq >= 10, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))

type_order_present <- type_order[type_order %in% select_cell]
mat_to_plot <- confusionRatio[type_order_present, type_order_present, drop = FALSE]

cat("\n热图包含类型数:", nrow(mat_to_plot), "\n")
cat("被排除的类型:", setdiff(type_order, type_order_present), "\n")

# ---- 数值标签 ----
num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

# ============================================
# ★ 核心：按 gtable 实际尺寸保存，绝不截断
# ============================================
save_pheatmap_exact <- function(ph_obj, file_base, dpi = 150) {

  # 从 gtable 计算图形实际需要的宽和高（英寸）
  w_in <- convertWidth(sum(ph_obj$gtable$widths),  "inch", valueOnly = TRUE)
  h_in <- convertHeight(sum(ph_obj$gtable$heights), "inch", valueOnly = TRUE)

  cat(sprintf("实际尺寸: %.1f x %.1f inch\n", w_in, h_in))

  # --- PNG ---
  png(paste0(file_base, ".png"),
      width = w_in, height = h_in, units = "in", res = dpi)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  # --- PDF ---
  pdf(paste0(file_base, ".pdf"), width = w_in, height = h_in)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ---- 8.1 带数值版本 ----
p_num <- pheatmap(mat_to_plot,
                  breaks = seq(0, 1, 0.01),
                  cluster_rows = FALSE,
                  cluster_cols = FALSE,
                  border_color = NA,
                  cellwidth = 30,
                  cellheight = 30,
                  fontsize = 10,
                  color = viridis(100),
                  main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
                  display_numbers = num_labels,
                  fontsize_number = 8,
                  number_color = "white")

save_pheatmap_exact(p_num, paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_withNum"))

# ---- 8.2 不带数值版本 ----
p_nonum <- pheatmap(mat_to_plot,
                    breaks = seq(0, 1, 0.01),
                    cluster_rows = FALSE,
                    cluster_cols = FALSE,
                    border_color = NA,
                    cellwidth = 30,
                    cellheight = 30,
                    fontsize = 10,
                    color = viridis(100),
                    main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
                    display_numbers = FALSE)

save_pheatmap_exact(p_nonum, paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_noNum"))

# ---- 屏幕显示 ----
grid.newpage()
grid.draw(p_nonum$gtable)

cat("Ranger 热图 PNG + PDF 已保存\n")

In [ ]:
library(viridis)
library(grid)

# ============================================
# 8. Ranger + 热图（每列只保留 Top2 着色和标数）
# ============================================
expr_matrix <- GetAssayData(non_ht_seurat_clean, layer = "data")
genes <- VariableFeatures(non_ht_seurat_clean)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(non_ht_seurat_clean$neuro_type)

rf_nonht <- ranger(x = X, y = y, num.trees = 500, num.threads = 8, classification = TRUE)

cat("\nOOB Error:", rf_nonht$prediction.error, "\n")

conf_mat <- rf_nonht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("对角线准确率:\n")
print(round(diag(confusionRatio), 2))

# ---- 准备矩阵 ----
df_num <- as.data.frame(table(non_ht_seurat_clean$neuro_type))
select_cell <- as.character(df_num[df_num$Freq >= 10, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))

type_order_present <- type_order[type_order %in% select_cell]
mat_to_plot <- confusionRatio[type_order_present, type_order_present, drop = FALSE]

cat("\n热图包含类型数:", nrow(mat_to_plot), "\n")

# ============================================
# ★ 每列只保留 Top2，其余颜色置 0
# ============================================
mat_color <- mat_to_plot
mat_labels <- matrix("", nrow = nrow(mat_to_plot),
                     ncol = ncol(mat_to_plot),
                     dimnames = dimnames(mat_to_plot))

for (j in seq_len(ncol(mat_to_plot))) {
  ord <- order(mat_to_plot[, j], decreasing = TRUE)
  top2 <- ord[1:2]
  mat_color[-top2, j] <- 0
  mat_labels[top2, j] <- sprintf("%.2f", mat_to_plot[top2, j])
}

# ---- 确保输出目录存在 ----
if (!dir.exists(output_path)) {
  dir.create(output_path, recursive = TRUE)
}

# ============================================
# ★ 按 gtable 实际尺寸保存，绝不截断
# ============================================
save_pheatmap_exact <- function(ph_obj, file_base, dpi = 150) {

  w_in <- convertWidth(sum(ph_obj$gtable$widths),  "inch", valueOnly = TRUE)
  h_in <- convertHeight(sum(ph_obj$gtable$heights), "inch", valueOnly = TRUE)

  cat(sprintf("实际尺寸: %.1f x %.1f inch\n", w_in, h_in))

  # --- PNG ---
  png(paste0(file_base, ".png"),
      width = w_in, height = h_in, units = "in", res = dpi)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  # --- PDF ---
  pdf(paste0(file_base, ".pdf"), width = w_in, height = h_in)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ---- 8.1 带数值版本（只标每列 Top2） ----
p_num <- pheatmap(mat_color,
                  breaks = seq(0, 1, 0.01),
                  cluster_rows = FALSE,
                  cluster_cols = FALSE,
                  border_color = NA,
                  cellwidth = 30, cellheight = 30, fontsize = 10,
                  color = viridis(100),
                  main = "non-5-HT Neurotransmitter Types Ranger (Top2 per column)",
                  display_numbers = mat_labels,
                  fontsize_number = 8, number_color = "white")

save_pheatmap_exact(p_num, paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_Top2_withNum"))

# ---- 8.2 不带数值版本（颜色同样只有 Top2） ----
p_nonum <- pheatmap(mat_color,
                    breaks = seq(0, 1, 0.01),
                    cluster_rows = FALSE,
                    cluster_cols = FALSE,
                    border_color = NA,
                    cellwidth = 30, cellheight = 30, fontsize = 10,
                    color = viridis(100),
                    main = "non-5-HT Neurotransmitter Types Ranger (Top2 per column)",
                    display_numbers = FALSE)

save_pheatmap_exact(p_nonum, paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_Top2_noNum"))

# ---- 屏幕显示 ----
grid.newpage()
grid.draw(p_nonum$gtable)

cat("Ranger Top2 热图 PNG + PDF 全部保存完成\n")

In [ ]:
library(Seurat)

# ============================================
# 读取 ht_seurat 对象
# ============================================
ht_seurat <- readRDS("./data/classFiltSeurat_neurotype.rds")

# ============================================
# 快速检查读取的对象
# ============================================
cat("===== 对象概况 =====\n")
cat("细胞数:", ncol(ht_seurat), "\n")
cat("基因数:", nrow(ht_seurat), "\n")
cat("Assays:", paste(Assays(ht_seurat), collapse = ", "), "\n")
cat("Reductions:", paste(Reductions(ht_seurat), collapse = ", "), "\n")
cat("默认 Assay:", DefaultAssay(ht_seurat), "\n")

# 查看元数据列
cat("\n===== 元数据列 =====\n")
print(colnames(ht_seurat@meta.data))

# 查看 5-HT 亚群分布（如果有 ht_subtype 列）
if("ht_subtype" %in% colnames(ht_seurat@meta.data)) {
  cat("\n===== 5-HT 亚群分布 =====\n")
  print(table(ht_seurat$ht_subtype))
}

# 查看当前 Ident
cat("\n===== 当前 Ident =====\n")
print(table(Idents(ht_seurat)))

In [ ]:
print(table(ht_seurat$cell_type))

In [ ]:
colnames(ht_seurat@meta.data)
head(ht_seurat@meta.data, 3)

In [ ]:
# 对5-HT 重新聚类

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(ranger)
library(pheatmap)
library(viridis)
library(grid)
library(gtable)
library(future)
library(harmony)

# ============================================
# 1. 提取 5-HT 神经元
# ============================================
# ============================================
# 1. 提取 5-HT 神经元（用 cells 参数，不走表达式求值）
# ============================================
cat("\n提取前总细胞数:", ncol(ht_seurat), "\n")

cells_keep <- rownames(ht_seurat@meta.data)[
  grepl("^5-HT_", as.character(ht_seurat$cell_type))
]

ht_5ht <- subset(ht_seurat, cells = cells_keep)

cat("\n提取后 5-HT 细胞数:", ncol(ht_5ht), "\n")
print(table(ht_5ht$cell_type))


# ---------- 1.0 nFeature>1500 过滤（对 ht_5ht！）----------
cat("5-HT 神经元数（过滤前）:", ncol(ht_5ht), "\n")

ht_5ht <- subset(ht_5ht, subset = nFeature_RNA > 1500)

cat("5-HT 神经元数（nFeature>1500 后）:", ncol(ht_5ht), "\n")

# ---------- 1.1 SCTransform + PCA ----------
ht_5ht <- SCTransform(ht_5ht,
                      vst.flavor = "v2",
                      variable.features.n = 3000,
                      vars.to.regress = "percent.mt",
                      method = "glmGamPoi",
                      verbose = FALSE)
ht_5ht <- RunPCA(ht_5ht, npcs = 50, verbose = FALSE)

# ---------- 1.2 Harmony（供体批次校正）----------
ht_5ht <- RunHarmony(ht_5ht,
                     group.by.vars = "donor",
                     reduction.use = "pca",
                     dims.use = 1:30,
                     assay.use = "SCT",
                     verbose = FALSE)

# ---------- 1.3 邻接图 ----------
ht_5ht <- FindNeighbors(ht_5ht, reduction = "harmony", dims = 1:30)

In [ ]:
save_path <- "./data/"

In [ ]:
# ============================================
# ★ 1.4 多分辨率聚类 + clustree（对 ht_5ht）
# ============================================
resolutions_to_try <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.8, 1.0, 1.2)

for (res in resolutions_to_try) {
  ht_5ht <- FindClusters(ht_5ht,
                         resolution = res,
                         future.seed = TRUE,
                         verbose = FALSE)
}

# clustree 图：看哪个分辨率下聚类结构最稳定
p_clustree <- clustree(ht_5ht, prefix = "SCT_snn_res.")

print(p_clustree)

ggsave(paste0(save_path, "ht_clustree.png"), p_clustree,
       width = 12, height = 10, dpi = 300)
ggsave(paste0(save_path, "ht_clustree.pdf"), p_clustree,
       width = 12, height = 10)

# 辅助：每个分辨率分了多少群 + 最小亚群细胞数
cat("\n各分辨率聚类情况:\n")
for (res in resolutions_to_try) {
  col <- paste0("SCT_snn_res.", res)
  tab <- table(ht_5ht@meta.data[[col]])
  cat(sprintf("res=%.1f: %d 群, 最小群 %d 细胞\n",
              res, length(tab), min(tab)))
}

In [ ]:
# ============================================
# ★ 2. 根据 clustree 选定的分辨率做正式聚类 + UMAP
# ============================================
final_res <- 0.5   # ← 看完 clustree 后改成你选的分辨率

# 对应列已在 1.4 循环里生成，直接设 ident 即可
ht_5ht <- SetIdent(ht_5ht,
                   value = paste0("SCT_snn_res.", final_res))

n_clust <- length(unique(Idents(ht_5ht)))

ht_5ht$ht_subtype <- paste0("5-HT_", as.character(Idents(ht_5ht)))
ht_5ht$ht_subtype <- factor(ht_5ht$ht_subtype,
                            levels = paste0("5-HT_", 0:(n_clust - 1)))
ht_5ht <- SetIdent(ht_5ht, value = "ht_subtype")

cat("\n最终亚群分布 (res =", final_res, "):\n")
print(table(ht_5ht$ht_subtype))

# ---------- UMAP ----------
ht_5ht <- RunUMAP(ht_5ht, reduction = "harmony", dims = 1:30)

# ---------- UMAP 总览图（顺手存 PNG + PDF）----------
p_umap <- DimPlot(ht_5ht, group.by = "ht_subtype",
                  label = TRUE, repel = TRUE) +
  coord_fixed() +
  ggtitle(paste0("5-HT Neuron Subtypes (res = ", final_res, ")")) +
  theme(plot.title = element_text(hjust = 0.5))
print(p_umap)

ggsave(paste0(save_path, "ht_5ht_UMAP_res", gsub("\\.", "", final_res), ".png"),
       p_umap, width = 7, height = 6, dpi = 300)
ggsave(paste0(save_path, "ht_5ht_UMAP_res", gsub("\\.", "", final_res), ".pdf"),
       p_umap, width = 7, height = 6)



In [ ]:
# ---------- 保存 5-HT 子集对象（文件名带真实分辨率）----------
saveRDS(ht_5ht,
        paste0(save_path, "ht_5ht_nFeature1500_SCT_harmony_res",
               gsub("\\.", "", final_res), ".rds"))
cat("5-HT 子集对象已保存\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(future)
library(harmony)

save_path <- "./data/"

# ============================================
# 0. 读取完整对象 + 立刻备份
# ============================================
all_seurat <- readRDS(paste0(save_path, "classFiltSeurat_neurotype.rds"))

cat("===== 完整对象概况 =====\n")
cat("细胞数:", ncol(all_seurat), "\n")
cat("元数据列:", paste(colnames(all_seurat@meta.data), collapse = ", "), "\n")

In [ ]:
# ============================================
# 3. 新亚群名写回完整对象 → 先看 table → 再保存
# ============================================

all_seurat$cell_type_new <- as.character(all_seurat$cell_type)

# 5-HT 新亚群映射表（对象名 ht_5ht，与聚类代码一致）
map_new <- setNames(as.character(ht_5ht$ht_subtype), colnames(ht_5ht))
all_seurat$cell_type_new[names(map_new)] <- map_new

# ---------- 处理被 nFeature 过滤丢弃的 5-HT 细胞 ----------
old_5ht_cells <- rownames(all_seurat@meta.data)[
  grepl("^5-HT_", as.character(all_seurat$cell_type))
]
not_updated <- setdiff(old_5ht_cells, names(map_new))

cat("未被新聚类覆盖的 5-HT 细胞数:", length(not_updated), "\n")

# 这批细胞标为 5-HT_filtered，避免和新亚群旧名混淆
all_seurat$cell_type_new[not_updated] <- "5-HT_filtered"

all_seurat$cell_type_new <- factor(all_seurat$cell_type_new)

# ---------- 检查 ----------
cat("\n===== 写回后的全体细胞分布 =====\n")
print(table(all_seurat$cell_type_new))

cat("\n===== 5-HT 部分核对 =====\n")
cat(sprintf("原 5-HT 细胞总数: %d\n", length(old_5ht_cells)))
cat(sprintf("重新聚类并更新命名: %d\n", length(old_5ht_cells) - length(not_updated)))
cat(sprintf("因 nFeature<1500 被过滤: %d（已标记为 5-HT_filtered）\n", length(not_updated)))
cat(sprintf("新 5-HT 亚群数: %d\n", length(unique(ht_5ht$ht_subtype))))

# 总数自检：写回前后全体细胞数应完全一致（只改标签，不删细胞）
cat(sprintf("全体细胞数核对: 写回前 %d → 写回后 %d\n",
            nrow(all_seurat@meta.data), length(all_seurat$cell_type_new)))



In [ ]:
# ---------- 确认 table 输出无误后再执行保存 ----------
saveRDS(all_seurat,
        paste0(save_path, "all_seurat_with_newHT_res",
               gsub("\\.", "", final_res), ".rds"))
cat("完整对象已保存\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)

# ============================================================
save_path <- "./data/"

ht_seurat <- readRDS(paste0(save_path, "ht_5ht_nFeature1500_SCT_harmony_res05.rds"))

# ============================================
# 0. 快速检查
# ============================================
cat("===== 对象概况 =====\n")
cat("细胞数:", ncol(ht_seurat), "\n")
cat("基因数:", nrow(ht_seurat), "\n")
cat("Assays:", paste(Assays(ht_seurat), collapse = ", "), "\n")
cat("Reductions:", paste(Reductions(ht_seurat), collapse = ", "), "\n")

cat("\n===== 亚群分布 =====\n")
print(table(ht_seurat$ht_subtype))

# 确保因子顺序和 Ident 正确
ht_levels <- paste0("5-HT_", sort(as.numeric(gsub("5-HT_", "", 
                        unique(as.character(ht_seurat$ht_subtype))))))
ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = ht_levels)
ht_seurat <- SetIdent(ht_seurat, value = "ht_subtype")

# ★ 统一保存函数：PNG + PDF
save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================
# 图 1：UMAP 总览
# ============================================
p_umap <- DimPlot(ht_seurat, group.by = "ht_subtype", 
                  label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() +
  ggtitle("5-HT Neuron Subtypes") +
  theme(plot.title = element_text(hjust = 0.5))
print(p_umap)
save_fig(p_umap, paste0(save_path, "Fig1_UMAP_subtypes"), 7, 6)

# ============================================
# 图 2：按 donor / 样本看批次
# ============================================
p_donor <- DimPlot(ht_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
p_sample <- DimPlot(ht_seurat, group.by = "sample_label", raster = FALSE) + coord_fixed()
print(p_donor)
print(p_sample)
save_fig(p_donor,  paste0(save_path, "Fig2_UMAP_donor"),  7, 6)
save_fig(p_sample, paste0(save_path, "Fig2_UMAP_sample"), 7, 6)

# ============================================
# 图 3：核心标志基因 FeaturePlot
# ============================================
p_feat <- FeaturePlot(ht_seurat, 
                      features = c("TPH2", "SLC6A4", "FEV", "DDC"),
                      ncol = 2, raster = FALSE) & coord_fixed()
print(p_feat)
save_fig(p_feat, paste0(save_path, "Fig3_Feature_5HT_markers"), 9, 8)

# ============================================
# 图 4：核心标志 + 污染标志 DotPlot（质量复查）
# ============================================
p_dot <- DotPlot(ht_seurat,
                 features = c("TPH2","SLC6A4","FEV","DDC","TPH1",
                              "CHAT","SLC5A7","SLC17A6","GAD1","GAD2",
                              "CST3","APOE","PTPRZ1","ARHGAP15"),
                 group.by = "ht_subtype") + 
  RotatedAxis() +
  scale_color_gradientn(colors = c("lightgrey", "red"))
print(p_dot)
save_fig(p_dot, paste0(save_path, "Fig4_DotPlot_QC"), 8, 5)

# ============================================
# 图 5：经典 marker 小提琴图
# ============================================
p_vln <- VlnPlot(ht_seurat,
                 features = c("TPH2", "SLC6A4", "SLC17A6", "GAD1", "TH", "FEV"),
                 pt.size = 0, ncol = 3)
print(p_vln)
save_fig(p_vln, paste0(save_path, "Fig5_VlnPlot_markers"), 10, 6)

# ============================================
# 图 6：各亚群细胞数柱状图
# ============================================
cell_counts <- as.data.frame(table(ht_seurat$ht_subtype))
colnames(cell_counts) <- c("subtype", "n")

p_bar <- ggplot(cell_counts, aes(x = subtype, y = n, fill = subtype)) +
  geom_col() +
  geom_text(aes(label = n), vjust = -0.5, size = 3) +
  theme_classic() +
  labs(x = "", y = "Cell number", title = "5-HT subtype cell counts") +
  theme(legend.position = "none", 
        axis.text.x = element_text(angle = 45, hjust = 1))
print(p_bar)
save_fig(p_bar, paste0(save_path, "Fig6_subtype_counts"), 6, 4.5)

# ============================================
# 图 7：各 donor 中亚群比例（堆叠柱状图）
# ============================================
prop_df <- ht_seurat@meta.data %>%
  count(donor, ht_subtype) %>%
  group_by(donor) %>%
  mutate(prop = n / sum(n))

p_prop <- ggplot(prop_df, aes(x = donor, y = prop, fill = ht_subtype)) +
  geom_col(position = "fill") +
  theme_classic() +
  labs(x = "", y = "Proportion", fill = "Subtype",
       title = "Subtype composition per donor") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
print(p_prop)
save_fig(p_prop, paste0(save_path, "Fig7_subtype_proportion_donor"), 7, 5)

cat("\n✅ 所有图已保存到:", save_path, "\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

output_path <- "./data/"

# ============================================
# 0. 读取 res 0.5 的 5-HT 对象并设置 Identity
# ============================================
ht_seurat <- readRDS(paste0(output_path, "ht_5ht_nFeature1500_SCT_harmony_res05.rds"))

cat("细胞数:", ncol(ht_seurat), "\n")
print(table(ht_seurat$ht_subtype))

# 按数字顺序固定因子水平（5-HT_0 ~ 5-HT_14）
ht_levels <- paste0("5-HT_", sort(as.numeric(gsub("5-HT_", "",
                        unique(as.character(ht_seurat$ht_subtype))))))
ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = ht_levels)
Idents(ht_seurat) <- ht_seurat$ht_subtype

# ★ 统一保存函数：PNG + PDF
save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================
# 1. 找 Marker 基因 (Wilcoxon)
# ============================================
# SCT 数据找 marker 前必须 PrepSCTFindMarkers（校正测序深度差异）
ht_seurat <- PrepSCTFindMarkers(ht_seurat)

ht_markers <- FindAllMarkers(ht_seurat,
                             only.pos = TRUE,
                             min.pct = 0.25,
                             logfc.threshold = 0.25,
                             test.use = "wilcox")

# 保存完整 marker 表 + Top20
write.csv(ht_markers, paste0(output_path, "ht5ht_AllMarkers.csv"), row.names = FALSE)

ht_top20 <- ht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(ht_top20, paste0(output_path, "ht5ht_Subtype_top20_markers.csv"), row.names = FALSE)
cat("5-HT Top20 markers 已保存\n")

# ============================================
# 2. Top5 DotPlot（15 群 × 最多 75 基因）
# ============================================
ht_top5 <- ht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

ht_gene_order5 <- unique(ht_top5$gene)

p_ht_dot5 <- DotPlot(ht_seurat, features = ht_gene_order5, group.by = "ht_subtype",
                     dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("5-HT subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_ht_dot5)
save_fig(p_ht_dot5, paste0(output_path, "ht5ht_Fig8_Top5_DotPlot"), 14, 14)

write.csv(ht_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "ht5ht_Subtype_Top5_table.csv"), row.names = FALSE)
cat("Top5 点图已保存\n")

# ============================================
# 3. Top10 DotPlot（15 群 × 最多 150 基因）
# ============================================
ht_top10 <- ht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

ht_gene_order10 <- unique(ht_top10$gene)

p_ht_dot10 <- DotPlot(ht_seurat, features = ht_gene_order10, group.by = "ht_subtype",
                      dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("5-HT subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_ht_dot10)
save_fig(p_ht_dot10, paste0(output_path, "ht5ht_Fig9_Top10_DotPlot"), 15, 18)

write.csv(ht_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "ht5ht_Subtype_Top10_table.csv"), row.names = FALSE)
cat("Top10 点图已保存\n")

# ============================================
# 4. Top20 DotPlot（15 群 × 最多 300 基因）
# ============================================
ht_gene_order20 <- unique(ht_top20$gene)

p_ht_dot20 <- DotPlot(ht_seurat, features = ht_gene_order20, group.by = "ht_subtype",
                      dot.scale = 3, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("5-HT subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 5.5),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_ht_dot20)
save_fig(p_ht_dot20, paste0(output_path, "ht5ht_Fig10_Top20_DotPlot"), 16, 26)

write.csv(ht_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "ht5ht_Subtype_Top20_table.csv"), row.names = FALSE)
cat("Top20 点图已保存\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)

output_path <- "./data"

# ============================================
# 0. 读取 5-HT 对象（如已在内存可跳过）
# ============================================
ht_seurat <- readRDS(paste0(output_path, "ht_5ht_nFeature1500_SCT_harmony_res05.rds"))

# 固定亚群顺序 5-HT_0 ~ 5-HT_14
ht_levels <- paste0("5-HT_", sort(as.numeric(gsub("5-HT_", "",
                        unique(as.character(ht_seurat$ht_subtype))))))
ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = ht_levels)

# ★ 统一保存函数：PNG + PDF
save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================
# 1. 指定基因列表（按 5-HT 亚群 0~14 分组，共 44 个）
# ============================================
ht_gene_list_raw <- c(
  # 5-HT_0
  "DAPL1", "CHIN2", "DOCK8",
  # 5-HT_1
  "NTS", "CNBD1", "MCOLN3",
  # 5-HT_2
  "SATB2", "LVRN", "ITPR2",
  # 5-HT_3
  "PDLIM3", "LDB2", "SYT10",
  # 5-HT_4
  "TLL1", "NXPH1", "SLCO1C1",
  # 5-HT_5
  "PLCH1", "DLK1", "BMPR1B",
  # 5-HT_6
  "RASSF6", "KCNH1", "KCNH7",
  # 5-HT_7
  "PDE1C", "LUZP2", "C1QL1",
  # 5-HT_8
  "BCL11A", "ANO1", "NFIX",
  # 5-HT_9
  "ABI3BP", "ARHGAP36", "RSPO2",
  # 5-HT_10
  "PENK", "TRPC4", "GRB14",
  # 5-HT_11
  "CALCR", "MBNL3", "SFRP1",
  # 5-HT_12
  "IGF1", "GRIN3A", "PARD3B",
  # 5-HT_13
  "UTS2B", "BCHE", "CLRN1",
  # 5-HT_14
  "EPHA4", "CHST9", "SDC2"
)

# ============================================
# 2. 检查基因可用性
# ============================================
ht_available_genes <- ht_gene_list_raw[ht_gene_list_raw %in% rownames(ht_seurat)]
ht_missing_genes <- ht_gene_list_raw[!ht_gene_list_raw %in% rownames(ht_seurat)]

cat("总基因数:", length(ht_gene_list_raw), "\n")
cat("可用基因数:", length(ht_available_genes), "\n")

if (length(ht_missing_genes) > 0) {
  cat("\n⚠️ 以下基因未找到，已跳过:\n")
  cat(paste(ht_missing_genes, collapse = ", "), "\n")
}

# ============================================
# 3. DotPlot（x 轴 = 亚群, y 轴 = 基因）
# ============================================
p_ht_custom <- DotPlot(
  ht_seurat,
  features = ht_available_genes,
  group.by = "ht_subtype",
  dot.scale = 4,
  cols = c("lightgrey", "red")
) +
  coord_flip() +
  RotatedAxis() +
  ggtitle("5-HT subtypes: selected markers") +
  xlab("5-HT subtype") +
  ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9, face = "bold"),
    axis.text.y = element_text(size = 8),
    axis.title = element_text(size = 12, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )

print(p_ht_custom)

save_fig(p_ht_custom, paste0(output_path, "ht5ht_Fig11_CustomMarkers_DotPlot"), 12, 14)
cat("5-HT 指定基因点图已保存!\n")

In [ ]:
library(viridis)
library(ranger)
library(pheatmap)
library(grid)

output_path <- "./data/"

# ============================================
# 0. 读取 5-HT 对象 + 固定亚群顺序
# ============================================
# ht_seurat <- readRDS(paste0(output_path, "ht_5ht_nFeature1500_SCT_harmony_res05.rds"))

ht_type_order <- paste0("5-HT_", sort(as.numeric(gsub("5-HT_", "",
                            unique(as.character(ht_seurat$ht_subtype))))))
ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = ht_type_order)
Idents(ht_seurat) <- ht_seurat$ht_subtype

# ============================================
# 1. Ranger（RNA assay 高变基因）
# ============================================
DefaultAssay(ht_seurat) <- "RNA"

if (length(VariableFeatures(ht_seurat, assay = "RNA")) == 0) {
  cat("RNA assay VariableFeatures 为空，重新计算...\n")
  ht_seurat <- FindVariableFeatures(ht_seurat, selection.method = "vst", nfeatures = 3000)
}

expr_matrix_ht <- GetAssayData(ht_seurat, layer = "data", assay = "RNA")
genes_ht <- intersect(VariableFeatures(ht_seurat, assay = "RNA"),
                      rownames(expr_matrix_ht))

cat("RNA assay 基因数:", nrow(expr_matrix_ht), "\n")
cat("高变基因数:", length(genes_ht), "\n")

X_ht <- t(as.matrix(expr_matrix_ht[genes_ht, , drop = FALSE]))
y_ht <- factor(ht_seurat$ht_subtype)

cat("X_ht 维度:", dim(X_ht), "\n")

rf_ht <- ranger(x = X_ht, y = y_ht,
                num.trees = 500, num.threads = 8,
                classification = TRUE)

cat("\n5-HT OOB Error:", rf_ht$prediction.error, "\n")

conf_mat_ht <- rf_ht$confusion.matrix
confusionRatio_ht <- t(apply(conf_mat_ht, 1, function(x) {
  if (sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("5-HT 对角线准确率:\n")
print(round(diag(confusionRatio_ht), 2))

# ---- 准备矩阵（固定亚群顺序，阈值 20）----
df_num_ht <- as.data.frame(table(ht_seurat$ht_subtype))
select_cell_ht <- as.character(df_num_ht[df_num_ht$Freq >= 20, ]$Var1)

ht_type_order_present <- ht_type_order[ht_type_order %in% select_cell_ht]
mat_to_plot_ht <- confusionRatio_ht[ht_type_order_present,
                                    ht_type_order_present, drop = FALSE]

cat("\n5-HT 热图包含类型数:", nrow(mat_to_plot_ht), "\n")

# ============================================
# 2. 精确尺寸保存函数（PNG + PDF，绝不截断）
# ============================================
save_pheatmap_exact <- function(ph_obj, file_base, dpi = 150) {

  w_in <- convertWidth(sum(ph_obj$gtable$widths),  "inch", valueOnly = TRUE)
  h_in <- convertHeight(sum(ph_obj$gtable$heights), "inch", valueOnly = TRUE)

  cat(sprintf("实际尺寸: %.1f x %.1f inch\n", w_in, h_in))

  png(paste0(file_base, ".png"),
      width = w_in, height = h_in, units = "in", res = dpi)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  pdf(paste0(file_base, ".pdf"), width = w_in, height = h_in)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ---- 带数值版本 ----
ht_num_labels <- matrix("", nrow = nrow(mat_to_plot_ht),
                        ncol = ncol(mat_to_plot_ht))
diag(ht_num_labels) <- sprintf("%.2f", diag(mat_to_plot_ht))

p_ht_num <- pheatmap(mat_to_plot_ht,
                     breaks = seq(0, 1, 0.01),
                     cluster_rows = FALSE,
                     cluster_cols = FALSE,
                     border_color = NA,
                     cellwidth = 30, cellheight = 30, fontsize = 10,
                     color = viridis(100),
                     main = "5-HT Subtypes Ranger Confusion Matrix",
                     display_numbers = ht_num_labels,
                     fontsize_number = 8, number_color = "white")

save_pheatmap_exact(p_ht_num,
                    paste0(output_path, "ht5ht_Ranger_Heatmap_withNum"))

# ---- 不带数值版本 ----
p_ht_nonum <- pheatmap(mat_to_plot_ht,
                       breaks = seq(0, 1, 0.01),
                       cluster_rows = FALSE,
                       cluster_cols = FALSE,
                       border_color = NA,
                       cellwidth = 30, cellheight = 30, fontsize = 10,
                       color = viridis(100),
                       main = "5-HT Subtypes Ranger Confusion Matrix",
                       display_numbers = FALSE)

save_pheatmap_exact(p_ht_nonum,
                    paste0(output_path, "ht5ht_Ranger_Heatmap_noNum"))

# ---- 屏幕显示 ----
grid.newpage()
grid.draw(p_ht_nonum$gtable)

cat("5-HT Ranger 热图全部处理完成\n")

In [ ]:
library(Seurat)
library(ggplot2)

output_path <- "./data/"

# ★ 统一保存函数：PNG + PDF
save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================
# 6. 小提琴图（按 5-HT 亚群）
# ============================================
ht_plot_data <- ht_seurat@meta.data
ht_plot_data$ht_subtype <- factor(ht_plot_data$ht_subtype, levels = ht_type_order)

# 6.1 nFeature_RNA（刻度每 1000）
ht_nfeature_max <- max(ht_plot_data$nFeature_RNA, na.rm = TRUE)
ht_nfeature_breaks <- seq(0, ceiling(ht_nfeature_max / 1000) * 1000, by = 1000)

p_ht_nfeature <- ggplot(ht_plot_data,
                        aes(x = ht_subtype, y = nFeature_RNA, fill = ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7,
              linewidth = 0.3, color = "black") +
  scale_y_continuous(breaks = ht_nfeature_breaks) +
  labs(x = "", y = "nFeature RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"),
    axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_ht_nfeature)
save_fig(p_ht_nfeature,
         paste0(output_path, "ht5ht_violin_nFeature_RNA_1k"), 12, 6)

# 6.2 nCount_RNA
p_ht_ncount <- ggplot(ht_plot_data,
                      aes(x = ht_subtype, y = nCount_RNA, fill = ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7,
              linewidth = 0.3, color = "black") +
  labs(x = "", y = "nCount RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"),
    axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_ht_ncount)
save_fig(p_ht_ncount,
         paste0(output_path, "ht5ht_violin_nCount_RNA"), 12, 6)

cat("5-HT 小提琴图已保存\n")

In [ ]:
library(Seurat)

cat("===== 读取保存的 Seurat 对象 =====\n")

save_path <- "./data/"

classFiltSeurat_filt <- readRDS(
  paste0(save_path, "all_seurat_with_newHT_res05.rds")
)

cat("读取成功，总细胞数:", ncol(classFiltSeurat_filt), "\n")
cat("总基因数:", nrow(classFiltSeurat_filt), "\n")

cat("\ncell_type 分布:\n")
print(table(classFiltSeurat_filt$cell_type))

if ("major_group" %in% colnames(classFiltSeurat_filt@meta.data)) {
  cat("\nmajor_group 分布:\n")
  print(table(classFiltSeurat_filt$major_group))
}

cat("\n对象已加载到变量: classFiltSeurat_filt\n")

In [ ]:
# 对非神经元分别聚类

In [ ]:
library(Seurat)
library(ggplot2)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

# ★ 统一保存函数：PNG + PDF
save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# # ============================================
# # 0. 读取完整对象
# # ============================================
# all_seurat <- readRDS(paste0(output_path, "classFiltSeurat_neurotype.rds"))

# cat("===== Astrocytes 重新聚类 =====\n")

# ---------- 1. 提取 Astrocytes ----------
astro_cells <- colnames(all_seurat)[all_seurat$cell_type == "Astrocytes"]
astro_seurat <- subset(all_seurat, cells = astro_cells)
cat("Astrocytes 细胞数:", ncol(astro_seurat), "\n")
print(table(astro_seurat$donor))

# ---------- 2. RNA 标准化 + PCA ----------
DefaultAssay(astro_seurat) <- "RNA"
astro_seurat <- NormalizeData(astro_seurat, verbose = FALSE)
astro_seurat <- FindVariableFeatures(astro_seurat, selection.method = "vst",
                                     nfeatures = 2000, verbose = FALSE)
astro_seurat <- ScaleData(astro_seurat, verbose = FALSE)
astro_seurat <- RunPCA(astro_seurat, features = VariableFeatures(astro_seurat),
                       npcs = 50, verbose = FALSE)

# ---------- 3. Harmony（donor 批次校正，与 5-HT 流程口径一致）----------
astro_seurat <- RunHarmony(astro_seurat,
                           group.by.vars = "donor",
                           reduction.use = "pca",
                           dims.use = 1:30,
                           verbose = FALSE)

# ---------- 4. 邻接图 + 多分辨率聚类 + clustree ----------
astro_seurat <- FindNeighbors(astro_seurat, reduction = "harmony", dims = 1:30,
                              verbose = FALSE)

resolutions_to_try <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.8, 1.0)
for (res in resolutions_to_try) {
  astro_seurat <- FindClusters(astro_seurat, resolution = res,
                               future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(astro_seurat, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "astro_clustree"), 12, 9)

cat("\n各分辨率聚类情况:\n")
for (res in resolutions_to_try) {
  tab <- table(astro_seurat@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.1f: %d 群, 最小群 %d 细胞\n",
              res, length(tab), min(tab)))
}



In [ ]:
# ---------- 5. 选定分辨率 → UMAP + 重命名 ----------
final_res <- 0.1   # ← 看 clustree 后修改

astro_seurat <- SetIdent(astro_seurat,
                         value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(astro_seurat)))
cl_new  <- as.character(Idents(astro_seurat))
cl_map  <- setNames(paste0("Astrocytes_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

astro_seurat$subtype <- factor(unname(cl_map[cl_new]),
                               levels = paste0("Astrocytes_", 0:(n_clust - 1)))
astro_seurat <- SetIdent(astro_seurat, value = "subtype")

cat("\nAstrocytes 亚群分布 (res =", final_res, "):\n")
print(table(astro_seurat$subtype))

astro_seurat <- RunUMAP(astro_seurat, reduction = "harmony", dims = 1:30,
                        verbose = FALSE)

# ---------- 6. 可视化 ----------
p1 <- DimPlot(astro_seurat, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Astrocytes Subtypes (res = ", final_res, ")"))
print(p1)
save_fig(p1, paste0(output_path, "astro_Fig1_UMAP_subtypes"), 7, 6)

p1b <- DimPlot(astro_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "astro_Fig2_UMAP_donor"), 7, 6)

p2 <- VlnPlot(astro_seurat, features = c("GFAP", "AQP4", "SLC1A3"),
              pt.size = 0, ncol = 3)
print(p2)
save_fig(p2, paste0(output_path, "astro_Fig3_Vln_ClassicMarkers"), 10, 4)

# 污染/身份复查：确认没有混入神经元或少突
p_qc <- DotPlot(astro_seurat,
                features = c("GFAP","AQP4","SLC1A3","ALDH1L1",
                             "SLC1A2","PLP1","MBP","SNAP25","SYT1","SLC17A6","GAD1"),
                group.by = "subtype") +
  RotatedAxis() +
  scale_color_gradientn(colors = c("lightgrey", "red"))
print(p_qc)
save_fig(p_qc, paste0(output_path, "astro_Fig4_DotPlot_QC"), 9, 5)



In [ ]:
# ============================================
# 身份确认 DotPlot
# ============================================
check_genes <- c(
  # 星形胶质（应只在 0/1 亮）
  "GFAP", "AQP4", "SLC1A2", "SLC1A3", "AGT", "FGFR3", "SLC39A12",
  # 神经元（应只在 2 亮）
  "SNAP25", "SYT1", "TUBB3", "NEFL", "CALB2", "TPH2", "ACHE",
  # 小胶质（应只在 3 亮）
  "CSF1R", "PTPRC", "AIF1", "TYROBP", "C1QA", "CX3CR1",
  # 排除其他谱系（应全暗）
  "PLP1", "CLDN5", "DCN"
)

p_dot <- DotPlot(astro_seurat, features = check_genes,
                 group.by = "subtype",
                 dot.scale = 6,
                 cols = c("lightgrey", "red")) +
  RotatedAxis() +
  ggtitle("Astro subtypes: identity check") +
  theme(
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10, face = "bold")
  )

print(p_dot)
save_fig(p_dot, paste0(output_path, "astro_Fig8_DotPlot_IdentityCheck"), 12, 4.5)

cat("确认图已保存\n")

In [ ]:
astro_removed <- subset(astro_seurat,
                        subset = subtype %in% c("Astrocytes_2", "Astrocytes_3"))
# saveRDS(astro_removed, paste0(output_path, "astro_removed_2_neuron_3_micro.rds"))
cat("已剔除并另存:", ncol(astro_removed), "个细胞\n")
print(table(astro_removed$subtype))

# 干净星形细胞
astro_clean <- subset(astro_seurat,
                      subset = subtype %in% c("Astrocytes_0", "Astrocytes_1"))
cat("清洗后星形细胞数:", ncol(astro_clean), "\n")

# ============================================
# 1. 重新标准化 + PCA（清洗后必须重跑，不能沿用旧空间）
# ============================================
DefaultAssay(astro_clean) <- "RNA"

astro_clean <- NormalizeData(astro_clean, verbose = FALSE)
astro_clean <- FindVariableFeatures(astro_clean, selection.method = "vst",
                                    nfeatures = 2000, verbose = FALSE)
astro_clean <- ScaleData(astro_clean, verbose = FALSE)
astro_clean <- RunPCA(astro_clean, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony（donor 校正）
# ============================================
astro_clean <- RunHarmony(astro_clean,
                          group.by.vars = "donor",
                          reduction.use = "pca",
                          dims.use = 1:30,
                          verbose = FALSE)

# ============================================
# 3. 邻接图 + 多分辨率聚类 + clustree
# ============================================
astro_clean <- FindNeighbors(astro_clean, reduction = "harmony",
                             dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.8, 1.0)
for (res in resolutions_to_try) {
  astro_clean <- FindClusters(astro_clean, resolution = res,
                              future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(astro_clean, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "astroClean_clustree"), 12, 9)

cat("\n各分辨率聚类情况:\n")
for (res in resolutions_to_try) {
  tab <- table(astro_clean@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.1f: %d 群, 最小群 %d 细胞\n",
              res, length(tab), min(tab)))
}


In [ ]:
# ============================================
# 4. 选定分辨率 → 正式聚类 + 命名 + UMAP
# ============================================
final_res <- 0.1   # ← 看 clustree 后修改

astro_clean <- SetIdent(astro_clean, value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(astro_clean)))
cl_new  <- as.character(Idents(astro_clean))
cl_map  <- setNames(paste0("Astrocytes_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

astro_clean$subtype <- factor(unname(cl_map[cl_new]),
                              levels = paste0("Astrocytes_", 0:(n_clust - 1)))
astro_clean <- SetIdent(astro_clean, value = "subtype")

cat("\n清洗后亚群分布 (res =", final_res, "):\n")
print(table(astro_clean$subtype))

astro_clean <- RunUMAP(astro_clean, reduction = "harmony", dims = 1:30,
                       verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(astro_clean, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Astrocytes (cleaned) res = ", final_res))
print(p1)
save_fig(p1, paste0(output_path, "astroClean_Fig1_UMAP"), 7, 6)

p1b <- DimPlot(astro_clean, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "astroClean_Fig2_UMAP_donor"), 7, 6)


# 差异 marker（清洗后重算）
astro_clean_markers <- FindAllMarkers(astro_clean, only.pos = TRUE,
                                      min.pct = 0.25, logfc.threshold = 0.25)
write.csv(astro_clean_markers,
          paste0(output_path, "astroClean_AllMarkers.csv"), row.names = FALSE)


In [ ]:

# ============================================
# 6. 保存
# ============================================
saveRDS(astro_clean,
        paste0(output_path, "astroClean_seurat_harmony_res",
               gsub("\\.", "", final_res), ".rds"))
cat("清洗后的 astro 对象已保存\n")

# ---------- 写回完整对象（先 table 确认再放开）----------
all_seurat <- readRDS(paste0(output_path, "classFiltSeurat_neurotype.rds"))

all_seurat$cell_type_new <- as.character(all_seurat$cell_type)

# 新亚群名写回
map_clean <- setNames(as.character(astro_clean$subtype), colnames(astro_clean))
all_seurat$cell_type_new[names(map_clean)] <- map_clean

# 被剔除的另标
map_2 <- colnames(astro_removed)[astro_removed$subtype == "Astrocytes_2"]
map_3 <- colnames(astro_removed)[astro_removed$subtype == "Astrocytes_3"]
all_seurat$cell_type_new[map_2] <- "Neuron_contam"
all_seurat$cell_type_new[map_3] <- "Microglia"

all_seurat$cell_type_new <- factor(all_seurat$cell_type_new)

cat("\n===== 写回后全体分布 =====\n")
print(table(all_seurat$cell_type_new))

In [ ]:
# ---------- 区分两批 Microglia ----------
is_original_micro <- all_seurat$cell_type_new == "Microglia" &
                     all_seurat$cell_type == "Microglia"

is_from_astro <- all_seurat$cell_type_new == "Microglia" &
                 all_seurat$cell_type == "Astrocytes"

cat("原有 Microglia:", sum(is_original_micro), "\n")
cat("从 Astro 改标的 Microglia:", sum(is_from_astro), "\n")

In [ ]:
cells_to_remove <- c(
  colnames(all_seurat)[all_seurat$cell_type_new == "Neuron_contam"],
  colnames(all_seurat)[is_from_astro]
)

cat("将删除:", length(cells_to_remove),
    "个细胞（Neuron_contam + Microglia_fromAstro）\n")

# saveRDS(all_seurat, paste0(output_path, "all_seurat_backup_beforeDel.rds"))

all_seurat <- subset(all_seurat, cells = setdiff(colnames(all_seurat), cells_to_remove))
all_seurat$cell_type_new <- factor(droplevels(all_seurat$cell_type_new))

cat("\n===== 删除后全体分布 =====\n")
print(table(all_seurat$cell_type_new))
# 核对：Neuron_contam 消失，Microglia 恰好 6277

# saveRDS(all_seurat, paste0(output_path, "all_seurat_astroClean_v2.rds"))

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

cat("===== Astrocytes (cleaned) Marker 分析 =====\n")

# ============================================
# 0. 从内存中的 all_seurat 提取清洗后的星形细胞
# ============================================
astro_seurat <- subset(all_seurat,
                       subset = cell_type_new %in% c("Astrocytes_0", "Astrocytes_1"))

cat("提取的星形细胞数:", ncol(astro_seurat), "\n")
print(table(astro_seurat$cell_type_new))

DefaultAssay(astro_seurat) <- "RNA"

astro_seurat$subtype <- factor(as.character(astro_seurat$cell_type_new),
                               levels = c("Astrocytes_0", "Astrocytes_1"))

# 1. 设置 Idents
Idents(astro_seurat) <- astro_seurat$subtype

# 2. 找 Marker 基因
astro_markers <- FindAllMarkers(astro_seurat,
                                only.pos = TRUE,
                                min.pct = 0.25,
                                logfc.threshold = 0.25,
                                test.use = "wilcox")

write.csv(astro_markers,
          paste0(output_path, "astroClean_AllMarkers.csv"), row.names = FALSE)

# 3. Top20 保存 CSV
astro_top20 <- astro_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(astro_top20,
          paste0(output_path, "astroClean_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers 已保存\n")

# 4. Top5 DotPlot
astro_top5 <- astro_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(astro_top5$gene)

p_astro_dot5 <- DotPlot(astro_seurat, features = gene_order5,
                        group.by = "subtype",
                        dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned Astro subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_astro_dot5)
save_fig(p_astro_dot5, paste0(output_path, "astroClean_Fig4_Top5_DotPlot"), 9, 6)

write.csv(astro_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "astroClean_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 点图已保存\n")

# 5. Top10 DotPlot
astro_top10 <- astro_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(astro_top10$gene)

p_astro_dot10 <- DotPlot(astro_seurat, features = gene_order10,
                         group.by = "subtype",
                         dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned Astro subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_astro_dot10)
save_fig(p_astro_dot10, paste0(output_path, "astroClean_Fig5_Top10_DotPlot"), 9, 8)

write.csv(astro_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "astroClean_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 点图已保存\n")

# 6. Top20 DotPlot
gene_order20 <- unique(astro_top20$gene)

p_astro_dot20 <- DotPlot(astro_seurat, features = gene_order20,
                         group.by = "subtype",
                         dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned Astro subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_astro_dot20)
save_fig(p_astro_dot20, paste0(output_path, "astroClean_Fig6_Top20_DotPlot"), 10, 12)

write.csv(astro_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "astroClean_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 点图已保存\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================
# 0. 读取完整对象 + 提取 Oligo
# ============================================
# all_seurat <- readRDS(paste0(output_path, "classFiltSeurat_neurotype.rds"))

cat("===== Oligo 重新聚类 =====\n")

oligo_cells <- colnames(all_seurat)[all_seurat$cell_type == "Oligo"]
oligo_seurat <- subset(all_seurat, cells = oligo_cells)
cat("Oligo 细胞数:", ncol(oligo_seurat), "\n")
print(table(oligo_seurat$donor))

# ============================================
# 1. RNA 标准化 + PCA
# ============================================
DefaultAssay(oligo_seurat) <- "RNA"
oligo_seurat <- NormalizeData(oligo_seurat, verbose = FALSE)
oligo_seurat <- FindVariableFeatures(oligo_seurat, selection.method = "vst",
                                     nfeatures = 2000, verbose = FALSE)
oligo_seurat <- ScaleData(oligo_seurat, verbose = FALSE)
oligo_seurat <- RunPCA(oligo_seurat, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony（donor 校正）
# ============================================
oligo_seurat <- RunHarmony(oligo_seurat,
                           group.by.vars = "donor",
                           reduction.use = "pca",
                           dims.use = 1:30,
                           verbose = FALSE)

# ============================================
# 3. 邻接图 + 多分辨率聚类 + clustree
# ============================================
oligo_seurat <- FindNeighbors(oligo_seurat, reduction = "harmony",
                              dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.8, 1.0)
for (res in resolutions_to_try) {
  oligo_seurat <- FindClusters(oligo_seurat, resolution = res,
                               future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(oligo_seurat, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "oligo_clustree"), 12, 9)

cat("\n各分辨率聚类情况:\n")
for (res in resolutions_to_try) {
  tab <- table(oligo_seurat@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.1f: %d 群, 最小群 %d 细胞\n",
              res, length(tab), min(tab)))
}

In [ ]:
# ============================================
# 4. 选定分辨率 → 正式聚类 + 命名 + UMAP
# ============================================
final_res <- 0.1   # ← 看 clustree 后修改（Oligo 通常 0.1~0.2 就够）

oligo_seurat <- SetIdent(oligo_seurat, value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(oligo_seurat)))
cl_new  <- as.character(Idents(oligo_seurat))
cl_map  <- setNames(paste0("Oligo_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

oligo_seurat$subtype <- factor(unname(cl_map[cl_new]),
                               levels = paste0("Oligo_", 0:(n_clust - 1)))
oligo_seurat <- SetIdent(oligo_seurat, value = "subtype")

cat("\nOligo 亚群分布 (res =", final_res, "):\n")
print(table(oligo_seurat$subtype))

oligo_seurat <- RunUMAP(oligo_seurat, reduction = "harmony", dims = 1:30,
                        verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(oligo_seurat, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Oligo Subtypes (res = ", final_res, ")"))
print(p1)
save_fig(p1, paste0(output_path, "oligo_Fig1_UMAP_subtypes"), 7, 6)

p1b <- DimPlot(oligo_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "oligo_Fig2_UMAP_donor"), 7, 6)

p1c <- DimPlot(oligo_seurat, group.by = "sample_label", raster = FALSE) + coord_fixed()
print(p1c)
save_fig(p1c, paste0(output_path, "oligo_Fig2_UMAP_sample"), 7, 6)

# ---------- 经典 marker 小提琴 ----------
p2 <- VlnPlot(oligo_seurat, features = c("MBP", "MOG", "PLP1", "MAG",
                                         "MOBP", "OPALIN"),
              pt.size = 0, ncol = 3)
print(p2)
save_fig(p2, paste0(output_path, "oligo_Fig3_Vln_ClassicMarkers"), 10, 6)


# ---------- marker 表 ----------
oligo_markers <- FindAllMarkers(oligo_seurat, only.pos = TRUE,
                                min.pct = 0.25, logfc.threshold = 0.25)
write.csv(oligo_markers,
          paste0(output_path, "oligo_AllMarkers.csv"), row.names = FALSE)

In [ ]:
DefaultAssay(oligo_seurat) <- "RNA"

oligo_levels <- paste0("Oligo_", sort(as.numeric(
  gsub("Oligo_", "", unique(as.character(oligo_seurat$subtype))))))
oligo_seurat$subtype <- factor(oligo_seurat$subtype, levels = oligo_levels)

Idents(oligo_seurat) <- oligo_seurat$subtype

# ============================================
# 1. 找 Marker
# ============================================
oligo_markers <- FindAllMarkers(oligo_seurat,
                                only.pos = TRUE,
                                min.pct = 0.25,
                                logfc.threshold = 0.25,
                                test.use = "wilcox")

write.csv(oligo_markers,
          paste0(output_path, "oligo_AllMarkers.csv"), row.names = FALSE)

# ============================================
# 2. Top20 CSV
# ============================================
oligo_top20 <- oligo_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(oligo_top20,
          paste0(output_path, "oligo_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers 已保存\n")

# ============================================
# 3. Top5 DotPlot
# ============================================
oligo_top5 <- oligo_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(oligo_top5$gene)

p_oligo_dot5 <- DotPlot(oligo_seurat, features = gene_order5,
                        group.by = "subtype",
                        dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Oligo subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_oligo_dot5)
save_fig(p_oligo_dot5, paste0(output_path, "oligo_Fig5_Top5_DotPlot"), 9, 6)

write.csv(oligo_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "oligo_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 点图已保存\n")

# ============================================
# 4. Top10 DotPlot
# ============================================
oligo_top10 <- oligo_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(oligo_top10$gene)

p_oligo_dot10 <- DotPlot(oligo_seurat, features = gene_order10,
                         group.by = "subtype",
                         dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Oligo subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_oligo_dot10)
save_fig(p_oligo_dot10, paste0(output_path, "oligo_Fig6_Top10_DotPlot"), 9, 8)

write.csv(oligo_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "oligo_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 点图已保存\n")

# ============================================
# 5. Top20 DotPlot
# ============================================
gene_order20 <- unique(oligo_top20$gene)

p_oligo_dot20 <- DotPlot(oligo_seurat, features = gene_order20,
                         group.by = "subtype",
                         dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Oligo subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_oligo_dot20)
save_fig(p_oligo_dot20, paste0(output_path, "oligo_Fig7_Top20_DotPlot"), 10, 12)

write.csv(oligo_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "oligo_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 点图已保存\n")

In [ ]:
# ---------- 1. 确认点图：重点看 Oligo_0 的 OPALIN/MYRF vs 神经元基因 ----------
DefaultAssay(oligo_seurat) <- "RNA"

check_genes <- c("OPALIN", "MYRF", "MAG", "CLDN11", "MBP",      # 少突（0/1 应亮）
                 "SNAP25", "SYT1", "TUBB3", "NEFL", "CHGB",     # 神经元（应只在 2 亮）
                 "PDGFRA", "CSPG4",                            # OPC
                 "GFAP", "AQP4",                               # 星形
                 "CSF1R", "AIF1")                              # 小胶质

p_check <- DotPlot(oligo_seurat, features = check_genes,
                   group.by = "subtype",
                   dot.scale = 6, cols = c("lightgrey", "red")) +
  RotatedAxis() +
  ggtitle("Oligo subtypes: identity check") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
        axis.text.y = element_text(size = 10, face = "bold"))
print(p_check)
ggsave(paste0(output_path, "oligo_Fig8_DotPlot_IdentityCheck.png"),
       p_check, width = 12, height = 4.5, dpi = 300)
ggsave(paste0(output_path, "oligo_Fig8_DotPlot_IdentityCheck.pdf"),
       p_check, width = 12, height = 4.5)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================
# 0. 剔除 Oligo_2（神经元），只留真少突
# ============================================
oligo_clean <- subset(oligo_seurat,
                      subset = subtype %in% c("Oligo_0", "Oligo_1"))
cat("清洗后少突细胞数:", ncol(oligo_clean), "\n")

# ============================================
# 1. 重新标准化 + PCA
# ============================================
DefaultAssay(oligo_clean) <- "RNA"

oligo_clean <- NormalizeData(oligo_clean, verbose = FALSE)
oligo_clean <- FindVariableFeatures(oligo_clean, selection.method = "vst",
                                    nfeatures = 2000, verbose = FALSE)
oligo_clean <- ScaleData(oligo_clean, verbose = FALSE)
oligo_clean <- RunPCA(oligo_clean, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony（donor 校正）
# ============================================
oligo_clean <- RunHarmony(oligo_clean,
                          group.by.vars = "donor",
                          reduction.use = "pca",
                          dims.use = 1:30,
                          verbose = FALSE)

# ============================================
# 3. 邻接图 + 多分辨率聚类 + clustree
# ============================================
oligo_clean <- FindNeighbors(oligo_clean, reduction = "harmony",
                             dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.8)
for (res in resolutions_to_try) {
  oligo_clean <- FindClusters(oligo_clean, resolution = res,
                              future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(oligo_clean, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "oligoClean_clustree"), 12, 9)

cat("\n各分辨率聚类情况:\n")
for (res in resolutions_to_try) {
  tab <- table(oligo_clean@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.1f: %d 群, 最小群 %d 细胞\n",
              res, length(tab), min(tab)))
}


In [ ]:
# ============================================
# 4. 选定分辨率 → 正式聚类 + 命名 + UMAP
# ============================================
final_res_clean <- 0.1   # ← 看 clustree 后修改

oligo_clean <- SetIdent(oligo_clean, value = paste0("RNA_snn_res.", final_res_clean))

n_clust <- length(unique(Idents(oligo_clean)))
cl_new  <- as.character(Idents(oligo_clean))
cl_map  <- setNames(paste0("Oligo_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

oligo_clean$subtype <- factor(unname(cl_map[cl_new]),
                              levels = paste0("Oligo_", 0:(n_clust - 1)))
oligo_clean <- SetIdent(oligo_clean, value = "subtype")

cat("\n清洗后亚群分布 (res =", final_res_clean, "):\n")
print(table(oligo_clean$subtype))

oligo_clean <- RunUMAP(oligo_clean, reduction = "harmony", dims = 1:30,
                       verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(oligo_clean, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Oligo (cleaned) res = ", final_res_clean))
print(p1)
save_fig(p1, paste0(output_path, "oligoClean_Fig1_UMAP"), 7, 6)

p1b <- DimPlot(oligo_clean, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "oligoClean_Fig2_UMAP_donor"), 7, 6)

# ---------- 经典 marker 小提琴 ----------
p2 <- VlnPlot(oligo_clean, features = c("MBP", "MOG", "PLP1", "MAG",
                                        "MOBP", "OPALIN", "ENPP6", "BCAS1"),
              pt.size = 0, ncol = 4)
print(p2)
save_fig(p2, paste0(output_path, "oligoClean_Fig3_Vln_ClassicMarkers"), 12, 6)


# ---------- marker 表 ----------
oligo_clean_markers <- FindAllMarkers(oligo_clean, only.pos = TRUE,
                                      min.pct = 0.25, logfc.threshold = 0.25)
write.csv(oligo_clean_markers,
          paste0(output_path, "oligoClean_AllMarkers.csv"), row.names = FALSE)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

cat("===== Oligo (cleaned) 重新计算 Marker =====\n")

# ============================================
# 0. 确保对象状态正确（已在内存可跳过）
# ============================================
DefaultAssay(oligo_clean) <- "RNA"

oligo_levels <- paste0("Oligo_", sort(as.numeric(
  gsub("Oligo_", "", unique(as.character(oligo_clean$subtype))))))
oligo_clean$subtype <- factor(oligo_clean$subtype, levels = oligo_levels)
Idents(oligo_clean) <- oligo_clean$subtype

# ============================================
# 1. ★ 重新计算 Marker
# ============================================
oligo_clean_markers <- FindAllMarkers(oligo_clean,
                                      only.pos = TRUE,
                                      min.pct = 0.25,
                                      logfc.threshold = 0.25,
                                      test.use = "wilcox")

write.csv(oligo_clean_markers,
          paste0(output_path, "oligoClean_AllMarkers.csv"), row.names = FALSE)
cat("Marker 重算完成，共", nrow(oligo_clean_markers), "行\n")

# ============================================
# 2. Top20 提取 + CSV
# ============================================
oligo_clean_top20 <- oligo_clean_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(oligo_clean_top20,
          paste0(output_path, "oligoClean_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers 已保存\n")

# ============================================
# 3. Top5 DotPlot
# ============================================
oligo_clean_top5 <- oligo_clean_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(oligo_clean_top5$gene)

p_dot5 <- DotPlot(oligo_clean, features = gene_order5,
                  group.by = "subtype",
                  dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned Oligo subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot5)
save_fig(p_dot5, paste0(output_path, "oligoClean_Fig5_Top5_DotPlot"), 9, 5)

write.csv(oligo_clean_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "oligoClean_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 点图已保存\n")

# ============================================
# 4. Top10 DotPlot
# ============================================
oligo_clean_top10 <- oligo_clean_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(oligo_clean_top10$gene)

p_dot10 <- DotPlot(oligo_clean, features = gene_order10,
                   group.by = "subtype",
                   dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned Oligo subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot10)
save_fig(p_dot10, paste0(output_path, "oligoClean_Fig6_Top10_DotPlot"), 9, 6)

write.csv(oligo_clean_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "oligoClean_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 点图已保存\n")

# ============================================
# 5. Top20 DotPlot
# ============================================
gene_order20 <- unique(oligo_clean_top20$gene)

p_dot20 <- DotPlot(oligo_clean, features = gene_order20,
                   group.by = "subtype",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned Oligo subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_dot20)
save_fig(p_dot20, paste0(output_path, "oligoClean_Fig7_Top20_DotPlot"), 10, 9)

write.csv(oligo_clean_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "oligoClean_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 点图已保存\n")

# ============================================
# 6. 最后保存清洗后的对象
# ============================================
saveRDS(oligo_clean,
        paste0(output_path, "oligoClean_seurat_harmony_res",
               gsub("\\.", "", final_res_clean), ".rds"))
cat("清洗后的 oligo 对象已保存\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================

cat("===== OPC 重新聚类 =====\n")

opc_cells <- colnames(all_seurat)[all_seurat$cell_type == "OPC"]
opc_seurat <- subset(all_seurat, cells = opc_cells)
cat("OPC 细胞数:", ncol(opc_seurat), "\n")
print(table(opc_seurat$donor))

# ============================================
# 1. RNA 标准化 + PCA
# ============================================
DefaultAssay(opc_seurat) <- "RNA"
opc_seurat <- NormalizeData(opc_seurat, verbose = FALSE)
opc_seurat <- FindVariableFeatures(opc_seurat, selection.method = "vst",
                                   nfeatures = 2000, verbose = FALSE)
opc_seurat <- ScaleData(opc_seurat, verbose = FALSE)
opc_seurat <- RunPCA(opc_seurat, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony（donor 校正）
# ============================================
opc_seurat <- RunHarmony(opc_seurat,
                         group.by.vars = "donor",
                         reduction.use = "pca",
                         dims.use = 1:30,
                         verbose = FALSE)

# ============================================
# 3. 邻接图 + 多分辨率聚类 + clustree
# ============================================
opc_seurat <- FindNeighbors(opc_seurat, reduction = "harmony",
                            dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.05, 0.1, 0.2, 0.3, 0.5, 0.8)
for (res in resolutions_to_try) {
  opc_seurat <- FindClusters(opc_seurat, resolution = res,
                             future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(opc_seurat, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "opc_clustree"), 12, 9)

cat("\n各分辨率聚类情况:\n")
for (res in resolutions_to_try) {
  tab <- table(opc_seurat@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.1f: %d 群, 最小群 %d 细胞\n",
              res, length(tab), min(tab)))
}

In [ ]:
# ============================================
# 4. 选定分辨率 → 正式聚类 + 命名 + UMAP
# ============================================
final_res <- 0.05   # ← 看 clustree 后修改（OPC 通常 0.05~0.2）

opc_seurat <- SetIdent(opc_seurat, value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(opc_seurat)))
cl_new  <- as.character(Idents(opc_seurat))
cl_map  <- setNames(paste0("OPC_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

opc_seurat$subtype <- factor(unname(cl_map[cl_new]),
                             levels = paste0("OPC_", 0:(n_clust - 1)))
opc_seurat <- SetIdent(opc_seurat, value = "subtype")

cat("\nOPC 亚群分布 (res =", final_res, "):\n")
print(table(opc_seurat$subtype))

opc_seurat <- RunUMAP(opc_seurat, reduction = "harmony", dims = 1:30,
                      verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(opc_seurat, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("OPC Subtypes (res = ", final_res, ")"))
print(p1)
save_fig(p1, paste0(output_path, "opc_Fig1_UMAP_subtypes"), 7, 6)

p1b <- DimPlot(opc_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "opc_Fig2_UMAP_donor"), 7, 6)

p1c <- DimPlot(opc_seurat, group.by = "sample_label", raster = FALSE) + coord_fixed()
print(p1c)
save_fig(p1c, paste0(output_path, "opc_Fig2_UMAP_sample"), 7, 6)

# ---------- 经典 marker 小提琴 ----------
p2 <- VlnPlot(opc_seurat, features = c("PDGFRA", "CSPG4", "SOX10",
                                       "PTPRZ1", "OLIG1", "OLIG2"),
              pt.size = 0, ncol = 3)
print(p2)
save_fig(p2, paste0(output_path, "opc_Fig3_Vln_ClassicMarkers"), 10, 6)

# ============================================
# 5. ★ 身份核查 DotPlot（OPC 里容易藏少突/神经元/增殖细胞）
# ============================================
check_genes <- c(
  # OPC
  "PDGFRA", "CSPG4", "PTPRZ1", "BCAN", "OLIG1", "OLIG2", "SOX10",
  # 成熟少突（如果某亚群这些高 → 其实是分化中的 OL）
  "MBP", "PLP1", "MOG", "MAG", "OPALIN", "ST18", "ENPP6",
  # 神经元（既往经验：可能有神经元混入）
  "SNAP25", "SYT1", "TUBB3", "NEFL", "CHGB",
  # 星形
  "GFAP", "AQP4", "SLC1A2",
  # 小胶质
  "CSF1R", "AIF1",
  # 增殖
  "MKI67", "TOP2A", "CENPF"
)

p_check <- DotPlot(opc_seurat, features = check_genes,
                   group.by = "subtype",
                   dot.scale = 6, cols = c("lightgrey", "red")) +
  RotatedAxis() +
  ggtitle("OPC subtypes: identity check") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
        axis.text.y = element_text(size = 10, face = "bold"))
print(p_check)
save_fig(p_check, paste0(output_path, "opc_Fig4_DotPlot_IdentityCheck"), 14, 5)

In [ ]:
library(dplyr)

# ============================================
# 6. 找 Marker（OPC 按 subtype 分组）
# ============================================
opc_markers <- FindAllMarkers(opc_seurat,
                              only.pos = TRUE,
                              min.pct = 0.25,
                              logfc.threshold = 0.25,
                              test.use = "wilcox")

write.csv(opc_markers,
          paste0(output_path, "opc_AllMarkers.csv"), row.names = FALSE)
cat("Marker 计算完成，共", nrow(opc_markers), "行\n")

# ============================================
# 7. Top20 提取 + CSV
# ============================================
opc_top20 <- opc_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(opc_top20,
          paste0(output_path, "opc_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers 已保存\n")

# ============================================
# 8. Top5 DotPlot
# ============================================
opc_top5 <- opc_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(opc_top5$gene)

p_opc_dot5 <- DotPlot(opc_seurat, features = gene_order5,
                      group.by = "subtype",
                      dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("OPC subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_opc_dot5)
save_fig(p_opc_dot5, paste0(output_path, "opc_Fig5_Top5_DotPlot"), 9, 5)

write.csv(opc_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "opc_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 点图已保存\n")

# ============================================
# 9. Top10 DotPlot
# ============================================
opc_top10 <- opc_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(opc_top10$gene)

p_opc_dot10 <- DotPlot(opc_seurat, features = gene_order10,
                       group.by = "subtype",
                       dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("OPC subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_opc_dot10)
save_fig(p_opc_dot10, paste0(output_path, "opc_Fig6_Top10_DotPlot"), 9, 6)

write.csv(opc_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "opc_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 点图已保存\n")

# ============================================
# 10. Top20 DotPlot
# ============================================
gene_order20 <- unique(opc_top20$gene)

p_opc_dot20 <- DotPlot(opc_seurat, features = gene_order20,
                       group.by = "subtype",
                       dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("OPC subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_opc_dot20)
save_fig(p_opc_dot20, paste0(output_path, "opc_Fig7_Top20_DotPlot"), 10, 9)

write.csv(opc_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "opc_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 点图已保存\n")



In [ ]:
library(Seurat)
library(ggplot2)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================
# 0. 对象状态（已在内存可跳过）
# ============================================
DefaultAssay(opc_seurat) <- "RNA"

opc_levels <- paste0("OPC_", sort(as.numeric(
  gsub("OPC_", "", unique(as.character(opc_seurat$subtype))))))
opc_seurat$subtype <- factor(opc_seurat$subtype, levels = opc_levels)
Idents(opc_seurat) <- opc_seurat$subtype

cat("各亚群细胞数:\n")
print(table(opc_seurat$subtype))

# ============================================
# 图 1：全套身份核查（一图看清所有亚群）
# ============================================
check_genes <- c(
  # OPC 静息（OPC_0 应亮）
  "PDGFRA", "CSPG4", "PTPRZ1", "SOX10", "CSPG5", "GPM6A", "EGFR",
  # 分化中（OPC_1 应亮）
  "GPR17", "MYRF", "ENPP6", "BCAS1", "PLP1", "MBP", "UGT8",
  # 增殖（OPC_4 应亮）
  "MKI67", "TOP2A", "CCNE2", "E2F1",
  # 小胶质（OPC_3 应亮，其余应全暗）
  "CSF1R", "AIF1", "TYROBP", "C1QA", "C1QB", "INPP5D", "ITGAM", "CX3CR1", "APBB1IP",
  # 神经元（OPC_2 重点排查）
  "SNAP25", "SYT1", "TUBB3", "NEFL", "CHGB", "SCG5",
  # 星形/内皮（排除其他污染）
  "GFAP", "AQP4", "SLC1A2", "CLDN5"
)

p_check <- DotPlot(opc_seurat, features = check_genes,
                   group.by = "subtype",
                   dot.scale = 6, cols = c("lightgrey", "red")) +
  RotatedAxis() +
  ggtitle("OPC subtypes: full identity check") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
        axis.text.y = element_text(size = 10, face = "bold"))

print(p_check)
save_fig(p_check, paste0(output_path, "opc_Fig5_DotPlot_IdentityConfirm"), 16, 5.5)

# ============================================
# 图 2：OPC_2 专项放大（它只有 SEMA4C/SCG5 两个信号，单独看）
# ============================================
opc2_genes <- c("SEMA4C", "SCG5", "MXD4", "FADS2", "CALR",
                # 对照：神经元 vs OPC
                "SNAP25", "SYT1", "NEFL", "PDGFRA", "CSPG4", "SOX10", "GPM6A")

p_opc2 <- DotPlot(opc_seurat, features = opc2_genes,
                  group.by = "subtype",
                  dot.scale = 7, cols = c("lightgrey", "red")) +
  RotatedAxis() +
  ggtitle("OPC_2 zoom-in: neuron or OPC?") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
        axis.text.y = element_text(size = 10, face = "bold"))

print(p_opc2)
save_fig(p_opc2, paste0(output_path, "opc_Fig6_DotPlot_OPC2_ZoomIn"), 11, 5)

# ============================================
# 图 3：小胶质专项（确认 OPC_3 身份，同时看有没有漏网的小胶质混在 0/1/4）
# ============================================
micro_genes <- c("CSF1R", "AIF1", "C1QA", "C1QB", "C1QC", "TYROBP",
                 "INPP5D", "ITGAM", "CX3CR1", "APBB1IP", "MS4A7", "CD74")

p_micro <- DotPlot(opc_seurat, features = micro_genes,
                   group.by = "subtype",
                   dot.scale = 7, cols = c("lightgrey", "red")) +
  RotatedAxis() +
  ggtitle("Microglia markers across OPC subtypes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
        axis.text.y = element_text(size = 10, face = "bold"))

print(p_micro)
save_fig(p_micro, paste0(output_path, "opc_Fig7_DotPlot_MicrogliaConfirm"), 12, 5)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================
# 0. 剔除 OPC_3（小胶质），只留真少突谱系
# ============================================
opc_removed <- subset(opc_seurat, subset = subtype == "OPC_3")
cat("剔除小胶质 OPC_3:", ncol(opc_removed), "个\n")
# 如需留底：saveRDS(opc_removed, paste0(output_path, "opc_removed_3_microglia.rds"))

opc_clean <- subset(opc_seurat,
                    subset = subtype %in% c("OPC_0", "OPC_1", "OPC_2", "OPC_4"))
cat("清洗后 OPC 细胞数:", ncol(opc_clean), "\n")
print(table(opc_clean$subtype))

# ============================================
# 1. 重新标准化 + PCA（清洗后必须重跑）
# ============================================
DefaultAssay(opc_clean) <- "RNA"

opc_clean <- NormalizeData(opc_clean, verbose = FALSE)
opc_clean <- FindVariableFeatures(opc_clean, selection.method = "vst",
                                  nfeatures = 2000, verbose = FALSE)
opc_clean <- ScaleData(opc_clean, verbose = FALSE)
opc_clean <- RunPCA(opc_clean, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony（donor 校正）
# ============================================
opc_clean <- RunHarmony(opc_clean,
                        group.by.vars = "donor",
                        reduction.use = "pca",
                        dims.use = 1:30,
                        verbose = FALSE)

# ============================================
# 3. 邻接图 + 多分辨率聚类 + clustree
# ============================================
opc_clean <- FindNeighbors(opc_clean, reduction = "harmony",
                           dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.05, 0.1, 0.2, 0.3, 0.5)
for (res in resolutions_to_try) {
  opc_clean <- FindClusters(opc_clean, resolution = res,
                            future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(opc_clean, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "opcClean_clustree"), 12, 9)

cat("\n各分辨率聚类情况:\n")
for (res in resolutions_to_try) {
  tab <- table(opc_clean@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.2f: %d 群, 最小群 %d 细胞\n",
              res, length(tab), min(tab)))
}




In [ ]:
# ============================================
# 4. 选定分辨率 → 正式聚类 + 命名 + UMAP
# ============================================
final_res_clean <- 0.05   # ← 看 clustree 后修改

opc_clean <- SetIdent(opc_clean, value = paste0("RNA_snn_res.", final_res_clean))

n_clust <- length(unique(Idents(opc_clean)))
cl_new  <- as.character(Idents(opc_clean))
cl_map  <- setNames(paste0("OPC_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

opc_clean$subtype <- factor(unname(cl_map[cl_new]),
                            levels = paste0("OPC_", 0:(n_clust - 1)))
opc_clean <- SetIdent(opc_clean, value = "subtype")

cat("\n清洗后亚群分布 (res =", final_res_clean, "):\n")
print(table(opc_clean$subtype))

opc_clean <- RunUMAP(opc_clean, reduction = "harmony", dims = 1:30,
                     verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(opc_clean, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("OPC (cleaned) res = ", final_res_clean))
print(p1)
save_fig(p1, paste0(output_path, "opcClean_Fig1_UMAP"), 7, 6)

p1b <- DimPlot(opc_clean, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "opcClean_Fig2_UMAP_donor"), 7, 6)

# ---------- 经典 marker 小提琴 ----------
p2 <- VlnPlot(opc_clean, features = c("PDGFRA", "CSPG4", "PTPRZ1", "SOX10",
                                      "GPR17", "MKI67"),
              pt.size = 0, ncol = 3)
print(p2)
save_fig(p2, paste0(output_path, "opcClean_Fig3_Vln_ClassicMarkers"), 10, 6)


In [ ]:
library(dplyr)

# ============================================
# 6. 重新计算 Marker（按清洗后 subtype）
# ============================================
opc_clean_markers <- FindAllMarkers(opc_clean,
                                    only.pos = TRUE,
                                    min.pct = 0.25,
                                    logfc.threshold = 0.25,
                                    test.use = "wilcox")

write.csv(opc_clean_markers,
          paste0(output_path, "opcClean_AllMarkers.csv"), row.names = FALSE)
cat("Marker 重算完成，共", nrow(opc_clean_markers), "行\n")

# ============================================
# 7. Top20 提取 + CSV
# ============================================
opc_clean_top20 <- opc_clean_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(opc_clean_top20,
          paste0(output_path, "opcClean_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers 已保存\n")

# ============================================
# 8. Top5 DotPlot
# ============================================
opc_clean_top5 <- opc_clean_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(opc_clean_top5$gene)

p_dot5 <- DotPlot(opc_clean, features = gene_order5,
                  group.by = "subtype",
                  dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned OPC subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot5)
save_fig(p_dot5, paste0(output_path, "opcClean_Fig5_Top5_DotPlot"), 9, 6)

write.csv(opc_clean_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "opcClean_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 点图已保存\n")

# ============================================
# 9. Top10 DotPlot
# ============================================
opc_clean_top10 <- opc_clean_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(opc_clean_top10$gene)

p_dot10 <- DotPlot(opc_clean, features = gene_order10,
                   group.by = "subtype",
                   dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned OPC subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot10)
save_fig(p_dot10, paste0(output_path, "opcClean_Fig6_Top10_DotPlot"), 10, 8)

write.csv(opc_clean_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "opcClean_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 点图已保存\n")

# ============================================
# 10. Top20 DotPlot
# ============================================
gene_order20 <- unique(opc_clean_top20$gene)

p_dot20 <- DotPlot(opc_clean, features = gene_order20,
                   group.by = "subtype",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned OPC subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_dot20)
save_fig(p_dot20, paste0(output_path, "opcClean_Fig7_Top20_DotPlot"), 11, 14)

write.csv(opc_clean_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "opcClean_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 点图已保存\n")



In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================
# 0. 提取 Microglia（统一从 cell_type 提，和前面大类一致）
# ============================================
micro_cells <- colnames(classFiltSeurat_filt)[
  classFiltSeurat_filt$cell_type == "Microglia"
]

micro_seurat <- subset(classFiltSeurat_filt, cells = micro_cells)
cat("Microglia 细胞数:", ncol(micro_seurat), "\n")
print(table(micro_seurat$donor))

# ============================================
# 1. RNA 标准化 + PCA
# ============================================
DefaultAssay(micro_seurat) <- "RNA"
micro_seurat <- NormalizeData(micro_seurat, verbose = FALSE)
micro_seurat <- FindVariableFeatures(micro_seurat, selection.method = "vst",
                                     nfeatures = 2000, verbose = FALSE)
micro_seurat <- ScaleData(micro_seurat, verbose = FALSE)
micro_seurat <- RunPCA(micro_seurat, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony（donor 校正）
# ============================================
micro_seurat <- RunHarmony(micro_seurat,
                           group.by.vars = "donor",
                           reduction.use = "pca",
                           dims.use = 1:30,
                           verbose = FALSE)

# ============================================
# 3. 邻接图 + 多分辨率聚类 + clustree
# ============================================
micro_seurat <- FindNeighbors(micro_seurat, reduction = "harmony",
                              dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.05, 0.1, 0.2, 0.3, 0.5)
for (res in resolutions_to_try) {
  micro_seurat <- FindClusters(micro_seurat, resolution = res,
                               future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(micro_seurat, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "micro_clustree"), 12, 9)

cat("\n各分辨率聚类情况:\n")
for (res in resolutions_to_try) {
  tab <- table(micro_seurat@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.2f: %d 群, 最小群 %d 细胞\n",
              res, length(tab), min(tab)))
}


In [ ]:
# ============================================
# 4. 选定分辨率 → 正式聚类 + 命名 + UMAP
# ============================================
final_res <- 0.05   # ← 看 clustree 后修改

micro_seurat <- SetIdent(micro_seurat, value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(micro_seurat)))
cl_new  <- as.character(Idents(micro_seurat))
cl_map  <- setNames(paste0("Microglia_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

micro_seurat$subtype <- factor(unname(cl_map[cl_new]),
                               levels = paste0("Microglia_", 0:(n_clust - 1)))
micro_seurat <- SetIdent(micro_seurat, value = "subtype")

cat("\nMicroglia 亚群分布 (res =", final_res, "):\n")
print(table(micro_seurat$subtype))

micro_seurat <- RunUMAP(micro_seurat, reduction = "harmony", dims = 1:30,
                        verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(micro_seurat, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Microglia Subtypes (res = ", final_res, ")"))
print(p1)
save_fig(p1, paste0(output_path, "micro_Fig1_UMAP_subtypes"), 7, 6)

p1b <- DimPlot(micro_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "micro_Fig2_UMAP_donor"), 7, 6)

p1c <- DimPlot(micro_seurat, group.by = "sample_label", raster = FALSE) + coord_fixed()
print(p1c)
save_fig(p1c, paste0(output_path, "micro_Fig2_UMAP_sample"), 7, 6)


In [ ]:
library(dplyr)

# ============================================
# 7. 重新计算 Marker（按 subtype 分组）
# ============================================
micro_markers <- FindAllMarkers(micro_seurat,
                                only.pos = TRUE,
                                min.pct = 0.25,
                                logfc.threshold = 0.25,
                                test.use = "wilcox")

write.csv(micro_markers,
          paste0(output_path, "micro_AllMarkers.csv"), row.names = FALSE)
cat("Marker 重算完成，共", nrow(micro_markers), "行\n")

# ============================================
# 8. Top20 提取 + CSV
# ============================================
micro_top20 <- micro_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(micro_top20,
          paste0(output_path, "micro_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers 已保存\n")

# ============================================
# 9. Top5 DotPlot
# ============================================
micro_top5 <- micro_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(micro_top5$gene)

p_dot5 <- DotPlot(micro_seurat, features = gene_order5,
                  group.by = "subtype",
                  dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Microglia subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot5)
save_fig(p_dot5, paste0(output_path, "micro_Fig5_Top5_DotPlot"), 9, 6)

write.csv(micro_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "micro_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 点图已保存\n")

# ============================================
# 10. Top10 DotPlot
# ============================================
micro_top10 <- micro_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(micro_top10$gene)

p_dot10 <- DotPlot(micro_seurat, features = gene_order10,
                   group.by = "subtype",
                   dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Microglia subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot10)
save_fig(p_dot10, paste0(output_path, "micro_Fig6_Top10_DotPlot"), 10, 8)

write.csv(micro_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "micro_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 点图已保存\n")

# ============================================
# 11. Top20 DotPlot
# ============================================
gene_order20 <- unique(micro_top20$gene)

p_dot20 <- DotPlot(micro_seurat, features = gene_order20,
                   group.by = "subtype",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Microglia subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_dot20)
save_fig(p_dot20, paste0(output_path, "micro_Fig7_Top20_DotPlot"), 11, 12)

write.csv(micro_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "micro_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 点图已保存\n")

In [ ]:
library(Seurat)
library(ggplot2)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ---------- 1. 剔除 Microglia_1 ----------
cat("删除前亚群分布:\n")
print(table(micro_seurat$subtype))

micro_removed <- subset(micro_seurat, subset = subtype == "Microglia_1")
saveRDS(micro_removed, paste0(output_path, "micro_removed_1.rds"))
cat("已剔除 Microglia_1:", ncol(micro_removed), "个细胞\n")

micro_clean <- subset(micro_seurat, subset = subtype != "Microglia_1")

# 亚群名重排（保持 0..n 连续，避免中间空号）
micro_clean$subtype <- factor(as.character(micro_clean$subtype))
print(table(micro_clean$subtype))

# ---------- 2. 用现有 harmony 空间重算 UMAP（不重聚类）----------
micro_clean <- RunUMAP(micro_clean, reduction = "harmony", dims = 1:30,
                       verbose = FALSE)

# ---------- 3. UMAP ----------
p_umap <- DimPlot(micro_clean, group.by = "subtype", label = TRUE, repel = TRUE,
                  raster = FALSE) +
  coord_fixed() +
  ggtitle("Microglia subtypes (Microglia_1 removed)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_umap)
save_fig(p_umap, paste0(output_path, "microClean_Fig1_UMAP"), 7, 6)

p_donor <- DimPlot(micro_clean, group.by = "donor", raster = FALSE) + coord_fixed()
print(p_donor)
save_fig(p_donor, paste0(output_path, "microClean_Fig2_UMAP_donor"), 7, 6)

cat("删除后细胞数:", ncol(micro_clean), "\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================
# 0. 提取 Endothelial（从原始 cell_type 提）
# ============================================
endo_cells <- colnames(classFiltSeurat_filt)[
  classFiltSeurat_filt$cell_type == "Endothelial"
]

endo_seurat <- subset(classFiltSeurat_filt, cells = endo_cells)
cat("Endothelial 细胞数:", ncol(endo_seurat), "\n")
print(table(endo_seurat$donor))

# ============================================
# 1. RNA 标准化 + PCA
# ============================================
DefaultAssay(endo_seurat) <- "RNA"
endo_seurat <- NormalizeData(endo_seurat, verbose = FALSE)
endo_seurat <- FindVariableFeatures(endo_seurat, selection.method = "vst",
                                    nfeatures = 2000, verbose = FALSE)
endo_seurat <- ScaleData(endo_seurat, verbose = FALSE)
endo_seurat <- RunPCA(endo_seurat, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony（donor 校正）
# ============================================
endo_seurat <- RunHarmony(endo_seurat,
                          group.by.vars = "donor",
                          reduction.use = "pca",
                          dims.use = 1:30,
                          verbose = FALSE)

# ============================================
# 3. 邻接图 + 多分辨率聚类 + clustree
# ============================================
endo_seurat <- FindNeighbors(endo_seurat, reduction = "harmony",
                             dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.02, 0.05, 0.1, 0.2, 0.3, 0.5)
for (res in resolutions_to_try) {
  endo_seurat <- FindClusters(endo_seurat, resolution = res,
                              future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(endo_seurat, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "endo_clustree"), 12, 9)

cat("\n各分辨率聚类情况:\n")
for (res in resolutions_to_try) {
  tab <- table(endo_seurat@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.2f: %d 群, 最小群 %d 细胞\n",
              res, length(tab), min(tab)))
}


In [ ]:
# ============================================
# 4. 选定分辨率 → 正式聚类 + 命名 + UMAP
# ============================================
final_res <- 0.02   # ← 看 clustree 后修改（内皮细胞数少，低分辨率起步）

endo_seurat <- SetIdent(endo_seurat, value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(endo_seurat)))
cl_new  <- as.character(Idents(endo_seurat))
cl_map  <- setNames(paste0("Endothelial_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

endo_seurat$subtype <- factor(unname(cl_map[cl_new]),
                              levels = paste0("Endothelial_", 0:(n_clust - 1)))
endo_seurat <- SetIdent(endo_seurat, value = "subtype")

cat("\nEndothelial 亚群分布 (res =", final_res, "):\n")
print(table(endo_seurat$subtype))

endo_seurat <- RunUMAP(endo_seurat, reduction = "harmony", dims = 1:30,
                       verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(endo_seurat, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Endothelial Subtypes (res = ", final_res, ")"))
print(p1)
save_fig(p1, paste0(output_path, "endo_Fig1_UMAP_subtypes"), 7, 6)

p1b <- DimPlot(endo_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "endo_Fig2_UMAP_donor"), 7, 6)

p1c <- DimPlot(endo_seurat, group.by = "sample_label", raster = FALSE) + coord_fixed()
print(p1c)
save_fig(p1c, paste0(output_path, "endo_Fig2_UMAP_sample"), 7, 6)

# ---------- 经典 marker 小提琴 ----------
p2 <- VlnPlot(endo_seurat, features = c("FLT1", "CLDN5", "PECAM1",
                                        "VWF", "KDR", "ESAM"),
              pt.size = 0, ncol = 3)
print(p2)
save_fig(p2, paste0(output_path, "endo_Fig3_Vln_ClassicMarkers"), 10, 6)

In [ ]:
library(dplyr)

# ============================================
# 7. 重新计算 Marker（按 subtype 分组）
# ============================================
endo_markers <- FindAllMarkers(endo_seurat,
                               only.pos = TRUE,
                               min.pct = 0.25,
                               logfc.threshold = 0.25,
                               test.use = "wilcox")

write.csv(endo_markers,
          paste0(output_path, "endo_AllMarkers.csv"), row.names = FALSE)
cat("Marker 重算完成，共", nrow(endo_markers), "行\n")

# ============================================
# 8. Top20 提取 + CSV
# ============================================
endo_top20 <- endo_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(endo_top20,
          paste0(output_path, "endo_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers 已保存\n")

# ============================================
# 9. Top5 DotPlot
# ============================================
endo_top5 <- endo_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(endo_top5$gene)

p_dot5 <- DotPlot(endo_seurat, features = gene_order5,
                  group.by = "subtype",
                  dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Endothelial subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot5)
save_fig(p_dot5, paste0(output_path, "endo_Fig5_Top5_DotPlot"), 9, 5)

write.csv(endo_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "endo_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 点图已保存\n")

# ============================================
# 10. Top10 DotPlot
# ============================================
endo_top10 <- endo_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(endo_top10$gene)

p_dot10 <- DotPlot(endo_seurat, features = gene_order10,
                   group.by = "subtype",
                   dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Endothelial subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot10)
save_fig(p_dot10, paste0(output_path, "endo_Fig6_Top10_DotPlot"), 9, 6)

write.csv(endo_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "endo_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 点图已保存\n")

# ============================================
# 11. Top20 DotPlot
# ============================================
gene_order20 <- unique(endo_top20$gene)

p_dot20 <- DotPlot(endo_seurat, features = gene_order20,
                   group.by = "subtype",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Endothelial subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_dot20)
save_fig(p_dot20, paste0(output_path, "endo_Fig7_Top20_DotPlot"), 10, 10)

write.csv(endo_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "endo_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 点图已保存\n")

# # ============================================
# # 12. 保存对象（带新亚群命名）
# # ============================================
# saveRDS(endo_seurat,
#         paste0(output_path, "endo_seurat_harmony_res",
#                gsub("\\.", "", final_res), ".rds"))
# cat("Endothelial 对象已保存\n")

In [ ]:
library(Seurat)
library(ggplot2)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ---------- 1. 剔除 Endothelial_1 ----------
cat("删除前亚群分布:\n")
print(table(endo_seurat$subtype))

endo_removed <- subset(endo_seurat, subset = subtype == "Endothelial_1")
saveRDS(endo_removed, paste0(output_path, "endo_removed_1.rds"))
cat("已剔除 Endothelial_1:", ncol(endo_removed), "个细胞\n")

endo_clean <- subset(endo_seurat, subset = subtype != "Endothelial_1")
endo_clean$subtype <- factor(as.character(endo_clean$subtype))
cat("删除后亚群分布:\n")
print(table(endo_clean$subtype))

# ---------- 2. 用现有 harmony 空间重算 UMAP ----------
endo_clean <- RunUMAP(endo_clean, reduction = "harmony", dims = 1:30,
                      verbose = FALSE)

# ---------- 3. UMAP ----------
p_umap <- DimPlot(endo_clean, group.by = "subtype", label = TRUE, repel = TRUE,
                  raster = FALSE) +
  coord_fixed() +
  ggtitle("Endothelial subtypes (Endothelial_1 removed)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_umap)
save_fig(p_umap, paste0(output_path, "endoClean_Fig1_UMAP"), 7, 6)

p_donor <- DimPlot(endo_clean, group.by = "donor", raster = FALSE) + coord_fixed()
print(p_donor)
save_fig(p_donor, paste0(output_path, "endoClean_Fig2_UMAP_donor"), 7, 6)

cat("删除后细胞数:", ncol(endo_clean), "\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================
# 0. 提取 Ependymal（从原始 cell_type 提）
# ============================================
epen_cells <- colnames(classFiltSeurat_filt)[
  classFiltSeurat_filt$cell_type == "Ependymal"
]

epen_seurat <- subset(classFiltSeurat_filt, cells = epen_cells)
cat("Ependymal 细胞数:", ncol(epen_seurat), "\n")
print(table(epen_seurat$donor))

# ============================================
# 1. RNA 标准化 + PCA
# ============================================
DefaultAssay(epen_seurat) <- "RNA"
epen_seurat <- NormalizeData(epen_seurat, verbose = FALSE)
epen_seurat <- FindVariableFeatures(epen_seurat, selection.method = "vst",
                                    nfeatures = 2000, verbose = FALSE)
epen_seurat <- ScaleData(epen_seurat, verbose = FALSE)
epen_seurat <- RunPCA(epen_seurat, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony（donor 校正）
# ============================================
epen_seurat <- RunHarmony(epen_seurat,
                          group.by.vars = "donor",
                          reduction.use = "pca",
                          dims.use = 1:30,
                          verbose = FALSE)

# ============================================
# 3. 邻接图 + 多分辨率聚类 + clustree
# ============================================
epen_seurat <- FindNeighbors(epen_seurat, reduction = "harmony",
                             dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.02, 0.05, 0.1, 0.2, 0.3, 0.5)
for (res in resolutions_to_try) {
  epen_seurat <- FindClusters(epen_seurat, resolution = res,
                              future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(epen_seurat, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "epen_clustree"), 12, 9)

cat("\n各分辨率聚类情况:\n")
for (res in resolutions_to_try) {
  tab <- table(epen_seurat@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.2f: %d 群, 最小群 %d 细胞\n",
              res, length(tab), min(tab)))
}

In [ ]:
# ============================================
# 4. 选定分辨率 → 正式聚类 + 命名 + UMAP
# ============================================
final_res <- 0.02   # ← 看 clustree 后修改

epen_seurat <- SetIdent(epen_seurat, value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(epen_seurat)))
cl_new  <- as.character(Idents(epen_seurat))
cl_map  <- setNames(paste0("Ependymal_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

epen_seurat$subtype <- factor(unname(cl_map[cl_new]),
                              levels = paste0("Ependymal_", 0:(n_clust - 1)))
epen_seurat <- SetIdent(epen_seurat, value = "subtype")

cat("\nEpendymal 亚群分布 (res =", final_res, "):\n")
print(table(epen_seurat$subtype))

epen_seurat <- RunUMAP(epen_seurat, reduction = "harmony", dims = 1:30,
                       verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(epen_seurat, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Ependymal Subtypes (res = ", final_res, ")"))
print(p1)
save_fig(p1, paste0(output_path, "epen_Fig1_UMAP_subtypes"), 7, 6)

p1b <- DimPlot(epen_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "epen_Fig2_UMAP_donor"), 7, 6)

p1c <- DimPlot(epen_seurat, group.by = "sample_label", raster = FALSE) + coord_fixed()
print(p1c)
save_fig(p1c, paste0(output_path, "epen_Fig2_UMAP_sample"), 7, 6)

In [ ]:
library(dplyr)

# ============================================
# 7. 重新计算 Marker（按 subtype 分组）
# ============================================
epen_markers <- FindAllMarkers(epen_seurat,
                               only.pos = TRUE,
                               min.pct = 0.25,
                               logfc.threshold = 0.25,
                               test.use = "wilcox")

write.csv(epen_markers,
          paste0(output_path, "epen_AllMarkers.csv"), row.names = FALSE)
cat("Marker 重算完成，共", nrow(epen_markers), "行\n")

# ============================================
# 8. Top20 提取 + CSV
# ============================================
epen_top20 <- epen_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(epen_top20,
          paste0(output_path, "epen_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers 已保存\n")

# ============================================
# 9. Top5 DotPlot
# ============================================
epen_top5 <- epen_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(epen_top5$gene)

p_dot5 <- DotPlot(epen_seurat, features = gene_order5,
                  group.by = "subtype",
                  dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Ependymal subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot5)
save_fig(p_dot5, paste0(output_path, "epen_Fig5_Top5_DotPlot"), 9, 5)

write.csv(epen_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "epen_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 点图已保存\n")

# ============================================
# 10. Top10 DotPlot
# ============================================
epen_top10 <- epen_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(epen_top10$gene)

p_dot10 <- DotPlot(epen_seurat, features = gene_order10,
                   group.by = "subtype",
                   dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Ependymal subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot10)
save_fig(p_dot10, paste0(output_path, "epen_Fig6_Top10_DotPlot"), 9, 6)

write.csv(epen_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "epen_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 点图已保存\n")

# ============================================
# 11. Top20 DotPlot
# ============================================
gene_order20 <- unique(epen_top20$gene)

p_dot20 <- DotPlot(epen_seurat, features = gene_order20,
                   group.by = "subtype",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Ependymal subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_dot20)
save_fig(p_dot20, paste0(output_path, "epen_Fig7_Top20_DotPlot"), 10, 10)

write.csv(epen_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "epen_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 点图已保存\n")

# # ============================================
# # 12. 保存对象（带新亚群命名）
# # ============================================
# saveRDS(epen_seurat,
#         paste0(output_path, "epen_seurat_harmony_res",
#                gsub("\\.", "", final_res), ".rds"))
# cat("Ependymal 对象已保存\n")

In [ ]:
library(Seurat)
library(ggplot2)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ---------- 1. 剔除 Ependymal_1 ----------
cat("删除前亚群分布:\n")
print(table(epen_seurat$subtype))

epen_removed <- subset(epen_seurat, subset = subtype == "Ependymal_1")
saveRDS(epen_removed, paste0(output_path, "epen_removed_1.rds"))
cat("已剔除 Ependymal_1:", ncol(epen_removed), "个细胞\n")

epen_clean <- subset(epen_seurat, subset = subtype != "Ependymal_1")
epen_clean$subtype <- factor(as.character(epen_clean$subtype))
cat("删除后亚群分布:\n")
print(table(epen_clean$subtype))

# ---------- 2. 用现有 harmony 空间重算 UMAP ----------
epen_clean <- RunUMAP(epen_clean, reduction = "harmony", dims = 1:30,
                      verbose = FALSE)

# ---------- 3. UMAP ----------
p_umap <- DimPlot(epen_clean, group.by = "subtype", label = TRUE, repel = TRUE,
                  raster = FALSE) +
  coord_fixed() +
  ggtitle("Ependymal subtypes (Ependymal_1 removed)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_umap)
save_fig(p_umap, paste0(output_path, "epenClean_Fig1_UMAP"), 7, 6)

p_donor <- DimPlot(epen_clean, group.by = "donor", raster = FALSE) + coord_fixed()
print(p_donor)
save_fig(p_donor, paste0(output_path, "epenClean_Fig2_UMAP_donor"), 7, 6)

cat("删除后细胞数:", ncol(epen_clean), "\n")

In [ ]:
library(Seurat)

# ============================================
# 0. 确保 classFiltSeurat_filt 在内存
# ============================================
# 已在内存则跳过

# ============================================
# 1. 汇总表：只用清洗后的对象
# ============================================
subtype_sources <- list(
  ht5ht  = list(obj = ht_5ht,       col = "ht_subtype"),  # 5-HT 亚群（你的干净版）
  astro  = list(obj = astro_clean,  col = "subtype"),      # Astrocytes_0/1（2/3 已删）
  oligo  = list(obj = oligo_clean,  col = "subtype"),      # Oligo_0/1（2 已删）
  opc    = list(obj = opc_clean,    col = "subtype"),      # OPC_*（3 已删，可能还有 2）
  micro  = list(obj = micro_clean,  col = "subtype"),      # Microglia_*（1 已删）
  endo   = list(obj = endo_clean,   col = "subtype"),      # Endothelial_*（1 已删）
  epen   = list(obj = epen_clean,   col = "subtype")       # Ependymal_*（1 已删）
)

# ============================================
# 2. 建统一列
# ============================================
classFiltSeurat_filt$cell_type_fine <- as.character(classFiltSeurat_filt$cell_type)

# 记录哪些大类被重聚类过（这些大类里"没被写回的细胞"就是被剔除的）
reclustered_classes <- c("Astrocytes", "Oligo", "OPC", "Microglia",
                         "Endothelial", "Ependymal")

for (nm in names(subtype_sources)) {
  obj <- subtype_sources[[nm]]$obj
  col <- subtype_sources[[nm]]$col

  if (is.null(obj)) {
    cat("⚠️ 跳过（对象不存在，确认是否已跑完）:", nm, "\n")
    next
  }

  map_back <- setNames(as.character(obj@meta.data[[col]]), colnames(obj))
  hit <- names(map_back) %in% colnames(classFiltSeurat_filt)

  classFiltSeurat_filt$cell_type_fine[names(map_back)] <- map_back

  cat(sprintf("%-8s 写回 %d 个细胞（匹配成功 %d）\n",
              nm, length(map_back), sum(hit)))
}

# ============================================
# 3. 被剔除的细胞单独标记（不属于任何剩余亚群）
# =========================================#
in_reclustered <- classFiltSeurat_filt$cell_type %in% reclustered_classes
got_label <- !is.na(classFiltSeurat_filt$cell_type_fine) &
             classFiltSeurat_filt$cell_type_fine != classFiltSeurat_filt$cell_type &
             !is.na(match(classFiltSeurat_filt$cell_type_fine,
                          unlist(lapply(subtype_sources,
                                        function(x) if(!is.null(x$obj))
                                          as.character(x$obj@meta.data[[x$col]])))))

# 更直接的算法：在大类范围内，cell_type_fine 仍等于原大类名的 = 没被写回 = 被剔除的
removed_mask <- in_reclustered &
                (classFiltSeurat_filt$cell_type_fine == classFiltSeurat_filt$cell_type)

classFiltSeurat_filt$cell_type_fine[removed_mask] <- "Removed"

n_removed <- sum(removed_mask)
cat("\n被剔除的细胞（标为 Removed）:", n_removed, "个\n")
print(sort(table(classFiltSeurat_filt$cell_type[removed_mask]), decreasing = TRUE))

classFiltSeurat_filt$cell_type_fine <- factor(classFiltSeurat_filt$cell_type_fine)



In [ ]:
# ============================================
# 4. 出总表（修复版排序）
# ============================================
tb <- table(classFiltSeurat_filt$cell_type_fine)

sort_key <- function(x) {
  num <- rep(-1L, length(x))
  has_num <- grepl("_?[0-9]+$", x)
  if (any(has_num)) {
    m <- regmatches(x[has_num], regexpr("_?[0-9]+$", x[has_num]))
    num[has_num] <- suppressWarnings(as.numeric(gsub("_", "", m)))
  }
  data.frame(name = x,
             base = gsub("_?[0-9]+$", "", x),
             num  = num,
             stringsAsFactors = FALSE)
}

sk <- sort_key(names(tb))
sk <- sk[order(sk$base, sk$num), ]
tb_sorted <- tb[sk$name]

cat("\n===== 汇总后全体分布（", length(tb_sorted), "类）=====\n")
print(tb_sorted)

# ============================================
# 5. 核对
# ============================================
cat("\n--- 核对 ---\n")
cat("总细胞数:", ncol(classFiltSeurat_filt), "\n")
cat("Removed:", sum(tb_sorted[names(tb_sorted) == "Removed"]),
    "，其余合计:", sum(tb_sorted[names(tb_sorted) != "Removed"]), "\n")

is_neuron  <- grepl("^(5-HT|Glut|GABA|DA)_", names(tb_sorted))
is_removed <- names(tb_sorted) == "Removed"
cat("神经元亚群合计:", sum(tb_sorted[is_neuron]),
    "，非神经元亚群合计:", sum(tb_sorted[!is_neuron & !is_removed]), "\n")

# ============================================
# 6. 保存（确认后放开）
# ============================================
# saveRDS(classFiltSeurat_filt,
#         paste0(output_path, "classFiltSeurat_filt_allSubtypes.rds"))
# write.csv(data.frame(cell_type = names(tb_sorted), n = as.integer(tb_sorted)),
#           paste0(output_path, "cell_type_fine_distribution.csv"),
#           row.names = FALSE)

In [ ]:
# ============================================
# 6. 保存（确认后放开）
# ============================================
saveRDS(classFiltSeurat_filt,
        paste0(output_path, "classFiltSeurat_filt_allSubtypes.rds"))
write.csv(data.frame(cell_type = names(tb_sorted), n = as.integer(tb_sorted)),
          paste0(output_path, "cell_type_fine_distribution.csv"),
          row.names = FALSE)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================
# 0. 剔除 Removed，只留干净细胞
# ============================================
cat("整合前总细胞数:", ncol(classFiltSeurat_filt), "\n")

seurat_all <- subset(classFiltSeurat_filt,
                     subset = cell_type_fine != "Removed")
cat("剔除 Removed 后:", ncol(seurat_all), "\n")

# ============================================
# 1. 建大类列：神经元亚群 → "Neurons"，其余按前缀
# ============================================
ct <- as.character(seurat_all$cell_type_fine)

seurat_all$cell_class <- dplyr::case_when(
  grepl("^(5-HT|Glut|GABA|Glut-GABA|DA)_", ct) ~ "Neurons",
  startsWith(ct, "Astrocytes")  ~ "Astrocytes",
  startsWith(ct, "Oligo")       ~ "Oligo",
  startsWith(ct, "OPC")         ~ "OPC",
  startsWith(ct, "Microglia")   ~ "Microglia",
  startsWith(ct, "Endothelial") ~ "Endothelial",
  startsWith(ct, "Ependymal")   ~ "Ependymal",
  TRUE ~ "Other"
)
seurat_all$cell_class <- factor(seurat_all$cell_class,
                                levels = c("Neurons","Astrocytes","Oligo","OPC",
                                           "Microglia","Endothelial","Ependymal"))

seurat_all$cell_type_fine <- factor(seurat_all$cell_type_fine)

cat("\n大类分布:\n")
print(table(seurat_all$cell_class))

# ============================================
# 2. RNA 标准化 + PCA + Harmony
# ============================================
DefaultAssay(seurat_all) <- "RNA"

seurat_all <- NormalizeData(seurat_all, verbose = FALSE)
seurat_all <- FindVariableFeatures(seurat_all, selection.method = "vst",
                                   nfeatures = 3000, verbose = FALSE)
seurat_all <- ScaleData(seurat_all, verbose = FALSE)
seurat_all <- RunPCA(seurat_all, npcs = 50, verbose = FALSE)

seurat_all <- RunHarmony(seurat_all,
                         group.by.vars = "donor",
                         reduction.use = "pca",
                         dims.use = 1:30,
                         verbose = FALSE)

# ============================================
# 3. UMAP
# ============================================
seurat_all <- RunUMAP(seurat_all, reduction = "harmony", dims = 1:30,
                      verbose = FALSE)

# ============================================
# 4. 图 1：按 8 个大类着色
# ============================================
p_class <- DimPlot(seurat_all, group.by = "cell_class",
                   raster = FALSE, shuffle = TRUE) +
  coord_fixed() +
  ggtitle("All cells (cleaned): major classes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
        legend.text = element_text(size = 10))
print(p_class)
save_fig(p_class, paste0(output_path, "all_Fig1_UMAP_majorClasses"), 8, 7)

# 无标签版（更大更清晰）
p_class_nolabel <- DimPlot(seurat_all, group.by = "cell_class",
                           raster = TRUE, shuffle = TRUE) +
  coord_fixed() +
  NoLegend() +
  ggtitle("All cells: major classes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14))
save_fig(p_class_nolabel, paste0(output_path, "all_Fig1b_UMAP_majorClasses_noLegend"),
         7, 6)

# ============================================
# 5. 图 2：按全部 55 个小类着色
# ============================================
# 小类顺序：按之前排好的 0→n 数字序
fine_levels <- names(tb_sorted)[names(tb_sorted) != "Removed"]
seurat_all$cell_type_fine <- factor(seurat_all$cell_type_fine,
                                    levels = fine_levels)

p_fine <- DimPlot(seurat_all, group.by = "cell_type_fine",
                  label = TRUE, repel = TRUE, raster = TRUE,
                  shuffle = TRUE) +
  coord_fixed() +
  NoLegend() +
  ggtitle("All cells: all subtypes (55)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14))
print(p_fine)
save_fig(p_fine, paste0(output_path, "all_Fig2_UMAP_allSubtypes"), 12, 10)

# ---------- 附带：大类分面的小类图（投稿常用）----------
p_split <- DimPlot(seurat_all, group.by = "cell_type_fine",
                   split.by = "cell_class",
                   raster = TRUE, shuffle = TRUE) +
  coord_fixed() +
  NoLegend() +
  ggtitle("All subtypes split by major class")
save_fig(p_split, paste0(output_path, "all_Fig3_UMAP_subtypes_split"), 20, 5)

# ============================================
# 6. 按 donor / 样本检查批次
# ============================================
p_donor <- DimPlot(seurat_all, group.by = "donor", raster = TRUE, shuffle = TRUE) +
  coord_fixed()
save_fig(p_donor, paste0(output_path, "all_Fig4_UMAP_donor"), 8, 7)

# # ============================================
# # 7. 保存
# # ============================================
# saveRDS(seurat_all,
#         paste0(output_path, "seurat_all_harmony_integrated.rds"))
# cat("整合后的全体对象已保存\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("✅ 已保存:", file_base, "（PNG + PDF）\n")
}

# ============================================
# 0. 提取所有非神经元亚群
# ============================================
# 从整合后的对象提（已剔除 Removed）；如果没有 seurat_all，就从 classFiltSeurat_filt 提
non_neuron_levels <- c("Astrocytes_0","Astrocytes_1",
                       "Oligo_0","Oligo_1",
                       "OPC_0","OPC_1","OPC_2","OPC_3",
                       "Microglia_0",
                       "Endothelial_0",
                       "Ependymal_0")

nn_seurat <- subset(seurat_all,
                    subset = cell_type_fine %in% non_neuron_levels)
# 若上一步对象不存在，改用：
# nn_seurat <- subset(classFiltSeurat_filt,
#                     subset = cell_type_fine %in% non_neuron_levels &
#                              cell_type_fine != "Removed")

nn_seurat$cell_type_fine <- factor(as.character(nn_seurat$cell_type_fine),
                                   levels = non_neuron_levels)

cat("非神经元细胞数:", ncol(nn_seurat), "\n")
print(table(nn_seurat$cell_type_fine))

# 大类列（分面用）
nn_seurat$cell_class <- dplyr::case_when(
  startsWith(as.character(nn_seurat$cell_type_fine), "Astrocytes")  ~ "Astrocytes",
  startsWith(as.character(nn_seurat$cell_type_fine), "Oligo")       ~ "Oligo",
  startsWith(as.character(nn_seurat$cell_type_fine), "OPC")         ~ "OPC",
  startsWith(as.character(nn_seurat$cell_type_fine), "Microglia")   ~ "Microglia",
  startsWith(as.character(nn_seurat$cell_type_fine), "Endothelial") ~ "Endothelial",
  startsWith(as.character(nn_seurat$cell_type_fine), "Ependymal")   ~ "Ependymal"
)
nn_seurat$cell_class <- factor(nn_seurat$cell_class,
                               levels = c("Oligo","OPC","Astrocytes","Microglia",
                                          "Endothelial","Ependymal"))
print(table(nn_seurat$cell_class))

# ============================================
# 1. RNA + PCA + Harmony（非神经元内部重新整合）
# ============================================
DefaultAssay(nn_seurat) <- "RNA"

nn_seurat <- NormalizeData(nn_seurat, verbose = FALSE)
nn_seurat <- FindVariableFeatures(nn_seurat, selection.method = "vst",
                                  nfeatures = 3000, verbose = FALSE)
nn_seurat <- ScaleData(nn_seurat, verbose = FALSE)
nn_seurat <- RunPCA(nn_seurat, npcs = 50, verbose = FALSE)

nn_seurat <- RunHarmony(nn_seurat,
                        group.by.vars = "donor",
                        reduction.use = "pca",
                        dims.use = 1:30,
                        verbose = FALSE)

nn_seurat <- RunUMAP(nn_seurat, reduction = "harmony", dims = 1:30,
                     verbose = FALSE)



In [ ]:
# ============================================
# 2. 图 1：按 6 个大类着色（先看谱系分离）
# ============================================
p_class <- DimPlot(nn_seurat, group.by = "cell_class",
                   raster = FALSE, shuffle = TRUE) +
  coord_fixed() +
  ggtitle("Non-neuronal cells: major classes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
        legend.text = element_text(size = 11))
print(p_class)
save_fig(p_class, paste0(output_path, "nn_Fig1_UMAP_classes"), 8, 7)

# ============================================
# 3. 图 2：按 11 个亚群着色（带标签）
# ============================================
p_fine <- DimPlot(nn_seurat, group.by = "cell_type_fine",
                  label = TRUE, repel = TRUE, raster = TRUE,
                  shuffle = TRUE) +
  coord_fixed() +
  NoLegend() +
  ggtitle("Non-neuronal subtypes (11)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14))
print(p_fine)
save_fig(p_fine, paste0(output_path, "nn_Fig2_UMAP_subtypes"), 9, 8)

# ---------- 图 3：按大类分面（看每个大类内部结构）----------
p_split <- DimPlot(nn_seurat, group.by = "cell_type_fine",
                   split.by = "cell_class",
                   label = TRUE, repel = TRUE,
                   raster = TRUE, shuffle = TRUE) +
  coord_fixed() +
  NoLegend() +
  ggtitle("Non-neuronal subtypes split by class")
save_fig(p_split, paste0(output_path, "nn_Fig3_UMAP_split"), 20, 5)

# ---------- 图 4：donor 批次检查 ----------
p_donor <- DimPlot(nn_seurat, group.by = "donor",
                   raster = TRUE, shuffle = TRUE) +
  coord_fixed() +
  ggtitle("Non-neuronal: donor distribution")
save_fig(p_donor, paste0(output_path, "nn_Fig4_UMAP_donor"), 8, 7)

# # ============================================
# # 4. 保存
# # ============================================
# saveRDS(nn_seurat,
#         paste0(output_path, "nn_seurat_harmony_integrated.rds"))
# cat("非神经元整合对象已保存\n")

In [ ]:
suppressMessages(library(tidyverse))
library(Seurat)
library(patchwork)

output_path <- "./data/"

# ============================================
# 0. 读取已保存的带亚群注释的对象
# ============================================
classFiltSeurat <- readRDS(paste0(output_path, "classFiltSeurat_filt_allSubtypes.rds"))
cat("读取完成，细胞数:", ncol(classFiltSeurat), "\n")
print(colnames(classFiltSeurat@meta.data))   # 确认有 cell_type_fine 列

## ===== 1. 准备 =====
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)

## ===== 2. 基因 + 细胞类型顺序 =====
top_genes <- c("SNAP25", "SLC17A6", "SLC17A8", "GAD1", "GAD2", "TH")

target_types <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 2:6),
  paste0("GABA_", 1:16),
  "DA_1"
)

type2class <- c(
  setNames(rep("GLUT", 7),      paste0("Glut_", 1:7)),
  setNames(rep("GLUT_GABA", 5), paste0("Glut-GABA_", 2:6)),
  setNames(rep("GABA", 16),     paste0("GABA_", 1:16)),
  c("DA_1" = "DA")
)

class_colors <- c("GLUT"="#4DAF4A", "GLUT_GABA"="#984EA3",
                  "GABA"="#E41A1C", "DA"="#333333")

sub_obj <- subset(classFiltSeurat, subset = cell_type_fine %in% target_types)
cat("提取细胞数:", ncol(sub_obj), "\n")
print(table(sub_obj$cell_type_fine))

## ===== 3. 提取 + 长表 =====
expr <- FetchData(sub_obj, vars = c("cell_type_fine", top_genes))

expr_long <- expr %>%
  pivot_longer(cols = -cell_type_fine, names_to = "gene", values_to = "Expression") %>%
  mutate(gene = factor(gene, levels = top_genes),
         cell_type_fine = factor(cell_type_fine, levels = target_types),
         class = type2class[as.character(cell_type_fine)],
         class = factor(class, levels = c("GLUT", "GLUT_GABA", "GABA", "DA")))

## ===== 4. 顶部大类色条 =====
p_bar <- ggplot(expr_long, aes(x = cell_type_fine, y = 1, fill = class)) +
  geom_tile() +
  scale_fill_manual(values = class_colors) +
  scale_x_discrete(expand = c(0, 0)) +
  theme_void() +
  theme(legend.position = "none",
        plot.margin = margin(0, 5.5, 0, 5.5))

## ===== 5. 主图：每个基因一排，全部排成一列 =====
p_main <- ggplot(expr_long, aes(x = cell_type_fine, y = Expression, fill = class)) +
  geom_violin(scale = "width", trim = TRUE, color = "grey30", linewidth = 0.3) +
  facet_wrap(~ gene, nrow = 6, ncol = 1,
             scales = "free_y", strip.position = "right") +
  scale_fill_manual(values = class_colors) +
  scale_x_discrete(expand = c(0, 0)) +
  labs(x = NULL, y = "Expression Level") +
  theme_bw(base_size = 13) +
  theme(legend.position = "none",
        panel.grid = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(face = "bold", size = 12),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9))

## ===== 6. 拼接 + 保存 =====
p <- p_bar / p_main + plot_layout(heights = c(1, 18))

print(p)

ggsave(paste0(output_path, "VlnPlot_genes_stacked_neuro_types.png"),
       p, width = 12, height = 14, dpi = 300)
ggsave(paste0(output_path, "VlnPlot_genes_stacked_neuro_types.pdf"),
       p, width = 12, height = 14)
cat("✅ 图已保存\n")